In [1]:
# Import required libraries
import requests
import re
import pandas as pd

# DWD historical daily climate data directory
dwd_url = (
    "https://opendata.dwd.de/climate_environment/"
    "CDC/observations_germany/climate/daily/kl/historical/"
)

# Retrieve the directory listing
response = requests.get(dwd_url)
response.raise_for_status()

# Extract historical ZIP filenames
zip_files = re.findall(
    r'tageswerte_KL_\d{5}_\d{8}_\d{8}_hist\.zip',
    response.text
)

# Define the five stations
station_ids = {
    "00400": "Buch",
    "00403": "Dahlem",
    "00420": "Marzahn",
    "00427": "Berlin Brandenburg",
    "00433": "Tempelhof"
}

# Find the archive matching each station
station_files = []

for station_id, station_name in station_ids.items():

    matches = [
        filename for filename in zip_files
        if f"_{station_id}_" in filename
    ]

    for filename in matches:
        station_files.append({
            "station": station_name,
            "station_id": station_id,
            "filename": filename
        })

# Display the results
station_files_df = pd.DataFrame(station_files)

display(station_files_df)

,station,station_id,filename
0,Buch,00400,tageswerte_KL_00400_19510101_20251231_hist.zip
1,Buch,00400,tageswerte_KL_00400_19510101_20251231_hist.zip
2,Dahlem,00403,tageswerte_KL_00403_19500101_20251231_hist.zip
3,Dahlem,00403,tageswerte_KL_00403_19500101_20251231_hist.zip
4,Marzahn,00420,tageswerte_KL_00420_19930501_20251231_hist.zip
5,Marzahn,00420,tageswerte_KL_00420_19930501_20251231_hist.zip
6,Berlin Brandenburg,00427,tageswerte_KL_00427_19570101_20251231_hist.zip
7,Berlin Brandenburg,00427,tageswerte_KL_00427_19570101_20251231_hist.zip
8,Tempelhof,00433,tageswerte_KL_00433_19480101_20251231_hist.zip
9,Tempelhof,00433,tageswerte_KL_00433_19480101_20251231_hist.zip


In [2]:
# Remove duplicate station archive entries
station_files_df = (
    station_files_df
    .drop_duplicates(subset=["station_id", "filename"])
    .reset_index(drop=True)
)

# Check the final archive list
display(station_files_df)

print("Number of unique station archives:", len(station_files_df))

,station,station_id,filename
0,Buch,00400,tageswerte_KL_00400_19510101_20251231_hist.zip
1,Dahlem,00403,tageswerte_KL_00403_19500101_20251231_hist.zip
2,Marzahn,00420,tageswerte_KL_00420_19930501_20251231_hist.zip
3,Berlin Brandenburg,00427,tageswerte_KL_00427_19570101_20251231_hist.zip
4,Tempelhof,00433,tageswerte_KL_00433_19480101_20251231_hist.zip


Number of unique station archives: 5


In [3]:
# Import libraries
import os
import requests

# Create a folder for historical DWD data
download_folder = "dwd_historical_2019_2025"
os.makedirs(download_folder, exist_ok=True)

# Download each station archive
for _, row in station_files_df.iterrows():

    filename = row["filename"]
    file_url = dwd_url + filename
    file_path = os.path.join(download_folder, filename)

    print(f"Downloading {row['station']}...")

    response = requests.get(file_url)
    response.raise_for_status()

    with open(file_path, "wb") as file:
        file.write(response.content)

    print(f"Saved: {filename}")

print("\nDownload completed.")

Saved: tageswerte_KL_00400_19510101_20251231_hist.zip
Saved: tageswerte_KL_00403_19500101_20251231_hist.zip
Saved: tageswerte_KL_00420_19930501_20251231_hist.zip
Saved: tageswerte_KL_00427_19570101_20251231_hist.zip
Saved: tageswerte_KL_00433_19480101_20251231_hist.zip

Download completed.


In [4]:
# Inspect the contents of each ZIP archive
import zipfile
import os

for filename in os.listdir(download_folder):

    if filename.endswith(".zip"):

        file_path = os.path.join(download_folder, filename)

        print(f"\n{filename}")

        with zipfile.ZipFile(file_path, "r") as archive:
            for item in archive.namelist():
                print("  ", item)


tageswerte_KL_00400_19510101_20251231_hist.zip
   Metadaten_Geraete_Luftdruck_Stationshoehe_00400.html
   Metadaten_Geraete_Luftdruck_Stationshoehe_00400.txt
   Metadaten_Stationsname_Betreibername_00400.html
   Metadaten_Stationsname_Betreibername_00400.txt
   Metadaten_Parameter_klima_tag_00400.html
   Metadaten_Parameter_klima_tag_00400.txt
   Metadaten_Geraete_Lufttemp_Am_Erdb_Minimum_00400.html
   Metadaten_Geraete_Lufttemp_Am_Erdb_Minimum_00400.txt
   Metadaten_Geraete_Lufttemperatur_00400.html
   Metadaten_Geraete_Lufttemperatur_00400.txt
   Metadaten_Geraete_Lufttemperatur_Maximum_00400.html
   Metadaten_Geraete_Lufttemperatur_Maximum_00400.txt
   Metadaten_Geraete_Lufttemperatur_Minimum_00400.html
   Metadaten_Geraete_Lufttemperatur_Minimum_00400.txt
   Metadaten_Geraete_Niederschlagshoehe_00400.html
   Metadaten_Geraete_Niederschlagshoehe_00400.txt
   Metadaten_Geraete_Rel_Feuchte_00400.html
   Metadaten_Geraete_Rel_Feuchte_00400.txt
   Metadaten_Geraete_Schneehoehe_00400.ht

In [5]:
# Extract daily observation files from each ZIP
import zipfile
import os

extracted_folder = os.path.join(
    download_folder, "extracted"
)

os.makedirs(extracted_folder, exist_ok=True)

for filename in os.listdir(download_folder):

    if filename.endswith(".zip"):

        zip_path = os.path.join(download_folder, filename)

        with zipfile.ZipFile(zip_path, "r") as archive:

            observation_files = [
                item for item in archive.namelist()
                if item.startswith("produkt_klima_tag_")
                and item.endswith(".txt")
            ]

            for item in observation_files:
                archive.extract(item, extracted_folder)

                print("Extracted:", item)

Extracted: produkt_klima_tag_19510101_20251231_00400.txt
Extracted: produkt_klima_tag_19500101_20251231_00403.txt
Extracted: produkt_klima_tag_19930501_20251231_00420.txt
Extracted: produkt_klima_tag_19570101_20251231_00427.txt
Extracted: produkt_klima_tag_19480101_20251231_00433.txt


In [11]:
# DWD station IDs and names
dwd_stations = {
    "00400": "Buch",
    "00403": "Dahlem",
    "00420": "Marzahn",
    "00427": "Berlin Brandenburg",
    "00433": "Tempelhof"
}

# DWD-to-final column mapping
dwd_column_mapping = {
    "MESS_DATUM": "date",
    "TMK": "tavg",
    "TNK": "tmin",
    "TXK": "tmax",
    "RSK": "prcp",
    "FM": "wspd",
    "PM": "pres"
}

# Analysis period
start_date = "2019-01-01"
end_date = "2025-04-30"

In [12]:
# Load the five DWD station files

dwd_data = {}

for station_id, station_name in dwd_stations.items():

    # Find the corresponding extracted file
    filename = [
        f for f in os.listdir(extracted_folder)
        if f.endswith(f"_{station_id}.txt")
    ][0]

    filepath = os.path.join(extracted_folder, filename)

    # Read the raw observations
    df = pd.read_csv(
        filepath,
        sep=";",
        encoding="latin-1"
    )

    # Remove whitespace from column names
    df.columns = df.columns.str.strip()

    # Store the DataFrame
    dwd_data[station_name] = df

    print(f"{station_name}: {df.shape}")

Buch: (27394, 19)
Dahlem: (27745, 19)
Marzahn: (10112, 19)
Berlin Brandenburg: (25202, 19)
Tempelhof: (28490, 19)


In [47]:
# Define the analysis period
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2025-04-30")

In [48]:
# Prepare each station's DWD observations

dwd_filtered = {}

for station_name, df in dwd_data.items():

    # Keep only the required columns
    df = df[list(dwd_column_mapping.keys())].copy()

    # Parse observation dates
    df["MESS_DATUM"] = pd.to_datetime(
        df["MESS_DATUM"].astype(str),
        format="%Y%m%d"
    )

    # Filter the analysis period
    df = df[
        (df["MESS_DATUM"] >= start_date) &
        (df["MESS_DATUM"] <= end_date)
    ].copy()

    # Rename columns
    df = df.rename(columns=dwd_column_mapping)

    # Add station and source information
    df["station"] = station_name
    df["source"] = "DWD"

    dwd_filtered[station_name] = df

    print(
        f"{station_name}: "
        f"{df['date'].min().date()} to "
        f"{df['date'].max().date()} | "
        f"{len(df):,} rows"
    )

Buch: 2022-01-01 to 2025-04-30 | 1,216 rows
Dahlem: 2022-01-01 to 2025-04-30 | 1,202 rows
Marzahn: 2022-01-01 to 2025-04-30 | 703 rows
Berlin Brandenburg: 2022-01-01 to 2025-04-30 | 1,216 rows
Tempelhof: 2022-01-01 to 2025-04-30 | 1,216 rows


In [49]:
# Check missing values in each station

for station_name, df in dwd_filtered.items():

    print(f"\n{station_name}")
    print("-" * 40)

    # Count actual NaN values
    print("NaN values:")
    print(df.isna().sum())

    # Count DWD missing-value sentinels
    print("\n-999 values:")
    print((df == -999).sum())


Buch
----------------------------------------
NaN values:
date       0
tavg       0
tmin       0
tmax       0
prcp       0
wspd       0
pres       0
station    0
source     0
dtype: int64

-999 values:
date          0
tavg          0
tmin          0
tmax          0
prcp          0
wspd       1216
pres       1216
station       0
source        0
dtype: int64

Dahlem
----------------------------------------
NaN values:
date       0
tavg       0
tmin       0
tmax       0
prcp       0
wspd       0
pres       0
station    0
source     0
dtype: int64

-999 values:
date          0
tavg          2
tmin          2
tmax          2
prcp          1
wspd       1202
pres          3
station       0
source        0
dtype: int64

Marzahn
----------------------------------------
NaN values:
date       0
tavg       0
tmin       0
tmax       0
prcp       0
wspd       0
pres       0
station    0
source     0
dtype: int64

-999 values:
date         0
tavg         7
tmin         7
tmax         7
prcp        

In [50]:
# Replace DWD missing-value sentinels with NaN

for station_name, df in dwd_filtered.items():

    # Replace -999 with NaN
    dwd_filtered[station_name] = df.replace(-999, float("nan"))

    print(f"\n{station_name}")
    print(dwd_filtered[station_name].isna().sum())


Buch
date          0
tavg          0
tmin          0
tmax          0
prcp          0
wspd       1216
pres       1216
station       0
source        0
dtype: int64

Dahlem
date          0
tavg          2
tmin          2
tmax          2
prcp          1
wspd       1202
pres          3
station       0
source        0
dtype: int64

Marzahn
date         0
tavg         7
tmin         7
tmax         7
prcp         7
wspd       703
pres       703
station      0
source       0
dtype: int64

Berlin Brandenburg
date        0
tavg        0
tmin        0
tmax        0
prcp        1
wspd       12
pres        7
station     0
source      0
dtype: int64

Tempelhof
date        0
tavg        0
tmin        0
tmax        0
prcp       10
wspd        9
pres        6
station     0
source      0
dtype: int64


In [51]:
# Check missing dates in each DWD station

expected_dates = pd.date_range(start=start_date, end=end_date, freq="D")

for station_name, df in dwd_filtered.items():

    # Identify dates missing from the dataset
    missing_dates = expected_dates.difference(df["date"])

    print(f"\n{station_name}")
    print("-" * 40)
    print(f"Expected dates: {len(expected_dates):,}")
    print(f"Available dates: {df['date'].nunique():,}")
    print(f"Missing dates: {len(missing_dates):,}")
    print(f"Duplicate dates: {df['date'].duplicated().sum():,}")

    if len(missing_dates) > 0:
        print("\nFirst 15 missing dates:")
        print(missing_dates[:15].strftime("%Y-%m-%d").tolist())


Buch
----------------------------------------
Expected dates: 1,216
Available dates: 1,216
Missing dates: 0
Duplicate dates: 0

Dahlem
----------------------------------------
Expected dates: 1,216
Available dates: 1,202
Missing dates: 14
Duplicate dates: 0

First 15 missing dates:
['2025-03-04', '2025-03-05', '2025-03-06', '2025-03-07', '2025-03-08', '2025-03-09', '2025-03-10', '2025-03-11', '2025-03-12', '2025-03-13', '2025-03-14', '2025-03-15', '2025-03-16', '2025-03-17']

Marzahn
----------------------------------------
Expected dates: 1,216
Available dates: 703
Missing dates: 513
Duplicate dates: 0

First 15 missing dates:
['2023-08-07', '2023-08-08', '2023-08-09', '2023-08-10', '2023-08-11', '2023-08-12', '2023-08-13', '2023-08-14', '2023-08-15', '2023-08-16', '2023-08-17', '2023-08-18', '2023-08-19', '2023-08-20', '2023-08-21']

Berlin Brandenburg
----------------------------------------
Expected dates: 1,216
Available dates: 1,216
Missing dates: 0
Duplicate dates: 0

Tempelhof

In [52]:
# Identify continuous periods of missing dates

for station_name, df in dwd_filtered.items():

    expected_dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="D"
    )

    missing_dates = expected_dates.difference(df["date"])

    print(f"\n{station_name}")
    print("-" * 40)

    if len(missing_dates) == 0:
        print("No missing dates")
        continue

    # Group consecutive missing dates into periods
    missing_series = pd.Series(missing_dates)

    groups = (
        missing_series.diff().dt.days.ne(1)
    ).cumsum()

    missing_periods = missing_series.groupby(groups).agg(
        start="min",
        end="max",
        days="count"
    )

    print(missing_periods.to_string(index=False))


Buch
----------------------------------------
No missing dates

Dahlem
----------------------------------------
     start        end  days
2025-03-04 2025-03-17    14

Marzahn
----------------------------------------
     start        end  days
2023-08-07 2024-12-31   513

Berlin Brandenburg
----------------------------------------
No missing dates

Tempelhof
----------------------------------------
No missing dates


In [59]:
# Standardize the DWD datasets

weather_columns = ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"]

standardized_dwd = []

for station_name, df in dwd_filtered.items():

    # Work on a copy
    df = df.copy()

    # Ensure consistent date format
    df["date"] = pd.to_datetime(df["date"])

    # Ensure weather variables are numeric
    for col in weather_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Standardize column order
    df = df[
        ["date", "tavg", "tmin", "tmax", "prcp",
         "wspd", "pres", "station", "source"]
    ]

    standardized_dwd.append(df)

# Combine all five stations
dwd_clean = pd.concat(standardized_dwd, ignore_index=True)

# Sort observations
dwd_clean = dwd_clean.sort_values(
    ["station", "date"]
).reset_index(drop=True)

# Inspect the result
print("Shape:", dwd_clean.shape)
print("\nData types:")
print(dwd_clean.dtypes)

print("\nRows per station:")
print(dwd_clean.groupby("station").size())

print("\nDuplicate station-date combinations:",
      dwd_clean.duplicated(["station", "date"]).sum())

Shape: (5553, 9)

Data types:
date       datetime64[us]
tavg              float64
tmin              float64
tmax              float64
prcp              float64
wspd              float64
pres              float64
station               str
source                str
dtype: object

Rows per station:
station
Berlin Brandenburg    1216
Buch                  1216
Dahlem                1202
Marzahn                703
Tempelhof             1216
dtype: int64

Duplicate station-date combinations: 0


In [60]:
# Check missing values and numeric ranges by station

weather_columns = ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"]

print("Missing values by station:")
display(
    dwd_clean.groupby("station")[weather_columns]
    .apply(lambda group: group.isna().sum())
)

print("\nMinimum values by station:")
display(dwd_clean.groupby("station")[weather_columns].min())

print("\nMaximum values by station:")
display(dwd_clean.groupby("station")[weather_columns].max())

# Check temperature consistency
print("\nTemperature consistency checks:")
print("tmin > tavg:", (dwd_clean["tmin"] > dwd_clean["tavg"]).sum())
print("tavg > tmax:", (dwd_clean["tavg"] > dwd_clean["tmax"]).sum())
print("tmin > tmax:", (dwd_clean["tmin"] > dwd_clean["tmax"]).sum())

# Check for negative precipitation and wind speed
print("\nNegative precipitation values:", (dwd_clean["prcp"] < 0).sum())
print("Negative wind speed values:", (dwd_clean["wspd"] < 0).sum())

Missing values by station:


,tavg,tmin,tmax,prcp,wspd,pres
station,,,,,,
Berlin Brandenburg,0,0,0,1,12,7
Buch,0,0,0,0,1216,1216
Dahlem,2,2,2,1,1202,3
Marzahn,7,7,7,7,703,703
Tempelhof,0,0,0,10,9,6



Minimum values by station:


,tavg,tmin,tmax,prcp,wspd,pres
station,,,,,,
Berlin Brandenburg,-8.8,-12.5,-4.4,0.0,1.0,975.7
Buch,-8.7,-12.5,-4.6,0.0,NaN,NaN
Dahlem,-7.5,-11.2,-4.4,0.0,NaN,971.8
Marzahn,-6.9,-10.6,-4.7,0.0,NaN,NaN
Tempelhof,-7.7,-11.6,-4.4,0.0,0.8,975.6



Maximum values by station:


,tavg,tmin,tmax,prcp,wspd,pres
station,,,,,,
Berlin Brandenburg,29.3,20.0,38.1,82.2,14.4,1037.2
Buch,29.0,20.4,38.3,27.0,NaN,NaN
Dahlem,27.9,20.5,37.6,27.5,NaN,1033.3
Marzahn,29.7,20.8,37.4,32.1,NaN,NaN
Tempelhof,30.0,20.8,37.9,36.3,10.6,1037.5



Temperature consistency checks:
tmin > tavg: 0
tavg > tmax: 0
tmin > tmax: 0

Negative precipitation values: 0
Negative wind speed values: 0


In [61]:
# Check completeness of core weather variables

core_weather_columns = ["tavg", "tmin", "tmax", "prcp"]

print("Missing core variables by station:")
print(
    dwd_clean.groupby("station")[core_weather_columns]
    .apply(lambda group: group.isna().sum())
)

print("\nRows with at least one missing core variable:")
print(
    dwd_clean.groupby("station")[core_weather_columns]
    .apply(lambda group: group.isna().any(axis=1).sum())
)

print("\nRows with all core variables available:")
print(
    dwd_clean.groupby("station")[core_weather_columns]
    .apply(lambda group: group.notna().all(axis=1).sum())
)

Missing core variables by station:
                    tavg  tmin  tmax  prcp
station                                   
Berlin Brandenburg     0     0     0     1
Buch                   0     0     0     0
Dahlem                 2     2     2     1
Marzahn                7     7     7     7
Tempelhof              0     0     0    10

Rows with at least one missing core variable:
station
Berlin Brandenburg     1
Buch                   0
Dahlem                 2
Marzahn                7
Tempelhof             10
dtype: int64

Rows with all core variables available:
station
Berlin Brandenburg    1215
Buch                  1216
Dahlem                1200
Marzahn                696
Tempelhof             1206
dtype: int64


In [62]:
# Check available observations by station and year

dwd_clean["year"] = dwd_clean["date"].dt.year

yearly_coverage = (
    dwd_clean.groupby(["station", "year"])
    .agg(
        recorded_days=("date", "nunique"),
        temperature_days=("tavg", "count"),
        precipitation_days=("prcp", "count")
    )
    .reset_index()
)

print(yearly_coverage.to_string(index=False))

           station  year  recorded_days  temperature_days  precipitation_days
Berlin Brandenburg  2022            365               365                 364
Berlin Brandenburg  2023            365               365                 365
Berlin Brandenburg  2024            366               366                 366
Berlin Brandenburg  2025            120               120                 120
              Buch  2022            365               365                 365
              Buch  2023            365               365                 365
              Buch  2024            366               366                 366
              Buch  2025            120               120                 120
            Dahlem  2022            365               365                 365
            Dahlem  2023            365               365                 365
            Dahlem  2024            366               366                 366
            Dahlem  2025            106               104       

In [63]:
# Summarize missing calendar dates by station

expected_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

coverage_summary = []

for station, group in dwd_clean.groupby("station"):

    recorded_dates = pd.DatetimeIndex(group["date"].unique())
    missing_dates = expected_dates.difference(recorded_dates)

    coverage_summary.append({
        "station": station,
        "expected_days": len(expected_dates),
        "recorded_days": len(recorded_dates),
        "missing_days": len(missing_dates),
        "coverage_pct": round(
            len(recorded_dates) / len(expected_dates) * 100, 2
        ),
        "first_missing_date": (
            missing_dates.min() if len(missing_dates) else None
        ),
        "last_missing_date": (
            missing_dates.max() if len(missing_dates) else None
        )
    })

coverage_summary_df = pd.DataFrame(coverage_summary)

print(coverage_summary_df.to_string(index=False))

           station  expected_days  recorded_days  missing_days  coverage_pct first_missing_date last_missing_date
Berlin Brandenburg           1216           1216             0        100.00                NaT               NaT
              Buch           1216           1216             0        100.00                NaT               NaT
            Dahlem           1216           1202            14         98.85         2025-03-04        2025-03-17
           Marzahn           1216            703           513         57.81         2023-08-07        2024-12-31
         Tempelhof           1216           1216             0        100.00                NaT               NaT


In [64]:
# Descriptive statistics for DWD weather variables

weather_columns = ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"]

print(
    dwd_clean.groupby("station")[weather_columns]
    .describe()
    .round(2)
    .to_string()
)

                      tavg                                              tmin                                              tmax                                                prcp                                          wspd                                          pres                                                      
                     count   mean   std  min  25%    50%   75%   max   count  mean   std   min   25%  50%    75%   max   count   mean   std  min   25%    50%    75%   max   count  mean   std  min  25%  50%  75%   max   count  mean   std  min  25%  50%  75%   max   count     mean   std    min     25%     50%     75%     max
station                                                                                                                                                                                                                                                                                                                             
Berlin Brandenburg  1216.

In [65]:
# Save cleaned DWD historical weather data

output_path = "dwd_historical_clean_2022_2025.csv"

# Remove the temporary year column if it exists
dwd_to_save = dwd_clean.drop(columns=["year"], errors="ignore").copy()

# Sort by station and date
dwd_to_save = dwd_to_save.sort_values(
    ["station", "date"]
).reset_index(drop=True)

# Save CSV
dwd_to_save.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print(f"Saved: {output_path}")
print("Shape:", dwd_to_save.shape)
print("Date range:", dwd_to_save["date"].min(), "to", dwd_to_save["date"].max())
print("Stations:", sorted(dwd_to_save["station"].unique()))

Saved: dwd_historical_clean_2022_2025.csv
Shape: (5553, 9)
Date range: 2022-01-01 00:00:00 to 2025-04-30 00:00:00
Stations: ['Berlin Brandenburg', 'Buch', 'Dahlem', 'Marzahn', 'Tempelhof']


In [66]:
# Check Meteostat package and existing data

import meteostat
import pandas as pd

print("Meteostat version:", meteostat.__version__)

print("\nExisting Meteostat variables:")
print("meteostat_data:", "meteostat_data" in globals())
print("meteostat_df:", "meteostat_df" in globals())

print("\nTarget period:")
print("Start:", "2022-01-01")
print("End:", "2025-04-30")

Meteostat version: 2.1.5

Existing Meteostat variables:
meteostat_data: True
meteostat_df: True

Target period:
Start: 2022-01-01
End: 2025-04-30


In [67]:
# Inspect Meteostat station metadata

import meteostat as ms

meteostat_station_ids = {
    "Alexanderplatz": "10389",
    "Tegel": "10382",
    "Dahlem": "10381",
    "Marzahn": "D0420"
}

meteostat_station_metadata = {}

for station_name, station_id in meteostat_station_ids.items():

    station = ms.stations.meta(station_id)
    meteostat_station_metadata[station_name] = station

    print(f"\n{station_name}")
    print("-" * 40)
    print(station)


Alexanderplatz
----------------------------------------
Station(id='10389', name='Berlin / Alexanderplatz', country='DE', region='BE', identifiers={'national': '00399', 'wmo': '10389', 'mosmix': '10389'}, latitude=52.5167, longitude=13.4167, elevation=37, timezone='Europe/Berlin')

Tegel
----------------------------------------
Station(id='10382', name='Berlin / Tegel', country='DE', region='BE', identifiers={'national': '00430', 'wmo': '10382', 'icao': 'EDDT', 'iata': 'TXL', 'ghcn': 'GME00121150', 'usaf': '103820', 'mosmix': '10382'}, latitude=52.5667, longitude=13.3167, elevation=37, timezone='Europe/Berlin')

Dahlem
----------------------------------------
Station(id='10381', name='Berlin / Dahlem', country='DE', region='BE', identifiers={'national': '00403', 'wmo': '10381', 'ghcn': 'GM000003319', 'mosmix': '10381'}, latitude=52.4667, longitude=13.3, elevation=51, timezone='Europe/Berlin')

Marzahn
----------------------------------------
Station(id='D0420', name='Berlin-Marzahn', 

In [ ]:
#importing meteostat missing datasets for 2022/2025

alex_hist= pd.read_csv("daily_weather_data/meteostat_alexanderplatz_2022_2025.csv")
dahlem_hist= pd.read_csv("daily_weather_data/meteostat_dahlem_2022_2025.csv")
tegel_hist= pd.read_csv("daily_weather_data/meteostat_tegel_2022_2025.csv")
marzahn_hist= pd.read_csv("daily_weather_data/meteostat_marzahn_2022_2025.csv")

In [76]:
# Inspect Meteostat historical datasets

meteostat_hist = {
    "Alexanderplatz": alex_hist,
    "Dahlem": dahlem_hist,
    "Tegel": tegel_hist,
    "Marzahn": marzahn_hist
}

for station, df in meteostat_hist.items():

    print(f"\n{'=' * 55}")
    print(f"{station}")
    print("=" * 55)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

    print("\nDate range:")
    print(df.iloc[:, 0].min(), "to", df.iloc[:, 0].max())


Alexanderplatz
Shape: (1097, 11)
Columns: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']

Data types:
date        str
tavg    float64
tmin    float64
tmax    float64
prcp    float64
snow    float64
wdir    float64
wspd    float64
wpgt    float64
pres    float64
tsun    float64
dtype: object

First 3 rows:
               date  tavg  tmin  tmax  prcp  snow  wdir  wspd  wpgt   pres  tsun
2022-01-01 00:00:00  11.2   9.5  11.7   1.7   NaN   NaN  16.4  42.6 1021.3   NaN
2022-01-02 00:00:00   9.5   8.1  10.9   2.0   NaN   NaN  17.7  46.3 1013.0   NaN
2022-01-03 00:00:00   8.7   7.0  10.0   4.0   NaN   NaN  20.2  51.8 1006.5   NaN

Date range:
2022-01-01 00:00:00 to 2025-01-01 00:00:00

Dahlem
Shape: (1097, 11)
Columns: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']

Data types:
date        str
tavg    float64
tmin    float64
tmax    float64
prcp    float64
snow      int64
wdir    float64
wspd    float64
wpgt 

In [77]:
# Inspect Meteostat historical data quality and date coverage

for station, df in meteostat_hist.items():

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    expected_dates = pd.date_range(
        start="2022-01-01",
        end="2025-04-30",
        freq="D"
    )

    missing_dates = expected_dates.difference(df["date"])

    print(f"\n{'=' * 45}")
    print(station)
    print("=" * 45)

    print("Rows:", len(df))
    print("First date:", df["date"].min())
    print("Last date:", df["date"].max())
    print("Duplicate dates:", df["date"].duplicated().sum())
    print("Missing calendar dates:", len(missing_dates))

    print("\nMissing values:")
    print(df.isna().sum().to_string())


Alexanderplatz
Rows: 1097
First date: 2022-01-01 00:00:00
Last date: 2025-01-01 00:00:00
Duplicate dates: 0
Missing calendar dates: 119

Missing values:
date       0
tavg      12
tmin      12
tmax      12
prcp     945
snow    1097
wdir    1097
wspd      12
wpgt      12
pres      12
tsun     221

Dahlem
Rows: 1097
First date: 2022-01-01 00:00:00
Last date: 2025-01-01 00:00:00
Duplicate dates: 0
Missing calendar dates: 119

Missing values:
date       0
tavg       0
tmin       0
tmax       0
prcp       0
snow       0
wdir    1097
wspd       0
wpgt      12
pres       0
tsun       0

Tegel
Rows: 1097
First date: 2022-01-01 00:00:00
Last date: 2025-01-01 00:00:00
Duplicate dates: 0
Missing calendar dates: 119

Missing values:
date       0
tavg       0
tmin       0
tmax       0
prcp     241
snow    1097
wdir    1097
wspd       0
wpgt      12
pres       0
tsun     221

Marzahn
Rows: 1097
First date: 2022-01-01 00:00:00
Last date: 2025-01-01 00:00:00
Duplicate dates: 0
Missing calendar dates: 

In [78]:
# Identify missing dates within the historical Meteostat files

for station, df in meteostat_hist.items():

    dates = pd.to_datetime(df["date"]).sort_values()

    expected_dates = pd.date_range(
        start="2022-01-01",
        end="2024-12-31",
        freq="D"
    )

    missing_dates = expected_dates.difference(dates)

    print(f"\n{'=' * 45}")
    print(station)
    print("=" * 45)

    print("Missing dates within 2022–2024:", len(missing_dates))

    if len(missing_dates) > 0:
        print("First missing dates:", missing_dates[:10].tolist())


Alexanderplatz
Missing dates within 2022–2024: 0

Dahlem
Missing dates within 2022–2024: 0

Tegel
Missing dates within 2022–2024: 0

Marzahn
Missing dates within 2022–2024: 0


In [79]:
# Standardise historical Meteostat datasets

meteostat_clean_list = []

for station, df in meteostat_hist.items():

    # Keep the variables shared with the DWD dataset
    df_clean = df[
        ["date", "tavg", "tmin", "tmax", "prcp", "wspd", "pres"]
    ].copy()

    # Standardise date format
    df_clean["date"] = pd.to_datetime(df_clean["date"])

    # Add station and source identifiers
    df_clean["station"] = station
    df_clean["source"] = "Meteostat"

    meteostat_clean_list.append(df_clean)

# Combine all four stations
meteostat_historical_clean = pd.concat(
    meteostat_clean_list,
    ignore_index=True
)

# Sort by station and date
meteostat_historical_clean = (
    meteostat_historical_clean
    .sort_values(["station", "date"])
    .reset_index(drop=True)
)

print("Shape:", meteostat_historical_clean.shape)
print("\nColumns:")
print(meteostat_historical_clean.columns.tolist())

print("\nRows per station:")
print(meteostat_historical_clean["station"].value_counts())

print("\nData types:")
print(meteostat_historical_clean.dtypes)

Shape: (4388, 9)

Columns:
['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres', 'station', 'source']

Rows per station:
station
Alexanderplatz    1097
Dahlem            1097
Marzahn           1097
Tegel             1097
Name: count, dtype: int64

Data types:
date       datetime64[us]
tavg              float64
tmin              float64
tmax              float64
prcp              float64
wspd              float64
pres              float64
station               str
source                str
dtype: object


In [80]:
# Validate missing values and temperature consistency

print("Missing values:")
print(meteostat_historical_clean.isna().sum())

print("\nTemperature consistency checks:")

print(
    "Days where tmin > tavg:",
    (meteostat_historical_clean["tmin"] >
     meteostat_historical_clean["tavg"]).sum()
)

print(
    "Days where tavg > tmax:",
    (meteostat_historical_clean["tavg"] >
     meteostat_historical_clean["tmax"]).sum()
)

print(
    "Days where tmin > tmax:",
    (meteostat_historical_clean["tmin"] >
     meteostat_historical_clean["tmax"]).sum()
)

print("\nDate range:")
print(meteostat_historical_clean["date"].min())
print(meteostat_historical_clean["date"].max())

Missing values:
date          0
tavg         23
tmin         25
tmax         25
prcp       1199
wspd         12
pres         23
station       0
source        0
dtype: int64

Temperature consistency checks:
Days where tmin > tavg: 0
Days where tavg > tmax: 0
Days where tmin > tmax: 0

Date range:
2022-01-01 00:00:00
2025-01-01 00:00:00


In [83]:
# Inspect the existing combined weather dataset

df_existing = pd.read_csv(
    "merged_complete_datasets/df_weather_combined_all_updated.csv"
)

df_existing["date"] = pd.to_datetime(df_existing["date"])

print("Shape:", df_existing.shape)

print("\nColumns:")
print(df_existing.columns.tolist())

print("\nData types:")
print(df_existing.dtypes)

print("\nDate range:")
print(df_existing["date"].min())
print(df_existing["date"].max())

print("\nRows per station and source:")
print(
    df_existing.groupby(["station", "source"])
    .size()
    .to_string()
)

Shape: (4995, 14)

Columns:
['Unnamed: 0', 'station_id', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'tsun', 'cloud_cover', 'pres_dwd', 'station', 'source', 'pres_meteostat']

Data types:
Unnamed: 0               float64
station_id               float64
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                     float64
cloud_cover              float64
pres_dwd                 float64
station                      str
source                       str
pres_meteostat           float64
dtype: object

Date range:
2025-01-01 00:00:00
2026-10-30 00:00:00

Rows per station and source:
station             source   
Alexanderplatz      Meteostat    668
Berlin Brandenburg  DWD          550
Buch                DWD          550
Dahlem              DWD          550
                    Meteostat    303
Marzahn             DWD          5

In [84]:
# Inspect station IDs and pressure columns by station and source

print("Station IDs by station and source:")
print(
    df_existing.groupby(["station", "source"])["station_id"]
    .agg(["count", "nunique", "first"])
    .to_string()
)

print("\nPressure availability by station and source:")
print(
    df_existing.groupby(["station", "source"])[
        ["pres_dwd", "pres_meteostat"]
    ]
    .agg(["count", "mean"])
    .to_string()
)

print("\nDuplicate station-date-source combinations:")
print(
    df_existing.duplicated(
        subset=["station", "date", "source"]
    ).sum()
)

Station IDs by station and source:
                              count  nunique  first
station            source                          
Alexanderplatz     Meteostat      0        0    NaN
Berlin Brandenburg DWD          550        1  427.0
Buch               DWD          550        1  400.0
Dahlem             DWD          550        1  403.0
                   Meteostat      0        0    NaN
Marzahn            DWD          550        1  420.0
                   Meteostat      0        0    NaN
Tegel              Meteostat      0        0    NaN
Tempelhof          DWD          550        1  433.0
                   Meteostat      0        0    NaN

Pressure availability by station and source:
                             pres_dwd              pres_meteostat             
                                count         mean          count         mean
station            source                                                     
Alexanderplatz     Meteostat        0          NaN        

In [85]:
# Compare historical Meteostat records with existing records on overlapping dates

comparison_columns = [
    "tavg", "tmin", "tmax", "prcp", "wspd"
]

for station in meteostat_historical_clean["station"].unique():

    historical_row = meteostat_historical_clean[
        (meteostat_historical_clean["station"] == station) &
        (meteostat_historical_clean["date"] == "2025-01-01")
    ]

    existing_row = df_existing[
        (df_existing["station"] == station) &
        (df_existing["source"] == "Meteostat") &
        (df_existing["date"] == "2025-01-01")
    ]

    print(f"\n{'=' * 45}")
    print(station)
    print("=" * 45)

    if historical_row.empty:
        print("No historical record found.")
        continue

    if existing_row.empty:
        print("No matching record in existing dataset.")
        continue

    print("\nHistorical record:")
    print(historical_row[comparison_columns + ["pres"]].to_string(index=False))

    print("\nExisting record:")
    print(
        existing_row[
            comparison_columns + ["pres_meteostat"]
        ].to_string(index=False)
    )


Alexanderplatz

Historical record:
 tavg  tmin  tmax  prcp  wspd   pres
  5.2   2.0   7.2   NaN  26.5 1011.5

Existing record:
 tavg  tmin  tmax  prcp  wspd  pres_meteostat
  5.2   2.0   7.2   NaN  26.5          1011.5

Dahlem
No matching record in existing dataset.

Marzahn
No matching record in existing dataset.

Tegel

Historical record:
 tavg  tmin  tmax  prcp  wspd   pres
  5.3   2.2   7.2   NaN  26.6 1011.2

Existing record:
 tavg  tmin  tmax  prcp  wspd  pres_meteostat
  5.3   2.2   7.2   NaN  26.6          1011.2


In [86]:
# Prepare historical Meteostat data to match the existing dataset

meteostat_to_append = meteostat_historical_clean[
    meteostat_historical_clean["date"] <= "2024-12-31"
].copy()

# Rename Meteostat pressure column
meteostat_to_append = meteostat_to_append.rename(
    columns={"pres": "pres_meteostat"}
)

# Add columns from the existing dataset that are not available historically
meteostat_to_append["station_id"] = pd.NA
meteostat_to_append["tsun"] = pd.NA
meteostat_to_append["cloud_cover"] = pd.NA
meteostat_to_append["pres_dwd"] = pd.NA

# Reorder columns to match the existing dataset
meteostat_to_append = meteostat_to_append[
    df_existing.columns.drop("Unnamed: 0", errors="ignore")
]

print("Shape:", meteostat_to_append.shape)
print("\nColumns:")
print(meteostat_to_append.columns.tolist())

print("\nDate range:")
print(meteostat_to_append["date"].min())
print(meteostat_to_append["date"].max())

print("\nRows per station:")
print(meteostat_to_append["station"].value_counts())

Shape: (4384, 13)

Columns:
['station_id', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'tsun', 'cloud_cover', 'pres_dwd', 'station', 'source', 'pres_meteostat']

Date range:
2022-01-01 00:00:00
2024-12-31 00:00:00

Rows per station:
station
Alexanderplatz    1096
Dahlem            1096
Marzahn           1096
Tegel             1096
Name: count, dtype: int64


In [87]:
# Compare data types before concatenation

print("EXISTING DATASET")
print(df_existing.dtypes.to_string())

print("\nHISTORICAL DATASET")
print(meteostat_to_append.dtypes.to_string())

print("\nColumn differences:")
print("Existing only:", set(df_existing.columns) - set(meteostat_to_append.columns))
print("Historical only:", set(meteostat_to_append.columns) - set(df_existing.columns))

EXISTING DATASET
Unnamed: 0               float64
station_id               float64
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                     float64
cloud_cover              float64
pres_dwd                 float64
station                      str
source                       str
pres_meteostat           float64

HISTORICAL DATASET
station_id                object
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                      object
cloud_cover               object
pres_dwd                  object
station                      str
source                       str
pres_meteostat           float64

Column differences:
Existing only: {'Unnamed: 0'}
Historical only: set(

In [88]:
# Align historical dataset data types with the existing dataset

numeric_columns = [
    "station_id",
    "tsun",
    "cloud_cover",
    "pres_dwd"
]

for col in numeric_columns:
    meteostat_to_append[col] = pd.to_numeric(
        meteostat_to_append[col],
        errors="coerce"
    ).astype("float64")

# Remove the exported CSV index column
df_existing_clean = df_existing.drop(
    columns=["Unnamed: 0"]
).copy()

# Reorder historical columns to match
meteostat_to_append = meteostat_to_append[
    df_existing_clean.columns
]

print("Existing dtypes:")
print(df_existing_clean.dtypes.to_string())

print("\nHistorical dtypes:")
print(meteostat_to_append.dtypes.to_string())

print("\nColumn order matches:",
      df_existing_clean.columns.equals(meteostat_to_append.columns))

Existing dtypes:
station_id               float64
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                     float64
cloud_cover              float64
pres_dwd                 float64
station                      str
source                       str
pres_meteostat           float64

Historical dtypes:
station_id               float64
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                     float64
cloud_cover              float64
pres_dwd                 float64
station                      str
source                       str
pres_meteostat           float64

Column order matches: True


In [89]:
# Append historical Meteostat records to the existing weather dataset

df_weather_extended = pd.concat(
    [df_existing_clean, meteostat_to_append],
    ignore_index=True
)

# Sort by station, date and source
df_weather_extended = (
    df_weather_extended
    .sort_values(["station", "date", "source"])
    .reset_index(drop=True)
)

print("Original dataset shape:", df_existing_clean.shape)
print("Historical records added:", len(meteostat_to_append))
print("Extended dataset shape:", df_weather_extended.shape)

print("\nDate range:")
print(df_weather_extended["date"].min())
print(df_weather_extended["date"].max())

print("\nRows per source:")
print(df_weather_extended["source"].value_counts())

print("\nRows per station and source:")
print(
    df_weather_extended.groupby(["station", "source"])
    .size()
    .to_string()
)

Original dataset shape: (4995, 13)
Historical records added: 4384
Extended dataset shape: (9379, 13)

Date range:
2022-01-01 00:00:00
2026-10-30 00:00:00

Rows per source:
source
Meteostat    6629
DWD          2750
Name: count, dtype: int64

Rows per station and source:
station             source   
Alexanderplatz      Meteostat    1764
Berlin Brandenburg  DWD           550
Buch                DWD           550
Dahlem              DWD           550
                    Meteostat    1399
Marzahn             DWD           550
                    Meteostat    1399
Tegel               Meteostat    1764
Tempelhof           DWD           550
                    Meteostat     303


In [90]:
# Validate the extended weather dataset

print("Duplicate station-date-source combinations:")
print(
    df_weather_extended.duplicated(
        subset=["station", "date", "source"]
    ).sum()
)

print("\nRecords per station and year:")
print(
    df_weather_extended
    .groupby([
        "station",
        "source",
        df_weather_extended["date"].dt.year
    ])
    .size()
    .to_string()
)

print("\nMissing values:")
print(
    df_weather_extended.isna().sum().to_string()
)

print("\nFinal shape:")
print(df_weather_extended.shape)

Duplicate station-date-source combinations:
0

Records per station and year:
station             source     date
Alexanderplatz      Meteostat  2022    365
                               2023    365
                               2024    366
                               2025    365
                               2026    303
Berlin Brandenburg  DWD        2025    267
                               2026    283
Buch                DWD        2025    267
                               2026    283
Dahlem              DWD        2025    267
                               2026    283
                    Meteostat  2022    365
                               2023    365
                               2024    366
                               2026    303
Marzahn             DWD        2025    267
                               2026    283
                    Meteostat  2022    365
                               2023    365
                               2024    366
                           

In [91]:
# Inspect missing values by station and source

missing_by_source = (
    df_weather_extended
    .groupby(["station", "source"])[
        ["tavg", "tmin", "tmax", "prcp", "wspd",
         "tsun", "cloud_cover", "pres_dwd", "pres_meteostat"]
    ]
    .agg(lambda col: col.isna().sum())
)

print(missing_by_source.to_string())

                              tavg  tmin  tmax  prcp  wspd  tsun  cloud_cover  pres_dwd  pres_meteostat
station            source                                                                              
Alexanderplatz     Meteostat    12    12    12  1131    12  1096         1764      1764              12
Berlin Brandenburg DWD           0     0     0     0    11   550            9        10             550
Buch               DWD           0     0     0     0   550   550          550       550             550
Dahlem             DWD           0     0     0     0   550     0            3         0             550
                   Meteostat     0     0     0     0     0  1096         1399      1399               0
Marzahn            DWD           7     7     7     7   550   550          550       550             550
                   Meteostat    19    21    21    21    10  1399         1399      1399              21
Tegel              Meteostat     0     0     0   427     0  1096

In [92]:
# Check unique calendar-day coverage per station and year

coverage = (
    df_weather_extended
    .groupby([
        "station",
        df_weather_extended["date"].dt.year
    ])["date"]
    .nunique()
    .unstack(fill_value=0)
)

print(coverage.to_string())

date                2022  2023  2024  2025  2026
station                                         
Alexanderplatz       365   365   366   365   303
Berlin Brandenburg     0     0     0   267   283
Buch                   0     0     0   267   283
Dahlem               365   365   366   267   303
Marzahn              365   365   366   267   303
Tegel                365   365   366   365   303
Tempelhof              0     0     0   267   303


In [93]:
# Find DWD-related DataFrames and inspect their date coverage

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.astype(str).tolist()

        # Look for likely DWD dataframes
        if (
            "dwd" in name.lower()
            or "MESS_DATUM" in cols
            or ("station" in cols and "date" in cols)
        ):
            date_col = next(
                (col for col in ["date", "MESS_DATUM"] if col in cols),
                None
            )

            print(f"\n{name}")
            print("Shape:", obj.shape)
            print("Columns:", cols)

            if date_col:
                dates = pd.to_datetime(
                    obj[date_col],
                    errors="coerce"
                )
                print("Date range:", dates.min(), "to", dates.max())


buch_raw
Shape: (27394, 19)
Columns: ['STATIONS_ID', 'MESS_DATUM', 'QN_3', '  FX', '  FM', 'QN_4', ' RSK', 'RSKF', ' SDK', 'SHK_TAG', '  NM', ' VPM', '  PM', ' TMK', ' UPM', ' TXK', ' TNK', ' TGK', 'eor']
Date range: 1970-01-01 00:00:00.019510101 to 1970-01-01 00:00:00.020251231

buch
Shape: (2312, 19)
Columns: ['STATIONS_ID', 'MESS_DATUM', 'QN_3', 'FX', 'FM', 'QN_4', 'RSK', 'RSKF', 'SDK', 'SHK_TAG', 'NM', 'VPM', 'PM', 'TMK', 'UPM', 'TXK', 'TNK', 'TGK', 'eor']
Date range: 2019-01-01 00:00:00 to 2025-04-30 00:00:00

dwd_clean
Shape: (5553, 10)
Columns: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres', 'station', 'source', 'year']
Date range: 2022-01-01 00:00:00 to 2025-04-30 00:00:00

group
Shape: (1216, 10)
Columns: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres', 'station', 'source', 'year']
Date range: 2022-01-01 00:00:00 to 2025-04-30 00:00:00

dwd_to_save
Shape: (5553, 9)
Columns: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres', 'station', 'source']
Date ran

In [94]:
# Audit coverage across DWD, Meteostat and the existing dataset

# Collect dates from all three datasets
coverage_parts = []

for df, source_name in [
    (dwd_clean, "DWD"),
    (meteostat_historical_clean, "Meteostat"),
    (df_existing, "existing")
]:
    temp = df[["date", "station", "source"]].copy()
    temp["date"] = pd.to_datetime(temp["date"])
    coverage_parts.append(temp)

coverage_data = pd.concat(coverage_parts, ignore_index=True)

# Keep only the requested stations and years
target_stations = [
    "Berlin Brandenburg",
    "Tegel",
    "Buch",
    "Marzahn",
    "Alexanderplatz",
    "Dahlem",
    "Tempelhof"
]

coverage_data = coverage_data[
    coverage_data["station"].isin(target_stations)
    & coverage_data["date"].dt.year.between(2022, 2025)
].copy()

coverage_data["year"] = coverage_data["date"].dt.year

# Count unique dates by station and year, regardless of source
coverage_union = (
    coverage_data
    .groupby(["station", "year"])["date"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=target_stations, columns=[2022, 2023, 2024, 2025], fill_value=0)
)

print("UNIQUE DATE COVERAGE — COMBINED SOURCES")
print(coverage_union.to_string())

# Also show coverage by source
coverage_by_source = (
    coverage_data
    .groupby(["station", "year", "source"])["date"]
    .nunique()
    .unstack(fill_value=0)
)

print("\nUNIQUE DATE COVERAGE — BY SOURCE")
print(coverage_by_source.to_string())

UNIQUE DATE COVERAGE — COMBINED SOURCES
year                2022  2023  2024  2025
station                                   
Berlin Brandenburg   365   365   366   365
Tegel                365   365   366   365
Buch                 365   365   366   365
Marzahn              365   365   366   365
Alexanderplatz       365   365   366   365
Dahlem               365   365   366   351
Tempelhof            365   365   366   365

UNIQUE DATE COVERAGE — BY SOURCE
source                   DWD  Meteostat
station            year                
Alexanderplatz     2022    0        365
                   2023    0        365
                   2024    0        366
                   2025    0        365
Berlin Brandenburg 2022  365          0
                   2023  365          0
                   2024  366          0
                   2025  365          0
Buch               2022  365          0
                   2023  365          0
                   2024  366          0
                   

In [95]:
# Check overlap between historical DWD and existing DWD records

dwd_history = dwd_clean[
    dwd_clean["date"].dt.year.between(2022, 2025)
].copy()

existing_dwd = df_existing_clean[
    df_existing_clean["source"] == "DWD"
].copy()

# Match records by station and date
overlap = dwd_history.merge(
    existing_dwd,
    on=["station", "date", "source"],
    how="inner",
    suffixes=("_historical", "_existing")
)

# Count overlapping dates per station
print("OVERLAPPING DWD RECORDS")
print(
    overlap.groupby("station")["date"]
    .nunique()
    .to_string()
)

# Count historical DWD records not yet in the existing dataset
historical_only = dwd_history.merge(
    existing_dwd[["station", "date", "source"]],
    on=["station", "date", "source"],
    how="left",
    indicator=True
)

historical_only = historical_only[
    historical_only["_merge"] == "left_only"
]

print("\nHISTORICAL DWD RECORDS NOT YET IN EXISTING DATASET")
print(
    historical_only.groupby("station")["date"]
    .nunique()
    .to_string()
)

print("\nTotal historical DWD rows:", len(dwd_history))
print("Overlapping rows:", len(overlap))
print("Historical-only rows:", len(historical_only))

OVERLAPPING DWD RECORDS
station
Berlin Brandenburg    22
Buch                  22
Dahlem                22
Marzahn               22
Tempelhof             22

HISTORICAL DWD RECORDS NOT YET IN EXISTING DATASET
station
Berlin Brandenburg    1194
Buch                  1194
Dahlem                1180
Marzahn                681
Tempelhof             1194

Total historical DWD rows: 5553
Overlapping rows: 110
Historical-only rows: 5443


In [97]:
# Compare overlapping DWD measurements (excluding pressure)

columns_to_compare = ["tavg", "tmin", "tmax", "prcp", "wspd"]

for variable in columns_to_compare:

    a = overlap[f"{variable}_historical"]
    b = overlap[f"{variable}_existing"]

    valid = a.notna() & b.notna()
    differences = ((a - b).abs() > 0.000001) & valid

    print(
        f"{variable}: {differences.sum()} differences "
        f"out of {valid.sum()} comparable records"
    )

tavg: 0 differences out of 103 comparable records
tmin: 0 differences out of 103 comparable records
tmax: 0 differences out of 103 comparable records
prcp: 0 differences out of 103 comparable records
wspd: 44 differences out of 44 comparable records


In [98]:
# Inspect wind speed differences in overlapping DWD records

wind_comparison = overlap[
    overlap["wspd_historical"].notna()
    & overlap["wspd_existing"].notna()
][
    ["station", "date", "wspd_historical", "wspd_existing"]
].copy()

wind_comparison["difference"] = (
    wind_comparison["wspd_historical"]
    - wind_comparison["wspd_existing"]
)

wind_comparison["ratio"] = (
    wind_comparison["wspd_historical"]
    / wind_comparison["wspd_existing"].replace(0, pd.NA)
)

print(wind_comparison.head(15).to_string(index=False))

           station       date  wspd_historical  wspd_existing  difference    ratio
Berlin Brandenburg 2025-04-09              3.6          12.96       -9.36 0.277778
Berlin Brandenburg 2025-04-10              5.3          19.08      -13.78 0.277778
Berlin Brandenburg 2025-04-11              6.0          21.60      -15.60 0.277778
Berlin Brandenburg 2025-04-12              2.2           7.92       -5.72 0.277778
Berlin Brandenburg 2025-04-13              4.2          15.12      -10.92 0.277778
Berlin Brandenburg 2025-04-14              2.3           8.28       -5.98 0.277778
Berlin Brandenburg 2025-04-15              5.7          20.52      -14.82 0.277778
Berlin Brandenburg 2025-04-16              3.1          11.16       -8.06 0.277778
Berlin Brandenburg 2025-04-17              3.8          13.68       -9.88 0.277778
Berlin Brandenburg 2025-04-18              4.2          15.12      -10.92 0.277778
Berlin Brandenburg 2025-04-19              4.1          14.76      -10.66 0.277778
Berl

In [99]:
# Verify DWD wind speed after conversion from m/s to km/h

wind_comparison["historical_kmh"] = (
    wind_comparison["wspd_historical"] * 3.6
)

wind_comparison["converted_difference"] = (
    wind_comparison["historical_kmh"]
    - wind_comparison["wspd_existing"]
)

print(
    "Maximum absolute difference after conversion:",
    wind_comparison["converted_difference"].abs().max()
)

print(
    "Matching records:",
    (wind_comparison["converted_difference"].abs() < 0.000001).sum(),
    "out of",
    len(wind_comparison)
)

Maximum absolute difference after conversion: 3.552713678800501e-15
Matching records: 44 out of 44


In [100]:
# Prepare historical DWD records for appending

dwd_to_append = dwd_clean[
    dwd_clean["date"].between("2022-01-01", "2025-12-31")
].copy()

# Exclude pressure and unused columns
dwd_to_append = dwd_to_append[
    ["date", "tavg", "tmin", "tmax", "prcp", "wspd", "station", "source"]
].copy()

# Convert DWD wind speed from m/s to km/h
dwd_to_append["wspd"] = dwd_to_append["wspd"] * 3.6

# Remove dates already present in the existing DWD dataset
existing_dwd_dates = df_existing_clean[
    df_existing_clean["source"] == "DWD"
][["station", "date"]].drop_duplicates()

dwd_to_append = dwd_to_append.merge(
    existing_dwd_dates,
    on=["station", "date"],
    how="left",
    indicator=True
)

dwd_to_append = dwd_to_append[
    dwd_to_append["_merge"] == "left_only"
].drop(columns="_merge")

print("Historical DWD rows to append:", len(dwd_to_append))
print("\nRows per station:")
print(dwd_to_append.groupby("station").size().to_string())

print("\nDate range:")
print(dwd_to_append["date"].min(), "to", dwd_to_append["date"].max())

Historical DWD rows to append: 5443

Rows per station:
station
Berlin Brandenburg    1194
Buch                  1194
Dahlem                1180
Marzahn                681
Tempelhof             1194

Date range:
2022-01-01 00:00:00 to 2025-04-08 00:00:00


In [101]:
# Align historical DWD records with the existing dataset

dwd_to_append["station_id"] = pd.NA
dwd_to_append["tsun"] = pd.NA
dwd_to_append["cloud_cover"] = pd.NA
dwd_to_append["pres_dwd"] = pd.NA
dwd_to_append["pres_meteostat"] = pd.NA

# Match the existing dataset's column order
dwd_to_append = dwd_to_append[
    df_existing_clean.columns
].copy()

# Convert additional columns to numeric types
numeric_columns = [
    "station_id",
    "tsun",
    "cloud_cover",
    "pres_dwd",
    "pres_meteostat"
]

for col in numeric_columns:
    dwd_to_append[col] = pd.to_numeric(
        dwd_to_append[col],
        errors="coerce"
    ).astype("float64")

print("Shape:", dwd_to_append.shape)
print("\nColumn alignment:")
print(dwd_to_append.dtypes)

Shape: (5443, 13)

Column alignment:
station_id               float64
date              datetime64[us]
tavg                     float64
tmin                     float64
tmax                     float64
prcp                     float64
wspd                     float64
tsun                     float64
cloud_cover              float64
pres_dwd                 float64
station                      str
source                       str
pres_meteostat           float64
dtype: object


In [102]:
# Append historical DWD records to the extended dataset

df_weather_final = pd.concat(
    [df_weather_extended, dwd_to_append],
    ignore_index=True
)

# Sort by station, date and source
df_weather_final = (
    df_weather_final
    .sort_values(["station", "date", "source"])
    .reset_index(drop=True)
)

print("Final dataset shape:", df_weather_final.shape)
print("\nDate range:")
print(df_weather_final["date"].min(), "to", df_weather_final["date"].max())

print("\nRows per source:")
print(df_weather_final["source"].value_counts().to_string())

print("\nRows per station and source:")
print(
    df_weather_final.groupby(["station", "source"])
    .size()
    .to_string()
)

Final dataset shape: (14822, 13)

Date range:
2022-01-01 00:00:00 to 2026-10-30 00:00:00

Rows per source:
source
DWD          8193
Meteostat    6629

Rows per station and source:
station             source   
Alexanderplatz      Meteostat    1764
Berlin Brandenburg  DWD          1744
Buch                DWD          1744
Dahlem              DWD          1730
                    Meteostat    1399
Marzahn             DWD          1231
                    Meteostat    1399
Tegel               Meteostat    1764
Tempelhof           DWD          1744
                    Meteostat     303


In [103]:
# Final validation of the merged weather dataset

target_stations = [
    "Berlin Brandenburg",
    "Tegel",
    "Buch",
    "Marzahn",
    "Alexanderplatz",
    "Dahlem",
    "Tempelhof"
]

# 1. Unique calendar dates per station and year
coverage_final = (
    df_weather_final
    .groupby([
        "station",
        df_weather_final["date"].dt.year
    ])["date"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        index=target_stations,
        columns=[2022, 2023, 2024, 2025, 2026],
        fill_value=0
    )
)

print("UNIQUE DATE COVERAGE")
print(coverage_final.to_string())

# 2. Duplicate station-date-source combinations
duplicates = df_weather_final.duplicated(
    subset=["station", "date", "source"]
).sum()

print("\nDuplicate station-date-source combinations:", duplicates)

# 3. Missing requested stations
missing_stations = set(target_stations) - set(
    df_weather_final["station"].unique()
)

print("Missing requested stations:", missing_stations)

# 4. Missing values in core weather variables
print("\nMissing values in core variables:")
print(
    df_weather_final[
        ["tavg", "tmin", "tmax", "prcp", "wspd"]
    ].isna().sum().to_string()
)

UNIQUE DATE COVERAGE
date                2022  2023  2024  2025  2026
station                                         
Berlin Brandenburg   365   365   366   365   283
Tegel                365   365   366   365   303
Buch                 365   365   366   365   283
Marzahn              365   365   366   365   303
Alexanderplatz       365   365   366   365   303
Dahlem               365   365   366   351   303
Tempelhof            365   365   366   365   303

Duplicate station-date-source combinations: 0
Missing requested stations: set()

Missing values in core variables:
tavg      40
tmin      43
tmax      43
prcp    1599
wspd    4762


In [104]:
# Save the final merged weather dataset

output_path = "df_weather_combined_2022_2026.csv"

df_weather_final.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", output_path)
print("Rows:", len(df_weather_final))
print("Columns:", len(df_weather_final.columns))

Saved: df_weather_combined_2022_2026.csv
Rows: 14822
Columns: 13


In [ ]:
pd.read_csv("merged_complete_datasets/df_weather_combined_2022_2026.csv")

Getting historical air quality data

In [15]:
import pandas as pd

df_air_quality = pd.read_csv("merged_complete_datasets/berlin_air_quality_master_2025_2026.csv")

In [17]:
# Get the unique station ID, name and station type combinations

air_stations = (
    df_air_quality[
        ["core", "station_name", "station_type"]
    ]
    .drop_duplicates()
    .sort_values("core")
    .reset_index(drop=True)
)

display(air_stations)

,core,station_name,station_type
0,co,Neukölln,Urban background
1,co,Schildhornstraße,Traffic
2,co,Frankfurter Allee,Traffic
3,ebc,Wedding,Urban background
4,ebc,Neukölln,Urban background
...,...,...,...
79,pm2,Silbersteinstraße,Traffic
80,pm2,Mitte,Urban background
81,pm2,Frankfurter Allee,Traffic
82,pm2,Leipziger Straße,Traffic


In [18]:
# Extract the unique monitoring stations

air_stations_unique = (
    df_air_quality[
        ["station_name", "station_type"]
    ]
    .drop_duplicates()
    .sort_values("station_name")
    .reset_index(drop=True)
)

display(air_stations_unique)

print("Number of unique stations:", len(air_stations_unique))

,station_name,station_type
0,Buch,Outskirts
1,Frankfurter Allee,Traffic
2,Friedrichshagen,Outskirts
3,Frohnau,Outskirts
4,Grunewald,Outskirts
5,Karl-Marx-Straße,Traffic
6,Karlshorst,Urban background
7,Leipziger Straße,Traffic
8,Mariendorfer Damm,Traffic
9,Marienfelde,Outskirts


Number of unique stations: 15


In [19]:
# Retrieve the station IDs used in the original dataset

air_station_mapping = (
    df_air_quality[
        ["station", "station_name", "station_type"]
    ]
    .drop_duplicates()
    .sort_values("station")
    .reset_index(drop=True)
)

display(air_station_mapping)

print("Number of unique stations:", len(air_station_mapping))

,station,station_name,station_type
0,mc010,Wedding,Urban background
1,mc027,Marienfelde,Outskirts
2,mc032,Grunewald,Outskirts
3,mc042,Neukölln,Urban background
4,mc077,Buch,Outskirts
5,mc085,Friedrichshagen,Outskirts
6,mc117,Schildhornstraße,Traffic
7,mc124,Mariendorfer Damm,Traffic
8,mc144,Silbersteinstraße,Traffic
9,mc145,Frohnau,Outskirts


Number of unique stations: 15


In [20]:
# Test the Berlin air-quality API for historical daily NO2 data

import requests

url = "https://luftdaten.berlin.de/api/stations/mc010/data"

params = {
    "core": "no2",
    "period": "24h",
    "timespan": "all"
}

response = requests.get(url, params=params, timeout=60)

print("Status code:", response.status_code)
print("Request URL:", response.url)

test_data = response.json()

print("Number of records:", len(test_data))

if test_data:
    display(pd.DataFrame(test_data).head())
    print("First date:", test_data[-1]["datetime"])
    print("Last date:", test_data[0]["datetime"])

Status code: 200
Request URL: https://luftdaten.berlin.de/api/stations/mc010/data?core=no2&period=24h&timespan=all
Number of records: 364


,datetime,station,core,component,period,value
0,2026-11-08T00:00:00+01:00,mc010,no2,no2_24h,24h,20.0
1,2026-11-07T00:00:00+01:00,mc010,no2,no2_24h,24h,20.0
2,2026-11-06T00:00:00+01:00,mc010,no2,no2_24h,24h,25.0
3,2026-11-05T00:00:00+01:00,mc010,no2,no2_24h,24h,31.0
4,2026-11-04T00:00:00+01:00,mc010,no2,no2_24h,24h,15.0


First date: 2025-11-10T00:00:00+01:00
Last date: 2026-11-08T00:00:00+01:00


In [21]:
# Test a custom date range for historical NO2 data

params = {
    "core": "no2",
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2019",
    "end[date]": "31.12.2019"
}

response = requests.get(url, params=params, timeout=60)

print("Status code:", response.status_code)
print("Request URL:", response.url)

historical_test = response.json()

print("Number of records:", len(historical_test))

if historical_test:
    historical_df = pd.DataFrame(historical_test)
    display(historical_df.head())
    print("Earliest:", historical_df["datetime"].min())
    print("Latest:", historical_df["datetime"].max())
else:
    print("No records returned.")

Status code: 200
Request URL: https://luftdaten.berlin.de/api/stations/mc010/data?core=no2&period=24h&timespan=custom&start%5Bdate%5D=01.01.2019&end%5Bdate%5D=31.12.2019
Number of records: 365


,datetime,station,core,component,period,value
0,2019-12-31T00:00:00+01:00,mc010,no2,no2_24h,24h,35.0
1,2019-12-30T00:00:00+01:00,mc010,no2,no2_24h,24h,40.0
2,2019-12-29T00:00:00+01:00,mc010,no2,no2_24h,24h,33.0
3,2019-12-28T00:00:00+01:00,mc010,no2,no2_24h,24h,22.0
4,2019-12-27T00:00:00+01:00,mc010,no2,no2_24h,24h,18.0


Earliest: 2019-01-01T00:00:00+01:00
Latest: 2019-12-31T00:00:00+01:00


In [22]:
# Check whether Wedding has NO2 data in 2018

params = {
    "core": "no2",
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2018",
    "end[date]": "31.12.2018"
}

response = requests.get(url, params=params, timeout=60)
data_2018 = response.json()

print("Status code:", response.status_code)
print("Records in 2018:", len(data_2018))

if data_2018:
    df_2018 = pd.DataFrame(data_2018)
    print("First available date:", df_2018["datetime"].min())
    print("Last available date:", df_2018["datetime"].max())
else:
    print("No records returned for 2018.")

Status code: 200
Records in 2018: 365
First available date: 2018-01-01T00:00:00+01:00
Last available date: 2018-12-31T00:00:00+01:00


In [44]:
# Check 2018 historical availability across all stations and pollutants

pollutants = ["no2", "o3", "pm10", "pm2"]

availability_2022 = []

for _, station_row in air_station_mapping.iterrows():

    station_id = station_row["station"]
    station_name = station_row["station_name"]

    for pollutant in pollutants:

        params = {
            "core": pollutant,
            "period": "24h",
            "timespan": "custom",
            "start[date]": "01.01.2022",
            "end[date]": "31.12.2022"
        }

        station_url = (
            f"https://luftdaten.berlin.de/api/stations/{station_id}/data"
        )

        response = requests.get(
            station_url,
            params=params,
            timeout=60
        )

        data = response.json()

        availability_2022.append({
            "station": station_id,
            "station_name": station_name,
            "pollutant": pollutant,
            "records": len(data),
            "first_date": (
                min(row["datetime"] for row in data)
                if data else None
            ),
            "last_date": (
                max(row["datetime"] for row in data)
                if data else None
            )
        })

availability_2022_df = pd.DataFrame(availability_2022)

display(
    availability_2022_df.sort_values(
        ["station_name", "pollutant"]
    ).reset_index(drop=True)
)

,station,station_name,pollutant,records,first_date,last_date
0,mc077,Buch,no2,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
1,mc077,Buch,o3,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
2,mc077,Buch,pm10,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
3,mc077,Buch,pm2,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
4,mc174,Frankfurter Allee,no2,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
5,mc174,Frankfurter Allee,o3,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
6,mc174,Frankfurter Allee,pm10,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
7,mc174,Frankfurter Allee,pm2,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
8,mc085,Friedrichshagen,no2,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00
9,mc085,Friedrichshagen,o3,365,2022-01-01T00:00:00+01:00,2022-12-31T00:00:00+01:00


In [43]:
import requests

test_params = {
    "core": "o3",
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2019",
    "end[date]": "31.12.2019"
}

try:
    response = requests.get(
        "https://luftdaten.berlin.de/api/stations/mc010/data",
        params=test_params,
        timeout=120
    )
    response.raise_for_status()

    print("Status:", response.status_code)
    print("Records returned:", len(response.json()))

except requests.exceptions.RequestException as e:
    print(type(e).__name__, e)

Status: 200
Records returned: 365


In [106]:
# Test Berlin air-quality API: O3, daily values, first week of 2022

import requests
import pandas as pd

url = "https://luftdaten.berlin.de/api/stations/mc010/data"

params = {
    "core": "o3",
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2022",
    "end[date]": "07.01.2022"
}

try:
    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    print("HTTP status:", response.status_code)
    response.raise_for_status()

    data = response.json()

    print("Records returned:", len(data))

    df_api_test = pd.json_normalize(data)

    print("\nColumns:")
    print(df_api_test.columns.tolist())

    print("\nReturned data:")
    print(df_api_test.to_string(index=False))

except requests.exceptions.RequestException as e:
    print("API request failed:", e)

except ValueError as e:
    print("Could not parse API response as JSON:", e)

HTTP status: 200
Records returned: 7

Columns:
['datetime', 'station', 'core', 'component', 'period', 'value']

Returned data:
                 datetime station core component period  value
2022-01-07T00:00:00+01:00   mc010   o3    o3_24h    24h   26.0
2022-01-06T00:00:00+01:00   mc010   o3    o3_24h    24h   31.0
2022-01-05T00:00:00+01:00   mc010   o3    o3_24h    24h   44.0
2022-01-04T00:00:00+01:00   mc010   o3    o3_24h    24h   37.0
2022-01-03T00:00:00+01:00   mc010   o3    o3_24h    24h   56.0
2022-01-02T00:00:00+01:00   mc010   o3    o3_24h    24h   38.0
2022-01-01T00:00:00+01:00   mc010   o3    o3_24h    24h   34.0


In [107]:
# Test historical API availability by station and pollutant

import requests
import pandas as pd
import time

stations = [
    "mc010", "mc027", "mc032", "mc042", "mc077",
    "mc085", "mc117", "mc124", "mc144", "mc145",
    "mc171", "mc174", "mc190", "mc221", "mc282"
]

pollutants = ["no2", "o3", "pm10", "pm2.5"]

results = []

url = "https://luftdaten.berlin.de/api/stations/{}/data"

for station in stations:
    for pollutant in pollutants:

        params = {
            "core": pollutant,
            "period": "24h",
            "timespan": "custom",
            "start[date]": "01.01.2022",
            "end[date]": "31.01.2022"
        }

        try:
            response = requests.get(
                url.format(station),
                params=params,
                timeout=30
            )

            response.raise_for_status()
            data = response.json()

            results.append({
                "station": station,
                "pollutant": pollutant,
                "records": len(data),
                "status": "OK"
            })

        except Exception as e:
            results.append({
                "station": station,
                "pollutant": pollutant,
                "records": 0,
                "status": str(e)[:100]
            })

        time.sleep(0.2)

df_api_coverage_test = pd.DataFrame(results)

print(df_api_coverage_test.to_string(index=False))

station pollutant  records status
  mc010       no2       31     OK
  mc010        o3       31     OK
  mc010      pm10       31     OK
  mc010     pm2.5      186     OK
  mc027       no2       31     OK
  mc027        o3       31     OK
  mc027      pm10        0     OK
  mc027     pm2.5      124     OK
  mc032       no2       31     OK
  mc032        o3       31     OK
  mc032      pm10       31     OK
  mc032     pm2.5      186     OK
  mc042       no2       31     OK
  mc042        o3       31     OK
  mc042      pm10       31     OK
  mc042     pm2.5      279     OK
  mc077       no2       31     OK
  mc077        o3       31     OK
  mc077      pm10       31     OK
  mc077     pm2.5      186     OK
  mc085       no2       31     OK
  mc085        o3       31     OK
  mc085      pm10       31     OK
  mc085     pm2.5      186     OK
  mc117       no2       31     OK
  mc117        o3        0     OK
  mc117      pm10       31     OK
  mc117     pm2.5      186     OK
  mc124       

In [108]:
# Inspect multiple API responses and measurement components

tests = [
    ("mc010", "pm2.5"),
    ("mc042", "pm2.5"),
    ("mc027", "pm10"),
    ("mc117", "o3"),
    ("mc144", "no2")
]

for station, pollutant in tests:

    params = {
        "core": pollutant,
        "period": "24h",
        "timespan": "custom",
        "start[date]": "01.01.2022",
        "end[date]": "07.01.2022"
    }

    response = requests.get(
        f"https://luftdaten.berlin.de/api/stations/{station}/data",
        params=params,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    df_test = pd.json_normalize(data)

    print(f"\n{'=' * 45}")
    print(f"Station: {station} | Pollutant: {pollutant}")
    print("Records:", len(df_test))

    if not df_test.empty:
        print("\nComponents:")
        print(df_test["component"].value_counts().to_string())

        print("\nPeriods:")
        print(df_test["period"].value_counts().to_string())

        print("\nRecords per date:")
        print(
            pd.to_datetime(df_test["datetime"])
            .dt.date
            .value_counts()
            .sort_index()
            .to_string()
        )

        print("\nSample:")
        print(df_test.head(5).to_string(index=False))


Station: mc010 | Pollutant: pm2.5
Records: 42

Components:
component
pm10_24h    7
pm2_24h     7
no2_24h     7
no_24h      7
nox_24h     7
o3_24h      7

Periods:
period
24h    42

Records per date:
datetime
2022-01-01    6
2022-01-02    6
2022-01-03    6
2022-01-04    6
2022-01-05    6
2022-01-06    6
2022-01-07    6

Sample:
                 datetime station core component period  value
2022-01-07T00:00:00+01:00   mc010 pm10  pm10_24h    24h   13.0
2022-01-07T00:00:00+01:00   mc010  pm2   pm2_24h    24h   11.0
2022-01-07T00:00:00+01:00   mc010  no2   no2_24h    24h   24.0
2022-01-07T00:00:00+01:00   mc010   no    no_24h    24h    7.0
2022-01-07T00:00:00+01:00   mc010  nox   nox_24h    24h   35.0

Station: mc042 | Pollutant: pm2.5
Records: 63

Components:
component
pm10_24h    7
pm2_24h     7
no2_24h     7
no_24h      7
nox_24h     7
o3_24h      7
chb_24h     7
cht_24h     7
co_24h      7

Periods:
period
24h    63

Records per date:
datetime
2022-01-01    9
2022-01-02    9
2022-01-0

In [109]:
# Compare API responses for different core parameters
station = "mc010"
date_params = {
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2022",
    "end[date]": "07.01.2022"
}

for pollutant in ["no2", "o3", "pm10", "pm2.5"]:
    params = {**date_params, "core": pollutant}

    response = requests.get(
        f"https://luftdaten.berlin.de/api/stations/{station}/data",
        params=params,
        timeout=30
    )
    response.raise_for_status()

    df_check = pd.json_normalize(response.json())

    print(f"\nRequested core: {pollutant}")
    print("Records:", len(df_check))
    print("Returned components:", sorted(df_check["component"].dropna().unique()))


Requested core: no2
Records: 7
Returned components: ['no2_24h']

Requested core: o3
Records: 7
Returned components: ['o3_24h']

Requested core: pm10
Records: 7
Returned components: ['pm10_24h']

Requested core: pm2.5
Records: 42
Returned components: ['no2_24h', 'no_24h', 'nox_24h', 'o3_24h', 'pm10_24h', 'pm2_24h']


In [110]:
# Test the PM2.5 endpoint and filter for the actual PM2.5 component

station = "mc010"

params = {
    "core": "pm2.5",
    "period": "24h",
    "timespan": "custom",
    "start[date]": "01.01.2022",
    "end[date]": "07.01.2022"
}

response = requests.get(
    f"https://luftdaten.berlin.de/api/stations/{station}/data",
    params=params,
    timeout=30
)

response.raise_for_status()

df_test = pd.json_normalize(response.json())

# Keep only the PM2.5 component
df_pm25_test = df_test[
    df_test["component"] == "pm2_24h"
].copy()

print("Total API records:", len(df_test))
print("PM2.5 records after filtering:", len(df_pm25_test))
print("\nFiltered records:")
print(df_pm25_test.to_string(index=False))

Total API records: 42
PM2.5 records after filtering: 7

Filtered records:
                 datetime station core component period  value
2022-01-07T00:00:00+01:00   mc010  pm2   pm2_24h    24h   11.0
2022-01-06T00:00:00+01:00   mc010  pm2   pm2_24h    24h    5.0
2022-01-05T00:00:00+01:00   mc010  pm2   pm2_24h    24h    5.0
2022-01-04T00:00:00+01:00   mc010  pm2   pm2_24h    24h    5.0
2022-01-03T00:00:00+01:00   mc010  pm2   pm2_24h    24h    5.0
2022-01-02T00:00:00+01:00   mc010  pm2   pm2_24h    24h    8.0
2022-01-01T00:00:00+01:00   mc010  pm2   pm2_24h    24h   15.0


In [111]:
# Find existing air-quality DataFrames in the notebook

for name, obj in globals().copy().items():
    if (
        isinstance(obj, pd.DataFrame)
        and ("air" in name.lower() or "quality" in name.lower())
    ):
        print(f"\nDataFrame: {name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())

        if "datetime" in obj.columns:
            print("Date range:", obj["datetime"].min(), "to", obj["datetime"].max())


DataFrame: df_air_quality
Shape: (54305, 8)
Columns: ['datetime', 'station', 'core', 'component', 'period', 'value', 'station_name', 'station_type']
Date range: 2024-12-31 23:00:00+00:00 to 2026-10-20 22:00:00+00:00

DataFrame: air_stations
Shape: (84, 3)
Columns: ['core', 'station_name', 'station_type']

DataFrame: air_stations_unique
Shape: (15, 2)
Columns: ['station_name', 'station_type']

DataFrame: air_station_mapping
Shape: (15, 3)
Columns: ['station', 'station_name', 'station_type']


In [112]:
# Inspect existing station and pollutant coverage

print("Datetime dtype:", df_air_quality["datetime"].dtype)

print("\nUnique stations:", df_air_quality["station"].nunique())
print(sorted(df_air_quality["station"].dropna().unique()))

print("\nComponents:")
print(df_air_quality["component"].value_counts(dropna=False))

print("\nStation mapping:")
print(air_station_mapping.sort_values("station").to_string(index=False))

print("\nMissing station names:", df_air_quality["station_name"].isna().sum())
print("Missing station types:", df_air_quality["station_type"].isna().sum())

print("\nDuplicate measurement keys:")
print(
    df_air_quality.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)

Datetime dtype: str

Unique stations: 15
['mc010', 'mc027', 'mc032', 'mc042', 'mc077', 'mc085', 'mc117', 'mc124', 'mc144', 'mc145', 'mc171', 'mc174', 'mc190', 'mc221', 'mc282']

Components:
component
no2_24h     9735
no_24h      9735
nox_24h     9735
pm10_24h    7758
pm2_24h     7758
o3_24h      5212
ebc_24h     2455
co_24h      1917
Name: count, dtype: int64

Station mapping:
station      station_name     station_type
  mc010           Wedding Urban background
  mc027       Marienfelde        Outskirts
  mc032         Grunewald        Outskirts
  mc042          Neukölln Urban background
  mc077              Buch        Outskirts
  mc085   Friedrichshagen        Outskirts
  mc117  Schildhornstraße          Traffic
  mc124 Mariendorfer Damm          Traffic
  mc144 Silbersteinstraße          Traffic
  mc145           Frohnau        Outskirts
  mc171             Mitte Urban background
  mc174 Frankfurter Allee          Traffic
  mc190  Leipziger Straße          Traffic
  mc221  Karl-Marx

In [113]:
# Inspect existing datetime values and their format

print("First 5 timestamps:")
print(df_air_quality["datetime"].head().to_list())

print("\nLast 5 timestamps:")
print(df_air_quality["datetime"].tail().to_list())

print("\nExample timestamp types:")
print(type(df_air_quality["datetime"].iloc[0]))

print("\nUnique periods:")
print(df_air_quality["period"].value_counts(dropna=False))

First 5 timestamps:
['2024-12-31 23:00:00+00:00', '2024-12-31 23:00:00+00:00', '2024-12-31 23:00:00+00:00', '2024-12-31 23:00:00+00:00', '2024-12-31 23:00:00+00:00']

Last 5 timestamps:
['2026-10-19 22:00:00+00:00', '2026-10-19 22:00:00+00:00', '2026-10-20 22:00:00+00:00', '2026-10-20 22:00:00+00:00', '2026-10-20 22:00:00+00:00']

Example timestamp types:
<class 'str'>

Unique periods:
period
24h    54305
Name: count, dtype: int64


In [114]:
# Convert existing timestamps to UTC-aware datetimes
existing_dates = pd.to_datetime(
    df_air_quality["datetime"],
    utc=True
)

# Display the first and last dates in Berlin local time
print("First date (Berlin):")
print(existing_dates.min().tz_convert("Europe/Berlin"))

print("\nLast date (Berlin):")
print(existing_dates.max().tz_convert("Europe/Berlin"))

print("\nRecords in 2025:")
print((existing_dates.dt.year == 2025).sum())

First date (Berlin):
2025-01-01 00:00:00+01:00

Last date (Berlin):
2026-10-21 00:00:00+02:00

Records in 2025:
30419


In [115]:
# Historical air-quality download configuration

stations = sorted(df_air_quality["station"].unique())

pollutants = {
    "no2": "no2_24h",
    "o3": "o3_24h",
    "pm10": "pm10_24h",
    "pm2.5": "pm2_24h"
}

start_date = "01.01.2022"
end_date = "31.12.2024"

print("Stations:", len(stations))
print(stations)

print("\nPollutants and component filters:")
for pollutant, component in pollutants.items():
    print(f"{pollutant}: {component}")

print("\nHistorical period:", start_date, "to", end_date)
print("\nExpected API combinations:", len(stations) * len(pollutants))

Stations: 15
['mc010', 'mc027', 'mc032', 'mc042', 'mc077', 'mc085', 'mc117', 'mc124', 'mc144', 'mc145', 'mc171', 'mc174', 'mc190', 'mc221', 'mc282']

Pollutants and component filters:
no2: no2_24h
o3: o3_24h
pm10: pm10_24h
pm2.5: pm2_24h

Historical period: 01.01.2022 to 31.12.2024

Expected API combinations: 60


In [116]:
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configure a session with automatic retries
session = requests.Session()

retries = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session.mount("https://", HTTPAdapter(max_retries=retries))

# Function to download one station, one pollutant and one date range
def download_air_quality(station, pollutant, start, end):

    params = {
        "core": pollutant,
        "period": "24h",
        "timespan": "custom",
        "start[date]": start,
        "end[date]": end
    }

    url = f"https://luftdaten.berlin.de/api/stations/{station}/data"

    response = session.get(
        url,
        params=params,
        timeout=(10, 45)
    )

    response.raise_for_status()

    data = response.json()

    if not data:
        return pd.DataFrame()

    df = pd.json_normalize(data)

    # Keep only the requested pollutant component
    df = df[df["component"] == pollutants[pollutant]].copy()

    # Convert timestamps to UTC, matching the existing dataset
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        utc=True
    )

    return df

# Test January 2022 for one station
test_results = []

for pollutant in pollutants:

    df_test = download_air_quality(
        station="mc010",
        pollutant=pollutant,
        start="01.01.2022",
        end="31.01.2022"
    )

    test_results.append({
        "station": "mc010",
        "pollutant": pollutant,
        "records": len(df_test),
        "components": df_test["component"].unique().tolist()
        if not df_test.empty else []
    })

    time.sleep(0.2)

pd.DataFrame(test_results)

,station,pollutant,records,components
0,mc010,no2,31,[no2_24h]
1,mc010,o3,31,[o3_24h]
2,mc010,pm10,31,[pm10_24h]
3,mc010,pm2.5,31,[pm2_24h]


In [117]:
# Test January 2022 across all stations and pollutants

january_results = []
january_data = []

for station in stations:
    for pollutant in pollutants:

        try:
            df_temp = download_air_quality(
                station=station,
                pollutant=pollutant,
                start="01.01.2022",
                end="31.01.2022"
            )

            january_data.append(df_temp)

            january_results.append({
                "station": station,
                "pollutant": pollutant,
                "records": len(df_temp),
                "status": "OK"
            })

        except Exception as e:
            january_results.append({
                "station": station,
                "pollutant": pollutant,
                "records": 0,
                "status": str(e)
            })

        time.sleep(0.2)

df_january_coverage = pd.DataFrame(january_results)

print("Total combinations:", len(df_january_coverage))
print("\nStatus summary:")
print(df_january_coverage["status"].value_counts())

print("\nRecords by station and pollutant:")
print(
    df_january_coverage.pivot(
        index="station",
        columns="pollutant",
        values="records"
    ).to_string()
)

# Keep the successfully downloaded January records for later validation
df_january_test = (
    pd.concat(january_data, ignore_index=True)
    if january_data else pd.DataFrame()
)

print("\nTotal January records:", len(df_january_test))

Total combinations: 60

Status summary:
status
OK    60
Name: count, dtype: int64

Records by station and pollutant:
pollutant  no2  o3  pm10  pm2.5
station                        
mc010       31  31    31     31
mc027       31  31     0      0
mc032       31  31    31     31
mc042       31  31    31     31
mc077       31  31    31     31
mc085       31  31    31     31
mc117       31   0    31     31
mc124       31   0    31     31
mc144        0   0     0      0
mc145       31  31     0      0
mc171       31   0    31     31
mc174       31  31    31     31
mc190       31   0    31     31
mc221       31   0    31     31
mc282       31   0     0      0

Total January records: 1364


In [118]:
# Check duplicate measurement keys in January test data

duplicate_keys = df_january_test.duplicated(
    subset=["datetime", "station", "component", "period"]
)

print("January records:", len(df_january_test))
print("Duplicate records:", duplicate_keys.sum())

print("\nRecords per station:")
print(df_january_test.groupby("station").size().sort_values())

print("\nRecords per component:")
print(df_january_test.groupby("component").size())

January records: 1364
Duplicate records: 0

Records per station:
station
mc282     31
mc027     62
mc145     62
mc124     93
mc171     93
mc190     93
mc117     93
mc221     93
mc077    124
mc085    124
mc032    124
mc042    124
mc010    124
mc174    124
dtype: int64

Records per component:
component
no2_24h     434
o3_24h      248
pm10_24h    341
pm2_24h     341
dtype: int64


In [119]:
import calendar

# Start with the January data already downloaded and validated
air_quality_2022_parts = [df_january_test.copy()]

# Preserve the January request log
air_quality_2022_log = []

for _, row in df_january_coverage.iterrows():
    air_quality_2022_log.append({
        "year": 2022,
        "month": 1,
        "station": row["station"],
        "pollutant": row["pollutant"],
        "records": row["records"],
        "status": row["status"]
    })

# Download February to December 2022
for month in range(2, 13):

    start = f"01.{month:02d}.2022"
    last_day = calendar.monthrange(2022, month)[1]
    end = f"{last_day:02d}.{month:02d}.2022"

    print(f"\nDownloading {month:02d}/2022: {start} to {end}")

    for station in stations:
        for pollutant in pollutants:

            try:
                df_temp = download_air_quality(
                    station=station,
                    pollutant=pollutant,
                    start=start,
                    end=end
                )

                air_quality_2022_parts.append(df_temp)

                air_quality_2022_log.append({
                    "year": 2022,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": len(df_temp),
                    "status": "OK"
                })

            except Exception as e:
                air_quality_2022_log.append({
                    "year": 2022,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": 0,
                    "status": f"ERROR: {str(e)}"
                })

            time.sleep(0.2)

# Combine the monthly results
df_air_quality_2022 = pd.concat(
    air_quality_2022_parts,
    ignore_index=True
)

df_air_quality_2022_log = pd.DataFrame(air_quality_2022_log)

print("\n2022 download completed.")
print("Dataset shape:", df_air_quality_2022.shape)
print("\nRequest status:")
print(df_air_quality_2022_log["status"].value_counts())

print("\nTotal records:", len(df_air_quality_2022))
print("Duplicate measurement keys:",
      df_air_quality_2022.duplicated(
          subset=["datetime", "station", "component", "period"]
      ).sum())













2022 download completed.
Dataset shape: (16019, 6)

Request status:
status
OK    720
Name: count, dtype: int64

Total records: 16019
Duplicate measurement keys: 0


In [120]:
import calendar

air_quality_2023_parts = []
air_quality_2023_log = []

for month in range(1, 13):

    start = f"01.{month:02d}.2023"
    last_day = calendar.monthrange(2023, month)[1]
    end = f"{last_day:02d}.{month:02d}.2023"

    print(f"\nDownloading {month:02d}/2023: {start} to {end}")

    for station in stations:
        for pollutant in pollutants:

            try:
                df_temp = download_air_quality(
                    station=station,
                    pollutant=pollutant,
                    start=start,
                    end=end
                )

                air_quality_2023_parts.append(df_temp)

                air_quality_2023_log.append({
                    "year": 2023,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": len(df_temp),
                    "status": "OK"
                })

            except Exception as e:
                air_quality_2023_log.append({
                    "year": 2023,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": 0,
                    "status": f"ERROR: {str(e)}"
                })

            time.sleep(0.2)

df_air_quality_2023 = pd.concat(
    air_quality_2023_parts,
    ignore_index=True
)

df_air_quality_2023_log = pd.DataFrame(air_quality_2023_log)

print("\n2023 download completed.")
print("Dataset shape:", df_air_quality_2023.shape)

print("\nRequest status:")
print(df_air_quality_2023_log["status"].value_counts())

print("\nTotal records:", len(df_air_quality_2023))

print(
    "Duplicate measurement keys:",
    df_air_quality_2023.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)














2023 download completed.
Dataset shape: (17083, 6)

Request status:
status
OK    720
Name: count, dtype: int64

Total records: 17083
Duplicate measurement keys: 0


In [121]:
import calendar

air_quality_2024_parts = []
air_quality_2024_log = []

for month in range(1, 13):

    start = f"01.{month:02d}.2024"
    last_day = calendar.monthrange(2024, month)[1]
    end = f"{last_day:02d}.{month:02d}.2024"

    print(f"\nDownloading {month:02d}/2024: {start} to {end}")

    for station in stations:
        for pollutant in pollutants:

            try:
                df_temp = download_air_quality(
                    station=station,
                    pollutant=pollutant,
                    start=start,
                    end=end
                )

                air_quality_2024_parts.append(df_temp)

                air_quality_2024_log.append({
                    "year": 2024,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": len(df_temp),
                    "status": "OK"
                })

            except Exception as e:
                air_quality_2024_log.append({
                    "year": 2024,
                    "month": month,
                    "station": station,
                    "pollutant": pollutant,
                    "records": 0,
                    "status": f"ERROR: {str(e)}"
                })

            time.sleep(0.2)

df_air_quality_2024 = pd.concat(
    air_quality_2024_parts,
    ignore_index=True
)

df_air_quality_2024_log = pd.DataFrame(air_quality_2024_log)

print("\n2024 download completed.")
print("Dataset shape:", df_air_quality_2024.shape)

print("\nRequest status:")
print(df_air_quality_2024_log["status"].value_counts())

print("\nTotal records:", len(df_air_quality_2024))

print(
    "Duplicate measurement keys:",
    df_air_quality_2024.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)














2024 download completed.
Dataset shape: (17195, 6)

Request status:
status
OK    720
Name: count, dtype: int64

Total records: 17195
Duplicate measurement keys: 0


In [122]:
# Combine the three historical years
df_air_quality_historical = pd.concat(
    [
        df_air_quality_2022,
        df_air_quality_2023,
        df_air_quality_2024
    ],
    ignore_index=True
)

# Add station names and types
df_air_quality_historical = df_air_quality_historical.merge(
    air_station_mapping,
    on="station",
    how="left",
    validate="many_to_one"
)

# Match the existing dataset's column order
df_air_quality_historical = df_air_quality_historical[
    df_air_quality.columns
]

# Validate historical dataset
print("Shape:", df_air_quality_historical.shape)

print("\nDate range:")
print(df_air_quality_historical["datetime"].min())
print(df_air_quality_historical["datetime"].max())

print("\nData types:")
print(df_air_quality_historical.dtypes)

print("\nRecords per year:")
print(
    df_air_quality_historical["datetime"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nComponents:")
print(df_air_quality_historical["component"].value_counts())

print("\nMissing station names:",
      df_air_quality_historical["station_name"].isna().sum())

print("Missing station types:",
      df_air_quality_historical["station_type"].isna().sum())

print("\nDuplicate measurement keys:")
print(
    df_air_quality_historical.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)

Shape: (50297, 8)

Date range:
2021-12-31 23:00:00+00:00
2024-12-30 23:00:00+00:00

Data types:
datetime        datetime64[us, UTC]
station                         str
core                            str
component                       str
period                          str
value                       float64
station_name                    str
station_type                    str
dtype: object

Records per year:
datetime
2021       44
2022    16022
2023    17083
2024    17148
Name: count, dtype: int64

Components:
component
no2_24h     16063
pm2_24h     12780
pm10_24h    12690
o3_24h       8764
Name: count, dtype: int64

Missing station names: 0
Missing station types: 0

Duplicate measurement keys:
0


In [123]:
# Convert UTC timestamps to Berlin local time
historical_berlin_dates = (
    df_air_quality_historical["datetime"]
    .dt.tz_convert("Europe/Berlin")
)

print("First date (Berlin):", historical_berlin_dates.min())
print("Last date (Berlin):", historical_berlin_dates.max())

print("\nRecords per calendar year (Berlin):")
print(
    historical_berlin_dates
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nRecords outside the intended historical period:")
print(
    (
        (historical_berlin_dates < pd.Timestamp("2022-01-01", tz="Europe/Berlin"))
        |
        (historical_berlin_dates >= pd.Timestamp("2025-01-01", tz="Europe/Berlin"))
    ).sum()
)

First date (Berlin): 2022-01-01 00:00:00+01:00
Last date (Berlin): 2024-12-31 00:00:00+01:00

Records per calendar year (Berlin):
datetime
2022    16019
2023    17083
2024    17195
Name: count, dtype: int64

Records outside the intended historical period:
0


In [124]:
# Convert existing timestamps to UTC-aware datetimes for comparison
existing_check = df_air_quality.copy()

existing_check["datetime"] = pd.to_datetime(
    existing_check["datetime"],
    utc=True
)

# Compare core values for each target component
print("CORE VALUES BY COMPONENT")

for component in pollutants.values():

    print(f"\n{component}")

    print("Historical:")
    print(
        df_air_quality_historical.loc[
            df_air_quality_historical["component"] == component,
            "core"
        ].value_counts().to_dict()
    )

    print("Existing:")
    print(
        existing_check.loc[
            existing_check["component"] == component,
            "core"
        ].value_counts().to_dict()
    )

# Check whether historical measurement keys already exist
key_columns = ["datetime", "station", "component", "period"]

historical_keys = df_air_quality_historical[key_columns].drop_duplicates()
existing_keys = existing_check[key_columns].drop_duplicates()

overlap = historical_keys.merge(
    existing_keys,
    on=key_columns,
    how="inner"
)

print("\nHISTORICAL / EXISTING OVERLAP")
print("Historical unique keys:", len(historical_keys))
print("Existing unique keys:", len(existing_keys))
print("Overlapping keys:", len(overlap))

print("\nDate boundary:")
print("Historical last UTC:", historical_keys["datetime"].max())
print("Existing first UTC:", existing_keys["datetime"].min())

CORE VALUES BY COMPONENT

no2_24h
Historical:
{'no2': 16063}
Existing:
{'no2': 9735}

o3_24h
Historical:
{'o3': 8764}
Existing:
{'o3': 5212}

pm10_24h
Historical:
{'pm10': 12690}
Existing:
{'pm10': 7758}

pm2_24h
Historical:
{'pm2': 12780}
Existing:
{'pm2': 7758}

HISTORICAL / EXISTING OVERLAP
Historical unique keys: 50297
Existing unique keys: 54305
Overlapping keys: 0

Date boundary:
Historical last UTC: 2024-12-30 23:00:00+00:00
Existing first UTC: 2024-12-31 23:00:00+00:00


In [125]:
# Create a copy of the existing dataset
df_air_quality_existing = df_air_quality.copy()

# Standardise existing timestamps to UTC-aware datetime
df_air_quality_existing["datetime"] = pd.to_datetime(
    df_air_quality_existing["datetime"],
    utc=True
)

# Append historical and existing records
df_air_quality_combined = pd.concat(
    [
        df_air_quality_historical,
        df_air_quality_existing
    ],
    ignore_index=True
)

# Sort chronologically
df_air_quality_combined = (
    df_air_quality_combined
    .sort_values(["datetime", "station", "component"])
    .reset_index(drop=True)
)

print("Combined dataset shape:", df_air_quality_combined.shape)

print("\nDate range (UTC):")
print(df_air_quality_combined["datetime"].min())
print(df_air_quality_combined["datetime"].max())

print("\nRecords per year (Berlin):")
print(
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nComponents:")
print(df_air_quality_combined["component"].value_counts())

print("\nDuplicate measurement keys:")
print(
    df_air_quality_combined.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)

Combined dataset shape: (104602, 8)

Date range (UTC):
2021-12-31 23:00:00+00:00
2026-10-20 22:00:00+00:00

Records per year (Berlin):
datetime
2022    16019
2023    17083
2024    17195
2025    30420
2026    23885
Name: count, dtype: int64

Components:
component
no2_24h     25798
pm2_24h     20538
pm10_24h    20448
o3_24h      13976
no_24h       9735
nox_24h      9735
ebc_24h      2455
co_24h       1917
Name: count, dtype: int64

Duplicate measurement keys:
0


In [126]:
# Create a local Berlin date and year for coverage analysis
df_air_quality_combined["date_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.date
)

df_air_quality_combined["year_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
)

# Count records and distinct measurement dates per station, component and year
coverage_by_station = (
    df_air_quality_combined
    .groupby(["station", "component", "year_berlin"])
    .agg(
        records=("datetime", "size"),
        unique_dates=("date_berlin", "nunique")
    )
    .reset_index()
)

# Display record counts by station and component, with years as columns
coverage_records = coverage_by_station.pivot_table(
    index=["station", "component"],
    columns="year_berlin",
    values="records",
    fill_value=0
)

print("Records per station, component and year:")
display(coverage_records)

# Display the number of distinct dates covered
coverage_dates = coverage_by_station.pivot_table(
    index=["station", "component"],
    columns="year_berlin",
    values="unique_dates",
    fill_value=0
)

print("\nDistinct measurement dates per station, component and year:")
display(coverage_dates)

Records per station, component and year:


year_berlin         2022   2023   2024   2025   2026
station component                                   
mc010   ebc_24h      0.0    0.0    0.0  305.0  294.0
        no2_24h    365.0  365.0  366.0  365.0  294.0
        no_24h       0.0    0.0    0.0  365.0  294.0
        nox_24h      0.0    0.0    0.0  365.0  294.0
        o3_24h     365.0  365.0  366.0  365.0  294.0
...                  ...    ...    ...    ...    ...
mc221   pm10_24h   365.0  363.0  366.0  335.0  294.0
        pm2_24h    365.0  363.0  366.0  335.0  294.0
mc282   no2_24h    365.0  365.0  366.0  365.0  294.0
        no_24h       0.0    0.0    0.0  365.0  294.0
        nox_24h      0.0    0.0    0.0  365.0  294.0

[84 rows x 5 columns]


Distinct measurement dates per station, component and year:


year_berlin         2022   2023   2024   2025   2026
station component                                   
mc010   ebc_24h      0.0    0.0    0.0  305.0  294.0
        no2_24h    365.0  365.0  366.0  365.0  294.0
        no_24h       0.0    0.0    0.0  365.0  294.0
        nox_24h      0.0    0.0    0.0  365.0  294.0
        o3_24h     365.0  365.0  366.0  365.0  294.0
...                  ...    ...    ...    ...    ...
mc221   pm10_24h   365.0  363.0  366.0  335.0  294.0
        pm2_24h    365.0  363.0  366.0  335.0  294.0
mc282   no2_24h    365.0  365.0  366.0  365.0  294.0
        no_24h       0.0    0.0    0.0  365.0  294.0
        nox_24h      0.0    0.0    0.0  365.0  294.0

[84 rows x 5 columns]

In [127]:
# Keep only the four pollutants used throughout the historical download
target_components = [
    "no2_24h",
    "o3_24h",
    "pm10_24h",
    "pm2_24h"
]

coverage_target = coverage_by_station[
    coverage_by_station["component"].isin(target_components)
].copy()

# Expected calendar days for each year in the dataset
expected_days = {
    2022: 365,
    2023: 365,
    2024: 366,
    2025: 365,
    2026: 294
}

# Calculate missing dates for each station, component and year
coverage_target["expected_days"] = (
    coverage_target["year_berlin"].map(expected_days)
)

coverage_target["missing_days"] = (
    coverage_target["expected_days"]
    - coverage_target["unique_dates"]
)

# Display the results in a compact table
coverage_missing = coverage_target.pivot_table(
    index=["station", "component"],
    columns="year_berlin",
    values="missing_days",
    fill_value=0
).astype(int)

print("Missing measurement dates per station and pollutant:")
display(coverage_missing)

print("\nTotal missing dates by pollutant and year:")
display(
    coverage_target.groupby(["component", "year_berlin"])
    ["missing_days"]
    .sum()
    .unstack(fill_value=0)
)

Missing measurement dates per station and pollutant:


year_berlin        2022  2023  2024  2025  2026
station component                              
mc010   no2_24h       0     0     0     0     0
        o3_24h        0     0     0     0     0
        pm10_24h      0     0     0     0     0
        pm2_24h       0     0     0     0     0
mc027   no2_24h       0     1     0     0     0
        o3_24h        0     1     0     0     0
mc032   no2_24h       0     0     4     0     0
        o3_24h        0     0     0     0     0
        pm10_24h     23    12     0     0     0
        pm2_24h       0     0     0     0     0
mc042   no2_24h       0     0     0     0     0
        o3_24h        0     0     0     0     0
        pm10_24h      0     0     3     0     0
        pm2_24h       0     0     0     0     0
mc077   no2_24h       0     0     0     0    60
        o3_24h        0     0     0     0    60
        pm10_24h      0     0     0     0    60
        pm2_24h       0     0     0     0    60
mc085   no2_24h       3     0     0     0     0
        o3_24h        3     0     0     0     0
        pm10_24h      3    15     0     0     0
        pm2_24h       3     0     0     0     0
mc117   no2_24h       2     0     0     0    60
        pm10_24h      2     0     0     0    60
        pm2_24h       2     0     0     0    60
mc124   no2_24h       0     0     0     0     0
        pm10_24h      0     2     0     0     0
        pm2_24h       0     0     0     0     0
mc144   no2_24h       0     0     0     0     0
        pm10_24h      0    28     0     0     0
        pm2_24h       0     0     0     0     0
mc145   no2_24h       0     0     0     0     0
        o3_24h        0     0     0     0     0
mc171   no2_24h       0     0     0     0     0
        pm10_24h      0     0     0     0     0
        pm2_24h       0     0     0     0     0
mc174   no2_24h       0     0     0     0     0
        o3_24h        0     0     0     0     0
        pm10_24h      0     7     0     0     0
        pm2_24h       0     0     0     0     0
mc190   no2_24h       0     0     0     0     0
        pm10_24h      0     0     0     0     0
        pm2_24h       0     0     0     0     0
mc221   no2_24h       0     2     0    30     0
        pm10_24h      0     2     0    30     0
        pm2_24h       0     2     0    30     0
mc282   no2_24h       0     0     0     0     0


Total missing dates by pollutant and year:


year_berlin,2022,2023,2024,2025,2026
component,,,,,
no2_24h,5,3,4,30,120
o3_24h,3,1,0,0,60
pm10_24h,28,66,3,30,120
pm2_24h,5,2,0,30,120


In [128]:
# Check missing date ranges for 2025 and 2026
for (station, component, year), group in coverage_target[
    coverage_target["missing_days"] > 0
].groupby(["station", "component", "year_berlin"]):

    if year not in [2025, 2026]:
        continue

    # Define the expected date range
    start_date = pd.Timestamp(f"{year}-01-01").date()
    end_date = (
        pd.Timestamp("2026-10-21").date()
        if year == 2026
        else pd.Timestamp("2025-12-31").date()
    )

    expected_dates = set(
        pd.date_range(start_date, end_date, freq="D").date
    )

    # Get dates actually present
    observed_dates = set(
        df_air_quality_combined.loc[
            (df_air_quality_combined["station"] == station)
            & (df_air_quality_combined["component"] == component)
            & (df_air_quality_combined["year_berlin"] == year),
            "date_berlin"
        ]
    )

    missing_dates = sorted(expected_dates - observed_dates)

    # Group consecutive missing dates into ranges
    ranges = []
    if missing_dates:
        range_start = previous_date = missing_dates[0]

        for current_date in missing_dates[1:]:
            if (current_date - previous_date).days > 1:
                ranges.append((range_start, previous_date))
                range_start = current_date

            previous_date = current_date

        ranges.append((range_start, previous_date))

    print(f"\n{station} | {component} | {year}")
    for range_start, range_end in ranges:
        print(f"  {range_start} to {range_end}")


mc077 | no2_24h | 2026
  2026-04-02 to 2026-05-31

mc077 | o3_24h | 2026
  2026-04-02 to 2026-05-31

mc077 | pm10_24h | 2026
  2026-04-02 to 2026-05-31

mc077 | pm2_24h | 2026
  2026-04-02 to 2026-05-31

mc117 | no2_24h | 2026
  2026-05-02 to 2026-05-31
  2026-08-02 to 2026-08-31

mc117 | pm10_24h | 2026
  2026-05-02 to 2026-05-31
  2026-08-02 to 2026-08-31

mc117 | pm2_24h | 2026
  2026-05-02 to 2026-05-31
  2026-08-02 to 2026-08-31

mc221 | no2_24h | 2025
  2025-05-02 to 2025-05-31

mc221 | pm10_24h | 2025
  2025-05-02 to 2025-05-31

mc221 | pm2_24h | 2025
  2025-05-02 to 2025-05-31


In [129]:
# Re-query the API for mc077 during its missing period
test_station = "mc077"
test_start = "01.04.2026"
test_end = "31.05.2026"

for pollutant in ["no2", "o3", "pm10", "pm2.5"]:
    test_df = download_air_quality(
        test_station,
        pollutant,
        test_start,
        test_end
    )

    print(
        f"{pollutant}: {len(test_df)} records",
        f"| components: {test_df['component'].unique().tolist()}"
        if not test_df.empty else ""
    )

no2: 61 records | components: ['no2_24h']
o3: 61 records | components: ['o3_24h']
pm10: 61 records | components: ['pm10_24h']
pm2.5: 61 records | components: ['pm2_24h']


In [130]:
# Check which dates are present in the fresh API response
for pollutant in ["no2", "o3", "pm10", "pm2.5"]:

    component = pollutants[pollutant]

    fresh_df = download_air_quality(
        "mc077",
        pollutant,
        "01.04.2026",
        "31.05.2026"
    )

    fresh_dates = set(
        fresh_df["datetime"]
        .dt.tz_convert("Europe/Berlin")
        .dt.date
    )

    existing_dates = set(
        df_air_quality_combined.loc[
            (df_air_quality_combined["station"] == "mc077")
            & (df_air_quality_combined["component"] == component)
            & (df_air_quality_combined["year_berlin"] == 2026),
            "date_berlin"
        ]
    )

    missing_from_combined = sorted(fresh_dates - existing_dates)

    print(f"\n{component}")
    print("Fresh API records:", len(fresh_df))
    print("Dates missing from combined dataset:", len(missing_from_combined))
    print("First missing date:", missing_from_combined[0] if missing_from_combined else "None")
    print("Last missing date:", missing_from_combined[-1] if missing_from_combined else "None")


no2_24h
Fresh API records: 61
Dates missing from combined dataset: 60
First missing date: 2026-04-02
Last missing date: 2026-05-31

o3_24h
Fresh API records: 61
Dates missing from combined dataset: 60
First missing date: 2026-04-02
Last missing date: 2026-05-31

pm10_24h
Fresh API records: 61
Dates missing from combined dataset: 60
First missing date: 2026-04-02
Last missing date: 2026-05-31

pm2_24h
Fresh API records: 61
Dates missing from combined dataset: 60
First missing date: 2026-04-02
Last missing date: 2026-05-31


In [131]:
# Define the station, pollutant and date combinations to recover
recovery_requests = [
    ("mc077", ["no2", "o3", "pm10", "pm2.5"], "01.04.2026", "31.05.2026"),
    ("mc117", ["no2", "pm10", "pm2.5"], "01.05.2026", "31.05.2026"),
    ("mc117", ["no2", "pm10", "pm2.5"], "01.08.2026", "31.08.2026"),
    ("mc221", ["no2", "pm10", "pm2.5"], "01.05.2025", "31.05.2025")
]

# Download the requested periods
recovered_parts = []

for station, pollutant_list, start, end in recovery_requests:
    for pollutant in pollutant_list:
        recovered = download_air_quality(
            station,
            pollutant,
            start,
            end
        )

        if not recovered.empty:
            recovered_parts.append(recovered)

# Combine all recovered API records
df_air_quality_recovered = pd.concat(
    recovered_parts,
    ignore_index=True
)

# Add station metadata
df_air_quality_recovered = df_air_quality_recovered.merge(
    air_station_mapping,
    on="station",
    how="left",
    validate="many_to_one"
)

# Match the existing dataset's original columns
original_columns = [
    "datetime", "station", "core", "component",
    "period", "value", "station_name", "station_type"
]

df_air_quality_recovered = df_air_quality_recovered[
    original_columns
]

# Identify records that are not already in the combined dataset
measurement_keys = [
    "datetime", "station", "component", "period"
]

existing_keys = df_air_quality_combined[measurement_keys]

recovered_keys = df_air_quality_recovered[measurement_keys]

missing_mask = ~recovered_keys.apply(
    tuple, axis=1
).isin(
    set(existing_keys.apply(tuple, axis=1))
)

df_air_quality_to_append = df_air_quality_recovered[
    missing_mask
].copy()

# Add the missing records
df_air_quality_combined = pd.concat(
    [
        df_air_quality_combined,
        df_air_quality_to_append
    ],
    ignore_index=True
)

# Recalculate helper columns for the newly appended records
df_air_quality_combined["date_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.date
)

df_air_quality_combined["year_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
)

# Sort chronologically
df_air_quality_combined = (
    df_air_quality_combined
    .sort_values(["datetime", "station", "component"])
    .reset_index(drop=True)
)

print("Fresh API records:", len(df_air_quality_recovered))
print("New records appended:", len(df_air_quality_to_append))
print("Combined dataset shape:", df_air_quality_combined.shape)

print("\nNew records by station and component:")
display(
    df_air_quality_to_append
    .groupby(["station", "component"])
    .size()
    .unstack(fill_value=0)
)

print("\nDuplicate measurement keys:")
print(
    df_air_quality_combined.duplicated(
        subset=measurement_keys
    ).sum()
)

Fresh API records: 523
New records appended: 510
Combined dataset shape: (105112, 10)

New records by station and component:


component,no2_24h,o3_24h,pm10_24h,pm2_24h
station,,,,
mc077,60,60,60,60
mc117,60,0,60,60
mc221,30,0,30,30



Duplicate measurement keys:
0


In [132]:
# Recalculate coverage for the four target pollutants
coverage_after_recovery = (
    df_air_quality_combined[
        df_air_quality_combined["component"].isin(target_components)
    ]
    .groupby(["station", "component", "year_berlin"])
    ["date_berlin"]
    .nunique()
    .reset_index(name="unique_dates")
)

# Expected calendar days
coverage_after_recovery["expected_days"] = (
    coverage_after_recovery["year_berlin"].map(expected_days)
)

# Calculate remaining missing dates
coverage_after_recovery["missing_days"] = (
    coverage_after_recovery["expected_days"]
    - coverage_after_recovery["unique_dates"]
)

# Display remaining gaps
remaining_gaps = coverage_after_recovery[
    coverage_after_recovery["missing_days"] > 0
].copy()

print("Remaining incomplete station-pollutant-year combinations:")
display(
    remaining_gaps.sort_values(
        ["year_berlin", "station", "component"]
    )
)

print("\nTotal remaining missing dates by pollutant and year:")
display(
    remaining_gaps
    .groupby(["component", "year_berlin"])["missing_days"]
    .sum()
    .unstack(fill_value=0)
)

Remaining incomplete station-pollutant-year combinations:


,station,component,year_berlin,unique_dates,expected_days,missing_days
40,mc032,pm10_24h,2022,342,365,23
90,mc085,no2_24h,2022,362,365,3
95,mc085,o3_24h,2022,362,365,3
100,mc085,pm10_24h,2022,362,365,3
105,mc085,pm2_24h,2022,362,365,3
110,mc117,no2_24h,2022,363,365,2
115,mc117,pm10_24h,2022,363,365,2
120,mc117,pm2_24h,2022,363,365,2
21,mc027,no2_24h,2023,364,365,1
26,mc027,o3_24h,2023,364,365,1



Total remaining missing dates by pollutant and year:


year_berlin,2022,2023,2024
component,,,
no2_24h,5,3,4
o3_24h,3,1,0
pm10_24h,28,66,3
pm2_24h,5,2,0


In [133]:
# Inspect exact missing dates for all remaining gaps
for _, row in remaining_gaps.iterrows():

    station = row["station"]
    component = row["component"]
    year = row["year_berlin"]

    # Expected calendar dates for the year
    expected_dates = set(
        pd.date_range(
            f"{year}-01-01",
            f"{year}-12-31",
            freq="D"
        ).date
    )

    # Dates available in the combined dataset
    observed_dates = set(
        df_air_quality_combined.loc[
            (df_air_quality_combined["station"] == station)
            & (df_air_quality_combined["component"] == component)
            & (df_air_quality_combined["year_berlin"] == year),
            "date_berlin"
        ]
    )

    missing_dates = sorted(expected_dates - observed_dates)

    # Group consecutive missing dates
    ranges = []

    if missing_dates:
        range_start = previous_date = missing_dates[0]

        for current_date in missing_dates[1:]:
            if (current_date - previous_date).days > 1:
                ranges.append((range_start, previous_date))
                range_start = current_date

            previous_date = current_date

        ranges.append((range_start, previous_date))

    print(f"\n{station} | {component} | {year}")

    for range_start, range_end in ranges:
        print(f"  {range_start} to {range_end}")


mc027 | no2_24h | 2023
  2023-12-01 to 2023-12-01

mc027 | o3_24h | 2023
  2023-12-01 to 2023-12-01

mc032 | no2_24h | 2024
  2024-07-01 to 2024-07-04

mc032 | pm10_24h | 2022
  2022-03-16 to 2022-04-07

mc032 | pm10_24h | 2023
  2023-08-20 to 2023-08-31

mc042 | pm10_24h | 2024
  2024-04-30 to 2024-05-02

mc085 | no2_24h | 2022
  2022-10-01 to 2022-10-03

mc085 | o3_24h | 2022
  2022-10-01 to 2022-10-03

mc085 | pm10_24h | 2022
  2022-10-01 to 2022-10-03

mc085 | pm10_24h | 2023
  2023-07-31 to 2023-08-14

mc085 | pm2_24h | 2022
  2022-10-01 to 2022-10-03

mc117 | no2_24h | 2022
  2022-06-30 to 2022-07-01

mc117 | pm10_24h | 2022
  2022-06-30 to 2022-07-01

mc117 | pm2_24h | 2022
  2022-06-30 to 2022-07-01

mc124 | pm10_24h | 2023
  2023-05-01 to 2023-05-02

mc144 | pm10_24h | 2023
  2023-07-30 to 2023-08-22
  2023-09-01 to 2023-09-04

mc174 | pm10_24h | 2023
  2023-06-28 to 2023-07-04

mc221 | no2_24h | 2023
  2023-10-01 to 2023-10-02

mc221 | pm10_24h | 2023
  2023-10-01 to 2023-10

In [134]:
# Define all remaining missing periods to recheck
recovery_requests = [
    ("mc027", ["no2", "o3"], "01.12.2023", "01.12.2023"),

    ("mc032", ["no2"], "01.07.2024", "04.07.2024"),
    ("mc032", ["pm10"], "16.03.2022", "07.04.2022"),
    ("mc032", ["pm10"], "20.08.2023", "31.08.2023"),

    ("mc042", ["pm10"], "30.04.2024", "02.05.2024"),

    ("mc085", ["no2", "o3", "pm10", "pm2.5"],
     "01.10.2022", "03.10.2022"),
    ("mc085", ["pm10"], "31.07.2023", "14.08.2023"),

    ("mc117", ["no2", "pm10", "pm2.5"],
     "30.06.2022", "01.07.2022"),

    ("mc124", ["pm10"], "01.05.2023", "02.05.2023"),

    ("mc144", ["pm10"], "30.07.2023", "22.08.2023"),
    ("mc144", ["pm10"], "01.09.2023", "04.09.2023"),

    ("mc174", ["pm10"], "28.06.2023", "04.07.2023"),

    ("mc221", ["no2", "pm10", "pm2.5"],
     "01.10.2023", "02.10.2023")
]

# Download all requested periods
recovered_parts = []

for station, pollutant_list, start, end in recovery_requests:
    for pollutant in pollutant_list:
        recovered = download_air_quality(
            station,
            pollutant,
            start,
            end
        )

        if not recovered.empty:
            recovered_parts.append(recovered)

# Combine downloaded records
df_air_quality_recovered = pd.concat(
    recovered_parts,
    ignore_index=True
)

# Add station metadata
df_air_quality_recovered = df_air_quality_recovered.merge(
    air_station_mapping,
    on="station",
    how="left",
    validate="many_to_one"
)

# Match the original dataset schema
original_columns = [
    "datetime", "station", "core", "component",
    "period", "value", "station_name", "station_type"
]

df_air_quality_recovered = df_air_quality_recovered[
    original_columns
]

# Identify records not already in the combined dataset
measurement_keys = [
    "datetime", "station", "component", "period"
]

existing_keys = set(
    df_air_quality_combined[measurement_keys].apply(tuple, axis=1)
)

recovered_keys = df_air_quality_recovered[
    measurement_keys
].apply(tuple, axis=1)

missing_mask = ~recovered_keys.isin(existing_keys)

df_air_quality_to_append = df_air_quality_recovered[
    missing_mask
].copy()

# Append only genuinely missing records
df_air_quality_combined = pd.concat(
    [
        df_air_quality_combined,
        df_air_quality_to_append
    ],
    ignore_index=True
)

# Refresh Berlin date and year helper columns
df_air_quality_combined["date_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.date
)

df_air_quality_combined["year_berlin"] = (
    df_air_quality_combined["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
)

# Sort chronologically
df_air_quality_combined = (
    df_air_quality_combined
    .sort_values(["datetime", "station", "component"])
    .reset_index(drop=True)
)

print("Fresh API records:", len(df_air_quality_recovered))
print("New records appended:", len(df_air_quality_to_append))
print("Combined dataset shape:", df_air_quality_combined.shape)

print("\nNew records by station and component:")
display(
    df_air_quality_to_append
    .groupby(["station", "component"])
    .size()
    .unstack(fill_value=0)
)

print("\nDuplicate measurement keys:")
print(
    df_air_quality_combined.duplicated(
        subset=measurement_keys
    ).sum()
)

ValueError: No objects to concatenate

In [135]:
# Test selected missing periods individually
diagnostic_requests = [
    ("mc032", "pm10", "16.03.2022", "07.04.2022"),
    ("mc085", "no2", "01.10.2022", "03.10.2022"),
    ("mc144", "pm10", "30.07.2023", "22.08.2023"),
    ("mc221", "no2", "01.10.2023", "02.10.2023")
]

for station, pollutant, start, end in diagnostic_requests:

    try:
        test_df = download_air_quality(
            station,
            pollutant,
            start,
            end
        )

        print(
            f"{station} | {pollutant} | {start}–{end}: "
            f"{len(test_df)} records"
        )

    except Exception as e:
        print(
            f"{station} | {pollutant} | {start}–{end}: "
            f"{type(e).__name__}: {e}"
        )

mc032 | pm10 | 16.03.2022–07.04.2022: 0 records
mc085 | no2 | 01.10.2022–03.10.2022: 0 records
mc144 | pm10 | 30.07.2023–22.08.2023: 0 records
mc221 | no2 | 01.10.2023–02.10.2023: 0 records


In [136]:
# Recalculate coverage for the four target pollutants
final_coverage = (
    df_air_quality_combined[
        df_air_quality_combined["component"].isin(target_components)
    ]
    .groupby(["station", "component", "year_berlin"])
    ["date_berlin"]
    .nunique()
    .reset_index(name="observed_days")
)

# Add expected days for each year
final_coverage["expected_days"] = (
    final_coverage["year_berlin"].map(expected_days)
)

# Calculate missing days and coverage percentage
final_coverage["missing_days"] = (
    final_coverage["expected_days"]
    - final_coverage["observed_days"]
)

final_coverage["coverage_pct"] = (
    final_coverage["observed_days"]
    / final_coverage["expected_days"]
    * 100
).round(2)

# Summarise by pollutant and year
coverage_summary = (
    final_coverage
    .groupby(["component", "year_berlin"])
    .agg(
        observed_days=("observed_days", "sum"),
        expected_days=("expected_days", "sum"),
        missing_days=("missing_days", "sum")
    )
    .reset_index()
)

coverage_summary["coverage_pct"] = (
    coverage_summary["observed_days"]
    / coverage_summary["expected_days"]
    * 100
).round(2)

print("Coverage summary by pollutant and year:")
display(coverage_summary)

print("\nRemaining incomplete combinations:")
display(
    final_coverage[
        final_coverage["missing_days"] > 0
    ].sort_values(
        ["year_berlin", "station", "component"]
    )
)

Coverage summary by pollutant and year:


,component,year_berlin,observed_days,expected_days,missing_days,coverage_pct
0,no2_24h,2022,5105,5110,5,99.90
1,no2_24h,2023,5472,5475,3,99.95
2,no2_24h,2024,5486,5490,4,99.93
3,no2_24h,2025,5475,5475,0,100.00
4,no2_24h,2026,4410,4410,0,100.00
5,o3_24h,2022,2917,2920,3,99.90
6,o3_24h,2023,2919,2920,1,99.97
7,o3_24h,2024,2928,2928,0,100.00
8,o3_24h,2025,2920,2920,0,100.00
9,o3_24h,2026,2352,2352,0,100.00



Remaining incomplete combinations:


,station,component,year_berlin,observed_days,expected_days,missing_days,coverage_pct
40,mc032,pm10_24h,2022,342,365,23,93.70
90,mc085,no2_24h,2022,362,365,3,99.18
95,mc085,o3_24h,2022,362,365,3,99.18
100,mc085,pm10_24h,2022,362,365,3,99.18
105,mc085,pm2_24h,2022,362,365,3,99.18
110,mc117,no2_24h,2022,363,365,2,99.45
115,mc117,pm10_24h,2022,363,365,2,99.45
120,mc117,pm2_24h,2022,363,365,2,99.45
21,mc027,no2_24h,2023,364,365,1,99.73
26,mc027,o3_24h,2023,364,365,1,99.73


In [137]:
# Define the original dataset columns
original_columns = [
    "datetime", "station", "core", "component",
    "period", "value", "station_name", "station_type"
]

# Create a clean export-ready copy without helper columns
df_air_quality_final = df_air_quality_combined[
    original_columns
].copy()

# Expected mapping between API core and measurement component
expected_mapping = {
    "no2": "no2_24h",
    "o3": "o3_24h",
    "pm10": "pm10_24h",
    "pm2.5": "pm2_24h"
}

# Check data integrity
print("Final shape:", df_air_quality_final.shape)
print("Columns:", df_air_quality_final.columns.tolist())

print("\nDate range in Berlin:")
print(
    df_air_quality_final["datetime"]
    .min()
    .tz_convert("Europe/Berlin")
)
print(
    df_air_quality_final["datetime"]
    .max()
    .tz_convert("Europe/Berlin")
)

print("\nMissing values:")
display(df_air_quality_final.isna().sum())

print("\nDuplicate measurement keys:")
print(
    df_air_quality_final.duplicated(
        subset=["datetime", "station", "component", "period"]
    ).sum()
)

print("\nUnique stations:", df_air_quality_final["station"].nunique())
print("Unique components:", sorted(df_air_quality_final["component"].unique()))

print("\nUnexpected core/component combinations:")
print(
    df_air_quality_final[
        df_air_quality_final.apply(
            lambda row: expected_mapping.get(row["core"]) != row["component"],
            axis=1
        )
    ][["core", "component"]]
    .drop_duplicates()
)

Final shape: (105112, 8)
Columns: ['datetime', 'station', 'core', 'component', 'period', 'value', 'station_name', 'station_type']

Date range in Berlin:
2022-01-01 00:00:00+01:00
2026-10-21 00:00:00+02:00

Missing values:


datetime          0
station           0
core              0
component         0
period            0
value           731
station_name      0
station_type      0
dtype: int64


Duplicate measurement keys:
0

Unique stations: 15
Unique components: ['co_24h', 'ebc_24h', 'no2_24h', 'no_24h', 'nox_24h', 'o3_24h', 'pm10_24h', 'pm2_24h']

Unexpected core/component combinations:
      core component
3      pm2   pm2_24h
50297  ebc   ebc_24h
50299   no    no_24h
50300  nox   nox_24h
50314   co    co_24h


In [138]:
# Inspect missing measurement values by component and year
df_air_quality_final["year_berlin"] = (
    df_air_quality_final["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
)

missing_values_summary = (
    df_air_quality_final
    .groupby(["component", "year_berlin"])
    .agg(
        total_records=("value", "size"),
        missing_values=("value", "isna")
    )
    .reset_index()
)

# Calculate missing-value percentages
missing_values_summary["missing_pct"] = (
    missing_values_summary["missing_values"]
    / missing_values_summary["total_records"]
    * 100
).round(2)

print("Missing measurement values by component and year:")
display(
    missing_values_summary[
        missing_values_summary["missing_values"] > 0
    ]
)

print("\nTotal missing measurement values:")
print(df_air_quality_final["value"].isna().sum())

AttributeError: 'SeriesGroupBy' object has no attribute 'isna'

In [139]:
# Create a helper column identifying missing measurements
df_air_quality_final["year_berlin"] = (
    df_air_quality_final["datetime"]
    .dt.tz_convert("Europe/Berlin")
    .dt.year
)

df_air_quality_final["value_missing"] = (
    df_air_quality_final["value"].isna()
)

# Summarise missing values by component and year
missing_values_summary = (
    df_air_quality_final
    .groupby(["component", "year_berlin"])
    .agg(
        total_records=("value", "size"),
        missing_values=("value_missing", "sum")
    )
    .reset_index()
)

# Calculate missing-value percentages
missing_values_summary["missing_pct"] = (
    missing_values_summary["missing_values"]
    / missing_values_summary["total_records"]
    * 100
).round(2)

print("Missing measurement values by component and year:")
display(
    missing_values_summary[
        missing_values_summary["missing_values"] > 0
    ]
)

print("\nTotal missing measurement values:")
print(df_air_quality_final["value"].isna().sum())

Missing measurement values by component and year:


,component,year_berlin,total_records,missing_values,missing_pct
1,co_24h,2026,822,18,2.19
2,ebc_24h,2025,1370,87,6.35
3,ebc_24h,2026,1085,8,0.74
4,no2_24h,2022,5105,32,0.63
5,no2_24h,2023,5472,17,0.31
6,no2_24h,2024,5486,12,0.22
7,no2_24h,2025,5475,4,0.07
8,no2_24h,2026,4410,49,1.11
9,no_24h,2025,5445,4,0.07
10,no_24h,2026,4290,47,1.10



Total missing measurement values:
731


In [140]:
# Complete mapping between API core and measurement component
expected_mapping = {
    "no2": "no2_24h",
    "o3": "o3_24h",
    "pm10": "pm10_24h",
    "pm2": "pm2_24h",
    "ebc": "ebc_24h",
    "no": "no_24h",
    "nox": "nox_24h",
    "co": "co_24h"
}

# Check whether every core matches its expected component
unexpected_combinations = (
    df_air_quality_final[
        df_air_quality_final.apply(
            lambda row: expected_mapping.get(row["core"]) != row["component"],
            axis=1
        )
    ][["core", "component"]]
    .drop_duplicates()
)

print("Unexpected core/component combinations:")
display(unexpected_combinations)

print("\nNumber of unexpected records:")
print(
    df_air_quality_final.apply(
        lambda row: expected_mapping.get(row["core"]) != row["component"],
        axis=1
    ).sum()
)

Unexpected core/component combinations:


,core,component



Number of unexpected records:
0


In [141]:
# Keep only the original eight dataset columns
original_columns = [
    "datetime", "station", "core", "component",
    "period", "value", "station_name", "station_type"
]

df_air_quality_export = df_air_quality_final[
    original_columns
].copy()

# Export the final dataset
filename = "berlin_air_quality_2022_2026.csv"

df_air_quality_export.to_csv(filename, index=False)

print("Export completed:", filename)
print("Shape:", df_air_quality_export.shape)
print("File size (MB):", round(
    __import__("os").path.getsize(filename) / (1024 ** 2), 2
))

Export completed: berlin_air_quality_2022_2026.csv
Shape: (105112, 8)
File size (MB): 7.77


Downloading historical data for water quality

In [143]:
water_combined = pd.read_csv("merged_complete_datasets/berlin_water_quality_cleaned.csv")

# Inspect the existing water-quality dataset
print("Shape:", water_combined.shape)

print("\nColumns:")
print(water_combined.columns.tolist())

print("\nData types:")
print(water_combined.dtypes)

print("\nFirst five records:")
display(water_combined.head())

print("\nDate range:")
print(water_combined["date"].min())
print(water_combined["date"].max())

Shape: (118, 28)

Columns:
['date', 'bathing_water_quality', 'coliform_bacteria', 'cyanobacteria_chl_a', 'e_coli', 'intestinal_enterococci', 'water_visibility_depth', 'water_temperature', 'lake', 'water_temperature_raw', 'water_temperature_qualifier', 'water_temperature_numeric', 'temperature_zero_flag', 'coliform_bacteria_raw', 'coliform_bacteria_qualifier', 'coliform_bacteria_numeric', 'e_coli_raw', 'e_coli_qualifier', 'e_coli_numeric', 'intestinal_enterococci_raw', 'intestinal_enterococci_qualifier', 'intestinal_enterococci_numeric', 'cyanobacteria_chl_a_raw', 'cyanobacteria_chl_a_qualifier', 'cyanobacteria_chl_a_numeric', 'water_visibility_depth_raw', 'water_visibility_depth_qualifier', 'water_visibility_depth_numeric']

Data types:
date                                    str
bathing_water_quality                   str
coliform_bacteria                       str
cyanobacteria_chl_a                     str
e_coli                                  str
intestinal_enterococci           

,date,bathing_water_quality,coliform_bacteria,cyanobacteria_chl_a,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature,lake,water_temperature_raw,...,e_coli_numeric,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric,cyanobacteria_chl_a_raw,cyanobacteria_chl_a_qualifier,cyanobacteria_chl_a_numeric,water_visibility_depth_raw,water_visibility_depth_qualifier,water_visibility_depth_numeric
0,2025-04-29,Suitable for bathing,< 300,"0,0000","< 15,00","< 15,00","> 1,00","16,00",Flughafensee,"16,00",...,15.0,"< 15,00",<,15.0,"0,0000",NaN,0.0,"> 1,00",>,1.0
1,2025-05-15,Suitable for bathing,< 300,"0,20","< 15,00","< 15,00","> 1,00","17,50",Flughafensee,"17,50",...,15.0,"< 15,00",<,15.0,"0,20",NaN,0.2,"> 1,00",>,1.0
2,2025-05-26,Suitable for bathing,360,"0,10","30,00","15,00",NaN,"17,70",Flughafensee,"17,70",...,30.0,"15,00",NaN,15.0,"0,10",NaN,0.1,NaN,NaN,NaN
3,2025-06-11,NaN,null 910,"null 0,30","null 30,00","null 15,00","> 1,00","null 17,80",Flughafensee,"null 17,80",...,30.0,"null 15,00",NaN,15.0,"null 0,30",NaN,0.3,"> 1,00",>,1.0
4,2025-07-10,NaN,< 300,"null 0,0000","null 15,00","null 15,00","> 1,00","null 21,50",Flughafensee,"null 21,50",...,15.0,"null 15,00",NaN,15.0,"null 0,0000",NaN,0.0,"> 1,00",>,1.0



Date range:
2025-04-28
2026-09-15


In [144]:
# Summarise existing samples by lake
lake_summary = (
    water_combined
    .groupby("lake")
    .agg(
        sample_count=("date", "size"),
        first_sample=("date", "min"),
        last_sample=("date", "max"),
        unique_dates=("date", "nunique")
    )
    .reset_index()
    .sort_values("lake")
)

display(lake_summary)

print("\nTotal lakes:", water_combined["lake"].nunique())
print("Total samples:", len(water_combined))

,lake,sample_count,first_sample,last_sample,unique_dates
0,Flughafensee,19,2025-04-29,2026-09-09,19
1,Müggelsee,20,2025-05-06,2026-09-08,20
2,Orankesee,20,2025-04-29,2026-09-09,20
3,Tegeler See,20,2025-04-28,2026-09-14,20
4,Wannsee,19,2025-04-29,2026-09-15,19
5,Weißensee,20,2025-04-29,2026-09-09,20



Total lakes: 6
Total samples: 118


In [145]:
import requests

# Official Berlin bathing-water data endpoint
url = "https://wasserportal.berlin.de/station.php"

params = {
    "anzeige": "d",
    "station": "B920",
    "thema": "bgq",
    "sreihe": "ew",
    "smode": "c",
    "sdatum": "01.01.2022",
    "senddatum": "31.12.2024"
}

# Request historical bathing-water measurements
response = requests.get(
    url,
    params=params,
    timeout=60
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

print("\nFirst 15 lines of response:")
print("\n".join(response.text.splitlines()[:15]))

Status code: 200
Content type: text/csv; charset=utf-8
Response size: 391 bytes

First 15 lines of response:
<br />
<b>Warning</b>:  Undefined variable $strMessZeitgrenze in <b>C:\web\bwpweb\inc\bwp_util.inc.php</b> on line <b>1624</b><br />
<br />
<b>Warning</b>:  Undefined variable $strMessZeitende in <b>C:\web\bwpweb\inc\bwp_util.inc.php</b> on line <b>1624</b><br />
<br />
<b>Warning</b>:  Undefined variable $strTabStamm in <b>C:\web\bwpweb\inc\bwp_util.inc.php</b> on line <b>1624</b><br />


In [146]:
import requests
import re

# Retrieve the Flughafensee bathing-water station page
url = "https://wasserportal.berlin.de/station.php"

params = {
    "anzeige": "d",
    "station": "B920",
    "thema": "bgq"
}

response = requests.get(url, params=params, timeout=60)

print("Status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

# Inspect HTML lines related to the download form
html_lines = response.text.splitlines()

keywords = [
    "form", "input", "sdatum", "senddatum",
    "download", "csv", "nstoffid", "sreihe",
    "exportthema"
]

for i, line in enumerate(html_lines):
    if any(keyword in line.lower() for keyword in keywords):
        print(f"Line {i}: {line.strip()[:500]}")

Status: 200
Content type: text/html; charset=ISO-8859-15
Line 80: const initDatePicker = function(inputId) {
Line 81: $("#" + inputId).flatpickr({
Line 82: dateFormat: "d.m.Y",
Line 85: allowInput: true,
Line 88: $("#" + inputId).blur();
Line 325: nstoffid: stoffid,
Line 639: document.getElementById("form_recherche").submit();
Line 785: <h3>Wo gibt es zusätzliche Informationen zur Barrierefreiheit im Land Berlin?</h3>
Line 786: <a href="https://www.berlin.de/moderne-verwaltung/barrierefreie-it/" class="extern" target="_blank" title=" (Öffnet in neuem Fenster)" rel="noreferrer">Barrierefreie Informations- und Kommunikationstechnik (IKT)</a>
Line 988: <li class='even pos1'><a href='stationen_start.php'>Messstelle</a></li><li class='odd pos2'><a href='station.php?anzeige=d&station=B920&thema=bgq'>Flughafensee: Oberflächengewässer - Wasserqualität (Badegewässer) (Download )</a></li>
Line 1014: <li style='margin-left:15px;font-size:95%' class=''> <a title='wechselt zur Seite' href='station.

In [147]:
import requests

# Create a session
session = requests.Session()

# Open the station's download page
url = "https://wasserportal.berlin.de/station.php"

params = {
    "anzeige": "d",
    "station": "B920",
    "thema": "bgq",
    "nstoffid": "9",
    "nstoffid2": "0"
}

page = session.get(url, params=params, timeout=60)
page.raise_for_status()

# Submit the download form using POST
form_data = {
    "sreihe": "ew",
    "smode": "c",
    "sdatum": "01.01.2022",
    "senddatum": "31.12.2024",
    "exportthema": "pq"
}

response = session.post(
    page.url,
    data=form_data,
    headers={"Referer": page.url},
    timeout=60
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

print("\nFirst 15 lines:")
print("\n".join(response.text.splitlines()[:15]))

Status code: 200
Content type: text/csv; charset=utf-8
Response size: 161 bytes

First 15 lines:
Messstellennummer;B920
L�ngengrad;13,28488900
Breitengrad;52,56978600
Projektion;"WGS 84"
;
Messstelle;Messstellennummer;Datum;Parameter;Vorzeichen;Wert;Einheit


In [148]:
# Print the complete CSV response
print(response.content.decode("latin-1"))

Messstellennummer;B920
Längengrad;13,28488900
Breitengrad;52,56978600
Projektion;"WGS 84"
;
Messstelle;Messstellennummer;Datum;Parameter;Vorzeichen;Wert;Einheit



In [151]:
# Find all form definitions and their hidden fields
from bs4 import BeautifulSoup

soup = BeautifulSoup(page.text, "html.parser")

for form in soup.find_all("form"):
    print("\nFORM:", form.get("id"))
    print("Action:", form.get("action"))
    print("Method:", form.get("method"))

    for field in form.find_all(["input", "select"]):
        print(
            field.name,
            "name:", field.get("name"),
            "value:", field.get("value"),
            "id:", field.get("id")
        )


FORM: None
Action: None
Method: None
input name: input_1_11b value: 29.04.2025 id: input_1_11b
input name: input_1_11c value: 09.09.2026 id: input_1_11c

FORM: form_gw_c
Action: 
Method: post
input name: sreihe value: ew id: sreihe
input name: smode value: c id: smode
input name: sdatum value:  id: sdatum7
input name: senddatum value:  id: sdatum8
input name: exportthema value: pq id: exportthema

FORM: None
Action: None
Method: None
input name: input_1_11d value: 06.09.2011 id: input_1_11d
input name: input_1_11e value: 09.09.2026 id: input_1_11e

FORM: form_gw_d
Action: 
Method: post
input name: sreihe value: wa id: sreihe
input name: smode value: c id: smode
input name: sdatum value:  id: sdatum9
input name: senddatum value:  id: sdatum10
input name: exportthema value: gw id: exportthema
input name: exportthema value: pq id: exportthema


In [152]:
# Inspect download buttons and related JavaScript
for form_id in ["form_gw_c", "form_gw_d"]:
    form = soup.find("form", id=form_id)

    print(f"\n{'=' * 50}")
    print("FORM:", form_id)
    print("FORM CONTENT:")
    print(form.parent.get_text(" ", strip=True)[:2000])

    print("\nBUTTONS:")
    for button in form.parent.find_all(["button", "input", "a"]):
        print(button)

print("\nRELATED JAVASCRIPT:")
for script in soup.find_all("script"):
    if script.string and any(term in script.string for term in ["form_gw_c", "form_gw_d", "formsubmit"]):
        print(script.string[:5000])


FORM: form_gw_c
FORM CONTENT:
Download

BUTTONS:
<input id="sreihe" name="sreihe" type="hidden" value="ew"/>
<input id="smode" name="smode" type="hidden" value="c"/>
<input id="sdatum7" name="sdatum" type="hidden" value=""/>
<input id="sdatum8" name="senddatum" type="hidden" value=""/>
<input id="exportthema" name="exportthema" type="hidden" value="pq"/>
<button aria-label="Datei herunterladen" class="btn button" onclick="javascript:formsubmit(7);">Download</button>

FORM: form_gw_d
FORM CONTENT:
csv-Download

BUTTONS:
<input id="sreihe" name="sreihe" type="hidden" value="wa"/>
<input id="smode" name="smode" type="hidden" value="c"/>
<input id="sdatum9" name="sdatum" type="hidden" value=""/>
<input id="sdatum10" name="senddatum" type="hidden" value=""/>
<input id="exportthema" name="exportthema" type="hidden" value="gw"/>
<input id="exportthema" name="exportthema" type="hidden" value="pq"/>
<button aria-label="Datei herunterladen" class="btn button" onclick="javascript:formsubmit(8);"

In [153]:
# Print the headings and labels surrounding each download form
for form_id in ["form_gw_c", "form_gw_d"]:
    form = soup.find("form", id=form_id)

    print(f"\n{'=' * 60}")
    print("FORM:", form_id)

    # Walk backwards to find the section heading and nearby labels
    for element in form.find_all_previous(["h1", "h2", "h3", "h4", "p", "label"], limit=12):
        print(element.get_text(" ", strip=True))


FORM: form_gw_c
Zeitreihe soll beginnen ab: (TT.MM.JJJJ) Zeitreihe soll enden am: (TT.MM.JJJJ) ? Wählen Sie bitte hier den Beginn des Zeitraums aus, für den die Messdaten bereitgestellt werden sollen.
            Machen Sie hier keine Angaben, wird die aktuelle Zeitreihe aufbereitet.
Wassertemperatur
Flughafensee - Wasserqualität (Badegewässer)

Weiteres
Hauptnavigation
Wo gibt es zusätzliche Informationen zur Barrierefreiheit im Land Berlin?
Kontakt zur Ansprechperson
Haben Sie Anmerkungen oder Fragen zur Barrierefreiheit dieser Webseite?
Erklärung zur Barrierefreiheit
Wie barrierefrei ist diese Webseite?
Einstellungen

FORM: form_gw_d
Zeitreihe soll beginnen ab: (TT.MM.JJJJ) Zeitreihe soll enden am: (TT.MM.JJJJ) ? Wählen Sie bitte hier den Beginn des Zeitraums aus, für den die Messdaten bereitgestellt werden sollen.
                 Machen Sie hier keine Angaben, wird die aktuelle Zeitreihe aufbereitet.
Alle Parameter der Messstelle
Zeitreihe soll beginnen ab: (TT.MM.JJJJ) Zeitreihe

In [154]:
# Prepare the date range for the "Alle Parameter der Messstelle" export
form_data_all = [
    ("sreihe", "wa"),
    ("smode", "c"),
    ("sdatum", "01.01.2022"),
    ("senddatum", "31.12.2024"),
    ("exportthema", "gw"),
    ("exportthema", "pq")
]

# Submit the all-parameters export
response_all = session.post(
    page.url,
    data=form_data_all,
    headers={"Referer": page.url},
    timeout=60
)

print("Status code:", response_all.status_code)
print("Content type:", response_all.headers.get("Content-Type"))
print("Response size:", len(response_all.content), "bytes")

print("\nComplete response:")
print(response_all.content.decode("latin-1"))

Status code: 200
Content type: text/csv; charset=utf-8
Response size: 6167 bytes

Complete response:
Messstellennummer;B920
Längengrad;13,28488900
Breitengrad;52,56978600
Projektion;"WGS 84"
;
Messstelle;Messstellennummer;Datum;Parameter;Vorzeichen;Wert;Einheit
Flughafensee;B920;20.04.2022;"Escherichia coli";<;15,0000;"KBE/100 ml"
Flughafensee;B920;20.04.2022;"Intestinale Enterokokken";<;15,0000;"KBE/100 ml"
Flughafensee;B920;20.04.2022;Sichttiefe;;0,0000;m
Flughafensee;B920;17.05.2022;"Escherichia coli";<;15,0000;"KBE/100 ml"
Flughafensee;B920;17.05.2022;"Intestinale Enterokokken";<;15,0000;"KBE/100 ml"
Flughafensee;B920;17.05.2022;Sichttiefe;;0,0000;m
Flughafensee;B920;31.05.2022;"Escherichia coli";<;15,0000;"KBE/100 ml"
Flughafensee;B920;31.05.2022;"Intestinale Enterokokken";;15,0000;"KBE/100 ml"
Flughafensee;B920;31.05.2022;Sichttiefe;>;1,0000;m
Flughafensee;B920;13.06.2022;"Escherichia coli";<;15,0000;"KBE/100 ml"
Flughafensee;B920;13.06.2022;"Intestinale Enterokokken";<;15,0000;"

In [155]:
# Read the historical export as a semicolon-separated CSV
from io import StringIO
import pandas as pd

df_test_water = pd.read_csv(
    StringIO(response_all.content.decode("latin-1")),
    sep=";",
    skiprows=5,
    dtype=str
)

# Inspect the returned parameters
print("Shape:", df_test_water.shape)

print("\nParameters:")
print(df_test_water["Parameter"].value_counts(dropna=False))

print("\nUnits:")
print(df_test_water.groupby("Parameter")["Einheit"].unique())

print("\nDate range:")
print(df_test_water["Datum"].min(), "to", df_test_water["Datum"].max())

Shape: (90, 7)

Parameters:
Parameter
Escherichia coli            30
Intestinale Enterokokken    30
Sichttiefe                  30
Name: count, dtype: int64

Units:
Parameter
Escherichia coli            [KBE/100 ml]
Intestinale Enterokokken    [KBE/100 ml]
Sichttiefe                           [m]
Name: Einheit, dtype: object

Date range:
01.06.2023 to 31.07.2024


In [156]:
# Convert German date strings into datetime
df_test_water["Datum"] = pd.to_datetime(
    df_test_water["Datum"],
    format="%d.%m.%Y"
)

# Check chronological coverage
print("Date range:")
print(df_test_water["Datum"].min(), "to", df_test_water["Datum"].max())

# Count unique sampling dates per year
print("\nSampling dates per year:")
print(
    df_test_water.groupby(df_test_water["Datum"].dt.year)["Datum"]
    .nunique()
)

# Check whether all three parameters are available on each sampling date
print("\nParameters per sampling date:")
print(
    df_test_water.groupby("Datum")["Parameter"]
    .nunique()
    .value_counts()
)

Date range:
2022-04-20 00:00:00 to 2024-09-09 00:00:00

Sampling dates per year:
Datum
2022     9
2023    11
2024    10
Name: Datum, dtype: int64

Parameters per sampling date:
Parameter
3    30
Name: count, dtype: int64


In [157]:
import pandas as pd
import requests
from io import StringIO

# Official station IDs
water_stations = {
    "Flughafensee": "B920",
    "Müggelsee": "B116",
    "Orankesee": "B911",
    "Tegeler See": "B309",
    "Wannsee": "B350",
    "Weißensee": "B910"
}

# Store individual exports
historical_water_data = {}

session = requests.Session()

for lake, station_id in water_stations.items():

    # Open the station's bathing-water download page
    url = "https://wasserportal.berlin.de/station.php"

    params = {
        "anzeige": "d",
        "station": station_id,
        "thema": "bgq"
    }

    page = session.get(url, params=params, timeout=60)
    page.raise_for_status()

    # Submit the verified all-parameters export
    form_data = [
        ("sreihe", "wa"),
        ("smode", "c"),
        ("sdatum", "01.01.2022"),
        ("senddatum", "31.12.2024"),
        ("exportthema", "gw"),
        ("exportthema", "pq")
    ]

    response = session.post(
        page.url,
        data=form_data,
        headers={"Referer": page.url},
        timeout=60
    )

    response.raise_for_status()

    # Parse the CSV response
    df = pd.read_csv(
        StringIO(response.content.decode("latin-1")),
        sep=";",
        skiprows=5,
        dtype=str
    )

    # Store the export
    historical_water_data[lake] = df

    print(
        f"{lake}: {len(df)} rows | "
        f"{df['Parameter'].nunique()} parameters | "
        f"{df['Datum'].nunique()} sampling dates"
    )

Flughafensee: 90 rows | 3 parameters | 30 sampling dates
Müggelsee: 93 rows | 3 parameters | 31 sampling dates
Orankesee: 80 rows | 3 parameters | 29 sampling dates
Tegeler See: 89 rows | 3 parameters | 30 sampling dates
Wannsee: 90 rows | 3 parameters | 30 sampling dates
Weißensee: 88 rows | 3 parameters | 30 sampling dates


In [158]:
# Combine the individual historical exports
df_water_historical_raw = pd.concat(
    historical_water_data.values(),
    ignore_index=True
)

# Parse sampling dates
df_water_historical_raw["Datum"] = pd.to_datetime(
    df_water_historical_raw["Datum"],
    format="%d.%m.%Y"
)

# Check parameter counts per lake
print("PARAMETER COUNTS PER LAKE")
print(
    df_water_historical_raw
    .groupby(["Messstelle", "Parameter"])
    .size()
    .unstack(fill_value=0)
)

# Check date coverage per lake
print("\nDATE COVERAGE PER LAKE")
print(
    df_water_historical_raw
    .groupby("Messstelle")
    .agg(
        first_date=("Datum", "min"),
        last_date=("Datum", "max"),
        sampling_dates=("Datum", "nunique"),
        total_rows=("Datum", "size")
    )
)

# Check duplicate measurements
print("\nDUPLICATE MEASUREMENT ROWS:")
print(
    df_water_historical_raw.duplicated(
        subset=["Messstellennummer", "Datum", "Parameter"]
    ).sum()
)

PARAMETER COUNTS PER LAKE
Parameter                  Escherichia coli  Intestinale Enterokokken  \
Messstelle                                                              
Flughafensee                             30                        30   
Müggelsee, Strandbad                     31                        31   
Orankesee, Strandbad                     29                        29   
Tegeler See, Freibad                     30                        30   
Wannsee, Strandbad                       30                        30   
Weißensee, Seebadeanstalt                30                        30   

Parameter                  Sichttiefe  
Messstelle                             
Flughafensee                       30  
Müggelsee, Strandbad               31  
Orankesee, Strandbad               22  
Tegeler See, Freibad               29  
Wannsee, Strandbad                 30  
Weißensee, Seebadeanstalt          28  

DATE COVERAGE PER LAKE
                          first_date  last_da

In [159]:
# Inspect existing lake names
print("EXISTING LAKE NAMES:")
print(sorted(water_combined["lake"].unique()))

# Inspect historical station names
print("\nHISTORICAL STATION NAMES:")
print(sorted(df_water_historical_raw["Messstelle"].unique()))

# Inspect existing column names and data types
print("\nEXISTING DATASET STRUCTURE:")
print(water_combined.dtypes.to_string())

# Inspect a few existing records
print("\nEXISTING DATA EXAMPLES:")
display(
    water_combined[
        ["date", "lake", "e_coli", "intestinal_enterococci",
         "water_visibility_depth", "water_temperature"]
    ].head(6)
)

# Inspect historical raw values and qualifiers
print("\nHISTORICAL DATA EXAMPLES:")
display(df_water_historical_raw.head(9))

EXISTING LAKE NAMES:
['Flughafensee', 'Müggelsee', 'Orankesee', 'Tegeler See', 'Wannsee', 'Weißensee']

HISTORICAL STATION NAMES:
['Flughafensee', 'Müggelsee, Strandbad', 'Orankesee, Strandbad', 'Tegeler See, Freibad', 'Wannsee, Strandbad', 'Weißensee, Seebadeanstalt']

EXISTING DATASET STRUCTURE:
date                                    str
bathing_water_quality                   str
coliform_bacteria                       str
cyanobacteria_chl_a                     str
e_coli                                  str
intestinal_enterococci                  str
water_visibility_depth                  str
water_temperature                       str
lake                                    str
water_temperature_raw                   str
water_temperature_qualifier         float64
water_temperature_numeric           float64
temperature_zero_flag                  bool
coliform_bacteria_raw                   str
coliform_bacteria_qualifier             str
coliform_bacteria_numeric           float

,date,lake,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature
0,2025-04-29,Flughafensee,"< 15,00","< 15,00","> 1,00","16,00"
1,2025-05-15,Flughafensee,"< 15,00","< 15,00","> 1,00","17,50"
2,2025-05-26,Flughafensee,"30,00","15,00",NaN,"17,70"
3,2025-06-11,Flughafensee,"null 30,00","null 15,00","> 1,00","null 17,80"
4,2025-07-10,Flughafensee,"null 15,00","null 15,00","> 1,00","null 21,50"
5,2025-07-24,Flughafensee,null 143,null 109,"> 1,00","null 21,20"



HISTORICAL DATA EXAMPLES:


,Messstelle,Messstellennummer,Datum,Parameter,Vorzeichen,Wert,Einheit
0,Flughafensee,B920,2022-04-20,Escherichia coli,<,"15,0000",KBE/100 ml
1,Flughafensee,B920,2022-04-20,Intestinale Enterokokken,<,"15,0000",KBE/100 ml
2,Flughafensee,B920,2022-04-20,Sichttiefe,NaN,"0,0000",m
3,Flughafensee,B920,2022-05-17,Escherichia coli,<,"15,0000",KBE/100 ml
4,Flughafensee,B920,2022-05-17,Intestinale Enterokokken,<,"15,0000",KBE/100 ml
5,Flughafensee,B920,2022-05-17,Sichttiefe,NaN,"0,0000",m
6,Flughafensee,B920,2022-05-31,Escherichia coli,<,"15,0000",KBE/100 ml
7,Flughafensee,B920,2022-05-31,Intestinale Enterokokken,NaN,"15,0000",KBE/100 ml
8,Flughafensee,B920,2022-05-31,Sichttiefe,>,"1,0000",m


In [160]:
# Inspect raw-value conventions in the existing dataset
for parameter in [
    "e_coli",
    "intestinal_enterococci",
    "water_visibility_depth"
]:
    print(f"\n{parameter.upper()}")

    print("Existing raw values:")
    print(water_combined[f"{parameter}_raw"].dropna().unique()[:12])

    print("Existing qualifiers:")
    print(water_combined[f"{parameter}_qualifier"].value_counts(dropna=False))

    print("Existing numeric examples:")
    print(
        water_combined[
            [parameter, f"{parameter}_raw",
             f"{parameter}_qualifier", f"{parameter}_numeric"]
        ].head(8).to_string(index=False)
    )

# Inspect historical qualifiers and value formats
print("\nHISTORICAL QUALIFIERS:")
print(
    df_water_historical_raw.groupby(
        ["Parameter", "Vorzeichen"], dropna=False
    ).size()
)

print("\nHISTORICAL VALUE EXAMPLES:")
print(
    df_water_historical_raw[
        ["Parameter", "Vorzeichen", "Wert", "Einheit"]
    ].drop_duplicates().head(20).to_string(index=False)
)


E_COLI
Existing raw values:
<ArrowStringArray>
[   '< 15,00',      '30,00', 'null 30,00', 'null 15,00',   'null 143',
      '15,00',        '740',      '61,00', 'null 45,00', 'null 46,00',
        '159',        '251']
Length: 12, dtype: str
Existing qualifiers:
e_coli_qualifier
<      73
NaN    45
Name: count, dtype: int64
Existing numeric examples:
    e_coli e_coli_raw e_coli_qualifier  e_coli_numeric
   < 15,00    < 15,00                <            15.0
   < 15,00    < 15,00                <            15.0
     30,00      30,00              NaN            30.0
null 30,00 null 30,00              NaN            30.0
null 15,00 null 15,00              NaN            15.0
  null 143   null 143              NaN           143.0
null 15,00 null 15,00              NaN            15.0
null 30,00 null 30,00              NaN            30.0

INTESTINAL_ENTEROCOCCI
Existing raw values:
<ArrowStringArray>
['< 15,00', '15,00', 'null 15,00', 'null 109', '109', '30,00', '0,0000']
Length: 7, dtyp

In [161]:
# Map portal station names to existing lake names
lake_name_mapping = {
    "Flughafensee": "Flughafensee",
    "Müggelsee, Strandbad": "Müggelsee",
    "Orankesee, Strandbad": "Orankesee",
    "Tegeler See, Freibad": "Tegeler See",
    "Wannsee, Strandbad": "Wannsee",
    "Weißensee, Seebadeanstalt": "Weißensee"
}

# Map portal parameter names to dataset column names
parameter_mapping = {
    "Escherichia coli": "e_coli",
    "Intestinale Enterokokken": "intestinal_enterococci",
    "Sichttiefe": "water_visibility_depth"
}

# Copy the historical export
df_historical_long = df_water_historical_raw.copy()

# Standardise lake names
df_historical_long["lake"] = (
    df_historical_long["Messstelle"].map(lake_name_mapping)
)

# Standardise parameter names
df_historical_long["parameter"] = (
    df_historical_long["Parameter"].map(parameter_mapping)
)

# Convert the German decimal format into numeric values
df_historical_long["numeric_value"] = pd.to_numeric(
    df_historical_long["Wert"].str.replace(",", ".", regex=False),
    errors="coerce"
)

# Convert the date into ISO format
df_historical_long["date"] = (
    df_historical_long["Datum"].dt.strftime("%Y-%m-%d")
)

# Preserve the original raw measurement
df_historical_long["raw_value"] = (
    df_historical_long["Vorzeichen"].fillna("")
    + df_historical_long["Wert"]
)

# Create the display value using a decimal comma
df_historical_long["display_value"] = (
    df_historical_long["Vorzeichen"].fillna("")
    + " "
    + df_historical_long["Wert"]
).str.strip()

# Inspect the transformed records
display(
    df_historical_long[
        ["date", "lake", "parameter", "raw_value",
         "display_value", "numeric_value", "Einheit"]
    ].head(12)
)

print("\nUnmapped lake names:", df_historical_long["lake"].isna().sum())
print("Unmapped parameters:", df_historical_long["parameter"].isna().sum())
print("Missing numeric values:", df_historical_long["numeric_value"].isna().sum())

,date,lake,parameter,raw_value,display_value,numeric_value,Einheit
0,2022-04-20,Flughafensee,e_coli,"<15,0000","< 15,0000",15.0,KBE/100 ml
1,2022-04-20,Flughafensee,intestinal_enterococci,"<15,0000","< 15,0000",15.0,KBE/100 ml
2,2022-04-20,Flughafensee,water_visibility_depth,"0,0000","0,0000",0.0,m
3,2022-05-17,Flughafensee,e_coli,"<15,0000","< 15,0000",15.0,KBE/100 ml
4,2022-05-17,Flughafensee,intestinal_enterococci,"<15,0000","< 15,0000",15.0,KBE/100 ml
5,2022-05-17,Flughafensee,water_visibility_depth,"0,0000","0,0000",0.0,m
6,2022-05-31,Flughafensee,e_coli,"<15,0000","< 15,0000",15.0,KBE/100 ml
7,2022-05-31,Flughafensee,intestinal_enterococci,"15,0000","15,0000",15.0,KBE/100 ml
8,2022-05-31,Flughafensee,water_visibility_depth,">1,0000","> 1,0000",1.0,m
9,2022-06-13,Flughafensee,e_coli,"<15,0000","< 15,0000",15.0,KBE/100 ml



Unmapped lake names: 0
Unmapped parameters: 0
Missing numeric values: 0


In [163]:
# Create one row per unique lake and sampling date
df_water_historical = (
    df_historical_long[["date", "lake"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Add the remaining columns from the existing dataset
for column in water_combined.columns:
    if column not in df_water_historical.columns:
        df_water_historical[column] = pd.NA

# Populate the three available historical parameters
for parameter in parameter_mapping.values():

    subset = df_historical_long[
        df_historical_long["parameter"] == parameter
    ].copy()

    # Use date and lake to match each measurement to its row
    subset = subset.set_index(["date", "lake"])

    # Format values to match the existing dataset
    formatted_values = subset["numeric_value"].map(
        lambda value: f"{value:.2f}".replace(".", ",")
    )

    display_values = (
        subset["Vorzeichen"].fillna("").astype(str)
        + " "
        + formatted_values
    ).str.strip()

    # Populate display, raw, qualifier and numeric columns
    for column, values in {
        parameter: display_values,
        f"{parameter}_raw": display_values,
        f"{parameter}_qualifier": subset["Vorzeichen"],
        f"{parameter}_numeric": subset["numeric_value"]
    }.items():

        lookup = values.to_dict()

        df_water_historical[column] = [
            lookup.get((date, lake), pd.NA)
            for date, lake in zip(
                df_water_historical["date"],
                df_water_historical["lake"]
            )
        ]

# Restore the original column order
df_water_historical = df_water_historical[
    water_combined.columns
]

# Inspect the result
print("Historical dataset shape:", df_water_historical.shape)

print(
    "\nColumns match existing dataset:",
    list(df_water_historical.columns) == list(water_combined.columns)
)

print("\nRecords per lake:")
print(df_water_historical["lake"].value_counts().sort_index())

display(
    df_water_historical[
        ["date", "lake", "e_coli", "intestinal_enterococci",
         "water_visibility_depth", "water_temperature"]
    ].head(10)
)

Historical dataset shape: (180, 28)

Columns match existing dataset: True

Records per lake:
lake
Flughafensee    30
Müggelsee       31
Orankesee       29
Tegeler See     30
Wannsee         30
Weißensee       30
Name: count, dtype: int64


,date,lake,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature
0,2022-04-20,Flughafensee,"< 15,00","< 15,00","0,00",<NA>
1,2022-05-17,Flughafensee,"< 15,00","< 15,00","0,00",<NA>
2,2022-05-31,Flughafensee,"< 15,00","15,00","> 1,00",<NA>
3,2022-06-13,Flughafensee,"< 15,00","< 15,00","> 1,00",<NA>
4,2022-06-27,Flughafensee,"15,00","15,00","> 1,00",<NA>
5,2022-07-12,Flughafensee,"< 15,00","< 15,00","> 1,00",<NA>
6,2022-07-26,Flughafensee,"< 15,00","15,00","> 1,00",<NA>
7,2022-08-09,Flughafensee,"45,00","< 15,00","> 1,00",<NA>
8,2022-08-23,Flughafensee,"15,00","< 15,00","> 1,00",<NA>
9,2023-04-18,Flughafensee,"< 15,00","< 15,00","< 1,00",<NA>


In [164]:
# Check for duplicate lake/date combinations in the historical data
historical_duplicates = df_water_historical.duplicated(
    subset=["date", "lake"]
).sum()

# Check for duplicate lake/date combinations in the existing data
existing_duplicates = water_combined.duplicated(
    subset=["date", "lake"]
).sum()

# Identify any overlapping lake/date combinations
historical_keys = set(
    zip(df_water_historical["date"], df_water_historical["lake"])
)

existing_keys = set(
    zip(water_combined["date"], water_combined["lake"])
)

overlap = historical_keys.intersection(existing_keys)

print("Historical duplicate records:", historical_duplicates)
print("Existing duplicate records:", existing_duplicates)
print("Overlapping lake/date combinations:", len(overlap))

if overlap:
    print("\nOverlapping records:")
    print(sorted(overlap)[:10])

Historical duplicate records: 0
Existing duplicate records: 0
Overlapping lake/date combinations: 0


In [165]:
# Combine historical and existing water-quality datasets
df_water_final = pd.concat(
    [df_water_historical, water_combined],
    ignore_index=True
)

# Sort chronologically by lake and date
df_water_final["date"] = pd.to_datetime(df_water_final["date"])

df_water_final = (
    df_water_final
    .sort_values(["lake", "date"])
    .reset_index(drop=True)
)

# Convert dates back to the original string format
df_water_final["date"] = df_water_final["date"].dt.strftime("%Y-%m-%d")

# Validate the combined dataset
print("Final dataset shape:", df_water_final.shape)
print("\nDate range:")
print(df_water_final["date"].min(), "to", df_water_final["date"].max())

print("\nRecords per lake:")
print(df_water_final["lake"].value_counts().sort_index())

print("\nRecords by year:")
print(
    pd.to_datetime(df_water_final["date"])
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nDuplicate lake/date combinations:",
      df_water_final.duplicated(subset=["date", "lake"]).sum())

print("\nMissing values in measurement columns:")
measurement_columns = [
    "e_coli_numeric",
    "intestinal_enterococci_numeric",
    "water_visibility_depth_numeric",
    "water_temperature_numeric",
    "cyanobacteria_chl_a_numeric",
    "coliform_bacteria_numeric"
]

print(df_water_final[measurement_columns].isna().sum())

Final dataset shape: (298, 28)

Date range:
2022-04-20 to 2026-09-15

Records per lake:
lake
Flughafensee    49
Müggelsee       51
Orankesee       49
Tegeler See     50
Wannsee         49
Weißensee       50
Name: count, dtype: int64

Records by year:
date
2022    56
2023    61
2024    63
2025    58
2026    60
Name: count, dtype: int64

Duplicate lake/date combinations: 0

Missing values in measurement columns:
e_coli_numeric                      0
intestinal_enterococci_numeric      0
water_visibility_depth_numeric     19
water_temperature_numeric         183
cyanobacteria_chl_a_numeric       244
coliform_bacteria_numeric         181
dtype: int64


In [166]:
# Export the final combined water-quality dataset
output_path = "berlin_water_quality_2022_2026.csv"

df_water_final.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset exported successfully:", output_path)
print("Shape:", df_water_final.shape)

Dataset exported successfully: berlin_water_quality_2022_2026.csv
Shape: (298, 28)


In [167]:
# Create a year column for the coverage analysis
df_water_final["year"] = pd.to_datetime(
    df_water_final["date"]
).dt.year

# Define the measurement columns and their readable names
measurement_columns = {
    "e_coli_numeric": "E. coli",
    "intestinal_enterococci_numeric": "Intestinal enterococci",
    "water_visibility_depth_numeric": "Visibility depth",
    "water_temperature_numeric": "Water temperature",
    "cyanobacteria_chl_a_numeric": "Cyanobacteria (chlorophyll-a)",
    "coliform_bacteria_numeric": "Coliform bacteria"
}

# Summarise available and missing values by year
coverage_rows = []

for year, group in df_water_final.groupby("year"):

    for column, parameter in measurement_columns.items():

        total = len(group)
        available = group[column].notna().sum()
        missing = group[column].isna().sum()

        coverage_rows.append({
            "year": year,
            "parameter": parameter,
            "total_samples": total,
            "available": available,
            "missing": missing,
            "coverage_pct": round(available / total * 100, 1)
        })

df_water_coverage = pd.DataFrame(coverage_rows)

display(
    df_water_coverage.pivot(
        index="parameter",
        columns="year",
        values="coverage_pct"
    )
)

print("\nDetailed coverage:")
display(df_water_coverage)

year,2022,2023,2024,2025,2026
parameter,,,,,
Coliform bacteria,0.0,0.0,0.0,98.3,100.0
Cyanobacteria (chlorophyll-a),0.0,0.0,0.0,93.1,0.0
E. coli,100.0,100.0,100.0,100.0,100.0
Intestinal enterococci,100.0,100.0,100.0,100.0,100.0
Visibility depth,96.4,100.0,87.3,96.6,88.3
Water temperature,0.0,0.0,0.0,98.3,96.7



Detailed coverage:


,year,parameter,total_samples,available,missing,coverage_pct
0,2022,E. coli,56,56,0,100.0
1,2022,Intestinal enterococci,56,56,0,100.0
2,2022,Visibility depth,56,54,2,96.4
3,2022,Water temperature,56,0,56,0.0
4,2022,Cyanobacteria (chlorophyll-a),56,0,56,0.0
5,2022,Coliform bacteria,56,0,56,0.0
6,2023,E. coli,61,61,0,100.0
7,2023,Intestinal enterococci,61,61,0,100.0
8,2023,Visibility depth,61,61,0,100.0
9,2023,Water temperature,61,0,61,0.0


In [168]:
# Define the water-quality parameter data dictionary
water_parameter_dictionary = pd.DataFrame([
    {
        "parameter": "E. coli",
        "column": "e_coli_numeric",
        "unit": "KBE/100 ml",
        "description": "Concentration of Escherichia coli bacteria in bathing water.",
        "historical_coverage": "2022–2026",
        "analysis_notes": "Available throughout the full period. Qualifiers such as < are retained separately."
    },
    {
        "parameter": "Intestinal enterococci",
        "column": "intestinal_enterococci_numeric",
        "unit": "KBE/100 ml",
        "description": "Concentration of intestinal enterococci bacteria in bathing water.",
        "historical_coverage": "2022–2026",
        "analysis_notes": "Available throughout the full period. Qualifiers such as < are retained separately."
    },
    {
        "parameter": "Visibility depth",
        "column": "water_visibility_depth_numeric",
        "unit": "m",
        "description": "Measured depth at which visibility in the water is lost.",
        "historical_coverage": "2022–2026",
        "analysis_notes": "Some observations are missing. Values with < or > qualifiers are preserved."
    },
    {
        "parameter": "Water temperature",
        "column": "water_temperature_numeric",
        "unit": "To verify",
        "description": "Measured bathing-water temperature.",
        "historical_coverage": "2025–2026",
        "analysis_notes": "Not available in the historical 2022–2024 export."
    },
    {
        "parameter": "Cyanobacteria (chlorophyll-a)",
        "column": "cyanobacteria_chl_a_numeric",
        "unit": "To verify",
        "description": "Chlorophyll-a measurement used as an indicator related to cyanobacterial presence.",
        "historical_coverage": "2025 only",
        "analysis_notes": "No numeric values are present in 2026 or in the historical export."
    },
    {
        "parameter": "Coliform bacteria",
        "column": "coliform_bacteria_numeric",
        "unit": "To verify",
        "description": "Measured concentration of coliform bacteria in bathing water.",
        "historical_coverage": "2025–2026",
        "analysis_notes": "Not available in the historical 2022–2024 export."
    }
])

display(water_parameter_dictionary)

,parameter,column,unit,description,historical_coverage,analysis_notes
0,E. coli,e_coli_numeric,KBE/100 ml,Concentration of Escherichia coli bacteria in ...,2022–2026,Available throughout the full period. Qualifie...
1,Intestinal enterococci,intestinal_enterococci_numeric,KBE/100 ml,Concentration of intestinal enterococci bacter...,2022–2026,Available throughout the full period. Qualifie...
2,Visibility depth,water_visibility_depth_numeric,m,Measured depth at which visibility in the wate...,2022–2026,Some observations are missing. Values with < o...
3,Water temperature,water_temperature_numeric,To verify,Measured bathing-water temperature.,2025–2026,Not available in the historical 2022–2024 export.
4,Cyanobacteria (chlorophyll-a),cyanobacteria_chl_a_numeric,To verify,Chlorophyll-a measurement used as an indicator...,2025 only,No numeric values are present in 2026 or in th...
5,Coliform bacteria,coliform_bacteria_numeric,To verify,Measured concentration of coliform bacteria in...,2025–2026,Not available in the historical 2022–2024 export.


In [17]:
# Retrieve the official Berlin Water Portal station register

import pandas as pd

stations = pd.read_json(
    "https://kwb-r.github.io/wasserportal/stations_crosstable.json"
)

# Show stations with continuous water-temperature measurements
temperature_stations = stations[
    stations["owt"].eq("x")
][["Messstellennummer", "Messstellenname", "owt"]]

display(temperature_stations.to_string(index=False))

'Messstellennummer                        Messstellenname owt\n          5865900                  Allee der Kosmonauten   x\n          5870400                            Alsenbrücke   x\n          5865300                            Am Bahndamm   x\n          5861101                             Am Reitweg   x\n          5800107                  Am Wolfsschluchtkanal   x\n          5800317                   Biesdorfer Baggersee   x\n          5867003                           Blankenfelde   x\n          5867401                             Bürgerpark   x\n          5800301                               Dianasee   x\n          5800302                              Dreipfuhl   x\n          5867900                        Eisenbahnbrücke   x\n          5867601                             Eisstadion   x\n          5827101                        Fähre Rahnsdorf   x\n          5800320                              Fennpfuhl   x\n          5867101                       Flaischlenstraße   x\n       

In [2]:
# Test the Berlin Water Portal data endpoint: Flughafensee

import requests

url = "https://wasserportal.berlin.de/station.php"

params = {
    "anzeige": "d",
    "station": "5800303",
    "thema": "owt",
    "sreihe": "tw",
    "smode": "c",
    "sdatum": "01.01.2022"
}

response = requests.get(url, params=params, timeout=30)

print("Status:", response.status_code)
print("Final URL:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))
print("Content-Disposition:", response.headers.get("Content-Disposition"))
print("Response size:", len(response.content), "bytes")
print("First 200 bytes:")
print(response.content[:200])

Status: 200
Final URL: https://wasserportal.berlin.de/station.php?anzeige=d&station=5800303&thema=owt&sreihe=tw&smode=c&sdatum=01.01.2022
Content-Type: text/csv; charset=utf-8
Content-Disposition: attachment; filename=5800303_wassertemperatur_tw_01_01_2022.csv
Response size: 116371 bytes
First 200 bytes:
b'Datum;Tagesminimum;"Zeitpunkt Minimum";Tagesmittelwert;Tagesmaximum;"Zeitpunkt Maximum";"Stationsnummer: 5800303";"Stationsname: Flughafensee";"Gew\xe4sser: Flughafensee";"Wasssertemperatur in \xb0C";"Fehlw'


In [3]:
# Save the original CSV and load it into pandas

from pathlib import Path
from io import BytesIO

output_dir = Path("final_datasets")
output_dir.mkdir(exist_ok=True)

file_path = output_dir / "flughafensee_water_temperature_daily_2022_2026.csv"

file_path.write_bytes(response.content)

df_flughafensee = pd.read_csv(
    BytesIO(response.content),
    sep=";",
    encoding="cp1252"
)

print("Shape:", df_flughafensee.shape)
print("\nColumns:")
print(df_flughafensee.columns.tolist())

display(df_flughafensee.head())

Shape: (1774, 11)

Columns:
['Datum', 'Tagesminimum', 'Zeitpunkt Minimum', 'Tagesmittelwert', 'Tagesmaximum', 'Zeitpunkt Maximum', 'Stationsnummer: 5800303', 'Stationsname: Flughafensee', 'Gewässer: Flughafensee', 'Wasssertemperatur in °C', 'Fehlwerte: -777']


,Datum,Tagesminimum,Zeitpunkt Minimum,Tagesmittelwert,Tagesmaximum,Zeitpunkt Maximum,Stationsnummer: 5800303,Stationsname: Flughafensee,Gewässer: Flughafensee,Wasssertemperatur in °C,Fehlwerte: -777
0,01.01.2022,"5,30",01.01.2022 00:15,"5,78","6,10",01.01.2022 21:15,NaN,NaN,NaN,NaN,NaN
1,02.01.2022,"5,80",02.01.2022 22:15,"6,00","6,10",02.01.2022 00:15,NaN,NaN,NaN,NaN,NaN
2,03.01.2022,"5,80",03.01.2022 07:15,"5,94","6,10",03.01.2022 05:15,NaN,NaN,NaN,NaN,NaN
3,04.01.2022,"5,70",04.01.2022 04:15,"5,76","5,83",04.01.2022 00:00,NaN,NaN,NaN,NaN,NaN
4,05.01.2022,"5,33",06.01.2022 00:00,"5,48","5,70",05.01.2022 00:15,NaN,NaN,NaN,NaN,NaN


In [15]:
# Download daily water-temperature data from the two confirmed lake stations

import requests
from pathlib import Path

output_dir = Path("final_datasets")
output_dir.mkdir(exist_ok=True)

lake_stations = {
    "flughafensee": "5800303",
    "orankesee": "5800316"
}

downloaded_files = {}

for lake, station_id in lake_stations.items():

    params = {
        "anzeige": "d",
        "station": station_id,
        "thema": "owt",
        "sreihe": "tw",
        "smode": "c",
        "sdatum": "01.01.2022"
    }

    response = requests.get(
        "https://wasserportal.berlin.de/station.php",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    if "text/csv" not in response.headers.get("Content-Type", ""):
        print(f"{lake}: Unexpected response format")
        continue

    file_path = output_dir / f"{lake}_water_temperature_daily.csv"
    file_path.write_bytes(response.content)

    downloaded_files[lake] = file_path

    print(
        f"{lake}: downloaded successfully | "
        f"{len(response.content):,} bytes | "
        f"{file_path}"
    )

flughafensee: downloaded successfully | 116,436 bytes | final_datasets\flughafensee_water_temperature_daily.csv
orankesee: downloaded successfully | 116,999 bytes | final_datasets\orankesee_water_temperature_daily.csv


In [16]:
# Find sampling stations for the four remaining lakes

lake_names = [
    "Müggelsee",
    "Tegeler See",
    "Wannsee",
    "Weißensee"
]

for lake in lake_names:

    matches = stations[
        stations["Messstellenname"].str.contains(
            lake,
            case=False,
            na=False
        )
    ]

    print(f"\n{lake}")
    display(matches.to_string(index=False))

NameError: name 'stations' is not defined

In [6]:
# Test sampling-data API for Müggelsee

import requests

url = "https://wasserportal.berlin.de/station.php"

params = {
    "anzeige": "d",
    "station": "115",
    "thema": "opq",
    "sreihe": "ew",
    "smode": "c",
    "exportthema": "pq",
    "sdatum": "01.01.2022",
    "senddatum": "31.12.2024"
}

response_mueggelsee = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response_mueggelsee.status_code)
print("Content-Type:", response_mueggelsee.headers.get("Content-Type"))
print("Content-Disposition:", response_mueggelsee.headers.get("Content-Disposition"))
print("Response size:", len(response_mueggelsee.content), "bytes")
print("\nFirst 500 bytes:")
print(response_mueggelsee.content[:500])

Status: 200
Content-Type: text/csv; charset=utf-8
Content-Disposition: attachment; filename=115_wq_probenahme__01_01_2022.csv
Response size: 3077 bytes

First 500 bytes:
b'Messstellennummer;115\nRechtswert;407918,55\nHochwert;5810524,99\nProjektion;"UTM 33N"\n;\nMessstelle;Messstellennummer;Datum;Parameter;"Entnahmetiefe [m]";Messmethode;Vorzeichen;Wert;Einheit;Bestimmungsgrenze\n"Gro\xdfer M\xfcggelsee - Seemitte";115;11.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;2,2000;\xb0C;\n"Gro\xdfer M\xfcggelsee - Seemitte";115;15.02.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;4,3000;\xb0C;\n"Gro\xdfer M\xfcggelsee - Seemitte";115;09.03.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;4,6000;\xb0C;\n"Gro\xdfer M\xfc'


In [7]:
# Download historical water-temperature sampling data

from pathlib import Path
import requests

output_dir = Path("final_datasets")
output_dir.mkdir(exist_ok=True)

sampling_stations = {
    "mueggelsee": "115",
    "tegeler_see": "311",
    "wannsee_grosser": "350",
    "wannsee_kleiner": "355"
}

downloaded_sampling = {}

# Save the already verified Müggelsee response
file_path = output_dir / "mueggelsee_water_temperature_2022_2024.csv"
file_path.write_bytes(response_mueggelsee.content)
downloaded_sampling["mueggelsee"] = file_path

# Download the remaining stations
for lake, station_id in sampling_stations.items():

    if lake == "mueggelsee":
        continue

    params = {
        "anzeige": "d",
        "station": station_id,
        "thema": "opq",
        "sreihe": "ew",
        "smode": "c",
        "exportthema": "pq",
        "sdatum": "01.01.2022",
        "senddatum": "31.12.2024"
    }

    response = requests.get(
        "https://wasserportal.berlin.de/station.php",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    if "text/csv" not in response.headers.get("Content-Type", ""):
        print(f"{lake}: unexpected response format")
        continue

    file_path = output_dir / f"{lake}_water_temperature_2022_2024.csv"
    file_path.write_bytes(response.content)

    downloaded_sampling[lake] = file_path

    print(
        f"{lake}: downloaded | "
        f"{len(response.content):,} bytes"
    )

print("\nAll saved files:")
for lake, path in downloaded_sampling.items():
    print(lake, "->", path)

tegeler_see: downloaded | 4,019 bytes
wannsee_grosser: downloaded | 3,295 bytes
wannsee_kleiner: downloaded | 3,611 bytes

All saved files:
mueggelsee -> final_datasets\mueggelsee_water_temperature_2022_2024.csv
tegeler_see -> final_datasets\tegeler_see_water_temperature_2022_2024.csv
wannsee_grosser -> final_datasets\wannsee_grosser_water_temperature_2022_2024.csv
wannsee_kleiner -> final_datasets\wannsee_kleiner_water_temperature_2022_2024.csv


In [9]:
import pandas as pd

file_path = "final_datasets/mueggelsee_water_temperature_2022_2024.csv"

df_mueggelsee = pd.read_csv(
    file_path,
    sep=";",
    encoding="cp1252",
    decimal=","
)

display(df_mueggelsee.head())
print("Shape:", df_mueggelsee.shape)
print("Columns:", df_mueggelsee.columns.tolist())

ParserError: Error tokenizing data. C error: Expected 2 fields in line 6, saw 10


In [2]:
from pathlib import Path
import pandas as pd

sampling_dir = Path("final_datasets")

sampling_files = {
    "Müggelsee": "mueggelsee_water_temperature_2022_2024.csv",
    "Tegeler See": "tegeler_see_water_temperature_2022_2024.csv",
    "Großer Wannsee": "wannsee_grosser_water_temperature_2022_2024.csv",
    "Kleiner Wannsee": "wannsee_kleiner_water_temperature_2022_2024.csv"
}

sampling_summary = []

for lake, filename in sampling_files.items():
    file_path = sampling_dir / filename

    # Read the measurement table after the four metadata rows
    df = pd.read_csv(
        file_path,
        sep=";",
        decimal=",",
        skiprows=4,
        encoding="utf-8-sig"
    )

    # Clean and parse dates
    df["Datum"] = pd.to_datetime(df["Datum"], dayfirst=True, errors="coerce")

    # Keep water-temperature measurements
    temp_df = df[df["Parameter"].str.contains(
        "Wassertemperatur", case=False, na=False
    )].copy()

    sampling_summary.append({
        "lake": lake,
        "total_rows": len(df),
        "temperature_rows": len(temp_df),
        "first_date": temp_df["Datum"].min(),
        "last_date": temp_df["Datum"].max(),
        "unique_sampling_dates": temp_df["Datum"].nunique(),
        "depths_m": sorted(temp_df["Entnahmetiefe [m]"].dropna().unique().tolist())
    })

    print(f"\n{lake}")
    print("Parameters:", df["Parameter"].dropna().unique().tolist())
    print("Temperature measurements:", len(temp_df))

sampling_summary_df = pd.DataFrame(sampling_summary)

print("\nDate coverage summary:")
display(sampling_summary_df)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xdf in position 209: invalid continuation byte

In [3]:
df_mueggelsee = pd.read_csv(
    file_path,
    sep=";",
    encoding="cp1252",
    decimal=",",
    skiprows=4
)

display(df_mueggelsee.head())
print("Shape:", df_mueggelsee.shape)
print("Columns:", df_mueggelsee.columns.tolist())

Unnamed: 0  \
Messstelle                  Messstellennummer Datum      Parameter        Entnahmetiefe [m] Messmethode    Vorzeichen Wert      Einheit   
Großer Müggelsee - Seemitte 115               11.01.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        2,2000         °C   
                                              15.02.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        4,3000         °C   
                                              09.03.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        4,6000         °C   
                                              05.04.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        6,5000         °C   

                                                                                                                                     Unnamed: 1  
Messstelle                  Messstellennummer Datum      Parameter        Entnahmetiefe [m] Messmethode    Vorzeichen Wert    Bestimmungsgrenze  
Großer Müggelsee - Seemitte 115               11.01.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        2,2000                NaN  
                                              15.02.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        4,3000                NaN  
                                              09.03.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        4,6000                NaN  
                                              05.04.2022 Wassertemperatur ,50               DIN 38404 (C4) nan        6,5000                NaN

Shape: (31, 2)
Columns: ['Unnamed: 0', 'Unnamed: 1']


In [4]:
with open(file_path, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 12:
            break

Line 0: 'Messstellennummer;115'
Line 1: 'Rechtswert;407918,55'
Line 2: 'Hochwert;5810524,99'
Line 3: 'Projektion;"UTM 33N"'
Line 4: ';'
Line 5: 'Messstelle;Messstellennummer;Datum;Parameter;"Entnahmetiefe [m]";Messmethode;Vorzeichen;Wert;Einheit;Bestimmungsgrenze'
Line 6: '"Großer Müggelsee - Seemitte";115;11.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;2,2000;°C;'
Line 7: '"Großer Müggelsee - Seemitte";115;15.02.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;4,3000;°C;'
Line 8: '"Großer Müggelsee - Seemitte";115;09.03.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;4,6000;°C;'
Line 9: '"Großer Müggelsee - Seemitte";115;05.04.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;6,5000;°C;'
Line 10: '"Großer Müggelsee - Seemitte";115;03.05.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;14,6000;°C;'
Line 11: '"Großer Müggelsee - Seemitte";115;09.06.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;20,2000;°C;'
Line 12: '"Großer Müggelsee - Seemitte";115;06.07.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;23,1000

In [5]:
df_mueggelsee = pd.read_csv(
    file_path,
    sep=";",
    encoding="cp1252",
    decimal=",",
    skiprows=5
)

display(df_mueggelsee.head())
print("Shape:", df_mueggelsee.shape)
print("Columns:", df_mueggelsee.columns.tolist())
print("\nMissing values:")
print(df_mueggelsee.isna().sum())

,Messstelle,Messstellennummer,Datum,Parameter,Entnahmetiefe [m],Messmethode,Vorzeichen,Wert,Einheit,Bestimmungsgrenze
0,Großer Müggelsee - Seemitte,115,11.01.2022,Wassertemperatur,0.5,DIN 38404 (C4),NaN,2.2,°C,NaN
1,Großer Müggelsee - Seemitte,115,15.02.2022,Wassertemperatur,0.5,DIN 38404 (C4),NaN,4.3,°C,NaN
2,Großer Müggelsee - Seemitte,115,09.03.2022,Wassertemperatur,0.5,DIN 38404 (C4),NaN,4.6,°C,NaN
3,Großer Müggelsee - Seemitte,115,05.04.2022,Wassertemperatur,0.5,DIN 38404 (C4),NaN,6.5,°C,NaN
4,Großer Müggelsee - Seemitte,115,03.05.2022,Wassertemperatur,0.5,DIN 38404 (C4),NaN,14.6,°C,NaN


Shape: (30, 10)
Columns: ['Messstelle', 'Messstellennummer', 'Datum', 'Parameter', 'Entnahmetiefe [m]', 'Messmethode', 'Vorzeichen', 'Wert', 'Einheit', 'Bestimmungsgrenze']

Missing values:
Messstelle            0
Messstellennummer     0
Datum                 0
Parameter             0
Entnahmetiefe [m]     0
Messmethode           0
Vorzeichen           30
Wert                  0
Einheit               0
Bestimmungsgrenze    30
dtype: int64


In [6]:
df_mueggelsee["Datum"] = pd.to_datetime(
    df_mueggelsee["Datum"],
    format="%d.%m.%Y"
)

print("Date range:", df_mueggelsee["Datum"].min(), "to", df_mueggelsee["Datum"].max())

print("\nMeasurements per year:")
print(df_mueggelsee.groupby(df_mueggelsee["Datum"].dt.year).size())

print("\nDuplicate dates:", df_mueggelsee["Datum"].duplicated().sum())

print("\nTemperature summary:")
print(df_mueggelsee["Wert"].describe())

Date range: 2022-01-11 00:00:00 to 2024-12-03 00:00:00

Measurements per year:
Datum
2022    12
2023     9
2024     9
dtype: int64

Duplicate dates: 0

Temperature summary:
count    30.000000
mean     14.366667
std       7.032993
min       2.200000
25%       7.650000
50%      14.550000
75%      20.450000
max      23.500000
Name: Wert, dtype: float64


In [7]:
from pathlib import Path

file_path_tegeler = Path(
    "final_datasets/tegeler_see_water_temperature_2022_2024.csv"
)

with open(file_path_tegeler, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 8:
            break

Line 0: 'Messstellennummer;311'
Line 1: 'Rechtswert;382466,46'
Line 2: 'Hochwert;5827122,82'
Line 3: 'Projektion;"UTM 33N"'
Line 4: ';'
Line 5: 'Messstelle;Messstellennummer;Datum;Parameter;"Entnahmetiefe [m]";Messmethode;Vorzeichen;Wert;Einheit;Bestimmungsgrenze'
Line 6: '"Tegeler See - Seemitte";311;03.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;5,3000;°C;'
Line 7: '"Tegeler See - Seemitte";311;31.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;3,2000;°C;'
Line 8: '"Tegeler See - Seemitte";311;28.03.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;8,1000;°C;'


In [8]:
df_tegeler = pd.read_csv(
    file_path_tegeler,
    sep=";",
    encoding="cp1252",
    decimal=",",
    skiprows=5
)

df_tegeler["Datum"] = pd.to_datetime(
    df_tegeler["Datum"],
    format="%d.%m.%Y"
)

print("Shape:", df_tegeler.shape)
print("Date range:", df_tegeler["Datum"].min(), "to", df_tegeler["Datum"].max())

print("\nMeasurements per year:")
print(df_tegeler.groupby(df_tegeler["Datum"].dt.year).size())

print("\nTemperature summary:")
print(df_tegeler["Wert"].describe())

display(df_tegeler.head())

Shape: (42, 10)
Date range: 2022-01-03 00:00:00 to 2024-10-30 00:00:00

Measurements per year:
Datum
2022    15
2023    14
2024    13
dtype: int64

Temperature summary:
count    42.000000
mean     17.285714
std       6.500040
min       3.200000
25%      13.425000
50%      19.650000
75%      22.425000
max      25.600000
Name: Wert, dtype: float64


,Messstelle,Messstellennummer,Datum,Parameter,Entnahmetiefe [m],Messmethode,Vorzeichen,Wert,Einheit,Bestimmungsgrenze
0,Tegeler See - Seemitte,311,2022-01-03,Wassertemperatur,0.5,DIN 38404 (C4),NaN,5.3,°C,NaN
1,Tegeler See - Seemitte,311,2022-01-31,Wassertemperatur,0.5,DIN 38404 (C4),NaN,3.2,°C,NaN
2,Tegeler See - Seemitte,311,2022-03-28,Wassertemperatur,0.5,DIN 38404 (C4),NaN,8.1,°C,NaN
3,Tegeler See - Seemitte,311,2022-04-25,Wassertemperatur,0.5,DIN 38404 (C4),NaN,10.7,°C,NaN
4,Tegeler See - Seemitte,311,2022-05-16,Wassertemperatur,0.5,DIN 38404 (C4),NaN,17.6,°C,NaN


In [9]:
file_path_wannsee = "final_datasets/wannsee_grosser_water_temperature_2022_2024.csv"

with open(file_path_wannsee, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 8:
            break

Line 0: 'Messstellennummer;350'
Line 1: 'Rechtswert;375928,49'
Line 2: 'Hochwert;5811447,98'
Line 3: 'Projektion;"UTM 33N"'
Line 4: ';'
Line 5: 'Messstelle;Messstellennummer;Datum;Parameter;"Entnahmetiefe [m]";Messmethode;Vorzeichen;Wert;Einheit;Bestimmungsgrenze'
Line 6: '"Großer Wannsee - Höhe Strandbad";350;04.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;5,2000;°C;'
Line 7: '"Großer Wannsee - Höhe Strandbad";350;01.02.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;3,3000;°C;'
Line 8: '"Großer Wannsee - Höhe Strandbad";350;01.03.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;5,3000;°C;'


In [10]:
df_wannsee_grosser = pd.read_csv(
    file_path_wannsee,
    sep=";",
    encoding="cp1252",
    decimal=",",
    skiprows=5
)

df_wannsee_grosser["Datum"] = pd.to_datetime(
    df_wannsee_grosser["Datum"],
    format="%d.%m.%Y"
)

print("Shape:", df_wannsee_grosser.shape)
print("Date range:", df_wannsee_grosser["Datum"].min(), "to", df_wannsee_grosser["Datum"].max())

print("\nMeasurements per year:")
print(df_wannsee_grosser.groupby(df_wannsee_grosser["Datum"].dt.year).size())

print("\nTemperature summary:")
print(df_wannsee_grosser["Wert"].describe())

display(df_wannsee_grosser.head())

Shape: (31, 10)
Date range: 2022-01-04 00:00:00 to 2024-10-29 00:00:00

Measurements per year:
Datum
2022    12
2023    11
2024     8
dtype: int64

Temperature summary:
count    31.000000
mean     14.367742
std       7.015335
min       3.300000
25%       7.950000
50%      14.200000
75%      20.950000
max      24.500000
Name: Wert, dtype: float64


,Messstelle,Messstellennummer,Datum,Parameter,Entnahmetiefe [m],Messmethode,Vorzeichen,Wert,Einheit,Bestimmungsgrenze
0,Großer Wannsee - Höhe Strandbad,350,2022-01-04,Wassertemperatur,0.5,DIN 38404 (C4),NaN,5.2,°C,NaN
1,Großer Wannsee - Höhe Strandbad,350,2022-02-01,Wassertemperatur,0.5,DIN 38404 (C4),NaN,3.3,°C,NaN
2,Großer Wannsee - Höhe Strandbad,350,2022-03-01,Wassertemperatur,0.5,DIN 38404 (C4),NaN,5.3,°C,NaN
3,Großer Wannsee - Höhe Strandbad,350,2022-03-29,Wassertemperatur,0.5,DIN 38404 (C4),NaN,9.0,°C,NaN
4,Großer Wannsee - Höhe Strandbad,350,2022-04-26,Wassertemperatur,0.5,DIN 38404 (C4),NaN,12.2,°C,NaN


In [11]:
file_path_wannsee_klein = "final_datasets/wannsee_kleiner_water_temperature_2022_2024.csv"

with open(file_path_wannsee_klein, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 8:
            break

Line 0: 'Messstellennummer;355'
Line 1: 'Rechtswert;375162,61'
Line 2: 'Hochwert;5809032,63'
Line 3: 'Projektion;"UTM 33N"'
Line 4: ';'
Line 5: 'Messstelle;Messstellennummer;Datum;Parameter;"Entnahmetiefe [m]";Messmethode;Vorzeichen;Wert;Einheit;Bestimmungsgrenze'
Line 6: '"Kleiner Wannsee - Seemitte";355;04.01.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;7,6000;°C;'
Line 7: '"Kleiner Wannsee - Seemitte";355;01.02.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;4,6000;°C;'
Line 8: '"Kleiner Wannsee - Seemitte";355;01.03.2022;Wassertemperatur;,50;"DIN 38404 (C4)";;5,8000;°C;'


In [12]:
df_wannsee_klein = pd.read_csv(
    file_path_wannsee_klein,
    sep=";",
    encoding="cp1252",
    decimal=",",
    skiprows=5
)

df_wannsee_klein["Datum"] = pd.to_datetime(
    df_wannsee_klein["Datum"],
    format="%d.%m.%Y"
)

print("Shape:", df_wannsee_klein.shape)
print("Date range:", df_wannsee_klein["Datum"].min(), "to", df_wannsee_klein["Datum"].max())

print("\nMeasurements per year:")
print(df_wannsee_klein.groupby(df_wannsee_klein["Datum"].dt.year).size())

print("\nTemperature summary:")
print(df_wannsee_klein["Wert"].describe())

display(df_wannsee_klein.head())

Shape: (36, 10)
Date range: 2022-01-04 00:00:00 to 2024-11-26 00:00:00

Measurements per year:
Datum
2022    12
2023    12
2024    12
dtype: int64

Temperature summary:
count    36.000000
mean     13.633333
std       7.045607
min       3.000000
25%       6.925000
50%      13.900000
75%      20.525000
max      26.500000
Name: Wert, dtype: float64


,Messstelle,Messstellennummer,Datum,Parameter,Entnahmetiefe [m],Messmethode,Vorzeichen,Wert,Einheit,Bestimmungsgrenze
0,Kleiner Wannsee - Seemitte,355,2022-01-04,Wassertemperatur,0.5,DIN 38404 (C4),NaN,7.6,°C,NaN
1,Kleiner Wannsee - Seemitte,355,2022-02-01,Wassertemperatur,0.5,DIN 38404 (C4),NaN,4.6,°C,NaN
2,Kleiner Wannsee - Seemitte,355,2022-03-01,Wassertemperatur,0.5,DIN 38404 (C4),NaN,5.8,°C,NaN
3,Kleiner Wannsee - Seemitte,355,2022-03-29,Wassertemperatur,0.5,DIN 38404 (C4),NaN,10.6,°C,NaN
4,Kleiner Wannsee - Seemitte,355,2022-04-26,Wassertemperatur,0.5,DIN 38404 (C4),NaN,14.5,°C,NaN


In [18]:
stations[
    stations["Messstellenname"].str.contains(
        r"weiß|weiss|wei",
        case=False,
        na=False,
        regex=True
    )
][["Messstellennummer", "Messstellenname", "opq", "owt"]]

,Messstellennummer,Messstellenname,opq,owt
61,5800314,Weißer See,NaN,NaN


In [19]:
stations.loc[
    stations["Messstellennummer"].astype(str) == "5800314"
].T

,61
Messstellennummer,5800314
Messstellenname,Weißer See
ows,x
owt,NaN
odf,NaN
olf,NaN
oph,NaN
oog,NaN
oos,NaN
opq,NaN


In [20]:
import requests
from pathlib import Path

output_dir = Path("final_datasets")
output_dir.mkdir(exist_ok=True)

continuous_stations = {
    "flughafensee": "5800303",
    "orankesee": "5800316"
}

downloaded_continuous = {}

for lake, station_id in continuous_stations.items():

    params = {
        "anzeige": "d",
        "station": station_id,
        "thema": "owt",
        "sreihe": "tw",
        "smode": "c",
        "sdatum": "01.01.2022"
    }

    response = requests.get(
        "https://wasserportal.berlin.de/station.php",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    if "text/csv" not in response.headers.get("Content-Type", ""):
        print(f"{lake}: Unexpected response format")
        continue

    file_path = output_dir / f"{lake}_water_temperature_daily.csv"
    file_path.write_bytes(response.content)

    downloaded_continuous[lake] = file_path

    print(f"{lake}: downloaded | {len(response.content):,} bytes")

print("\nSaved files:")
for lake, path in downloaded_continuous.items():
    print(f"{lake}: {path}")

flughafensee: downloaded | 116,436 bytes
orankesee: downloaded | 116,999 bytes

Saved files:
flughafensee: final_datasets\flughafensee_water_temperature_daily.csv
orankesee: final_datasets\orankesee_water_temperature_daily.csv


In [21]:
file_path_flughafen = "final_datasets/flughafensee_water_temperature_daily.csv"

with open(file_path_flughafen, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 12:
            break

Line 0: 'Datum;Tagesminimum;"Zeitpunkt Minimum";Tagesmittelwert;Tagesmaximum;"Zeitpunkt Maximum";"Stationsnummer: 5800303";"Stationsname: Flughafensee";"Gewässer: Flughafensee";"Wasssertemperatur in °C";"Fehlwerte: -777"'
Line 1: '01.01.2022;5,30;"01.01.2022 00:15";5,78;6,10;"01.01.2022 21:15"'
Line 2: '02.01.2022;5,80;"02.01.2022 22:15";6,00;6,10;"02.01.2022 00:15"'
Line 3: '03.01.2022;5,80;"03.01.2022 07:15";5,94;6,10;"03.01.2022 05:15"'
Line 4: '04.01.2022;5,70;"04.01.2022 04:15";5,76;5,83;"04.01.2022 00:00"'
Line 5: '05.01.2022;5,33;"06.01.2022 00:00";5,48;5,70;"05.01.2022 00:15"'
Line 6: '06.01.2022;4,90;"06.01.2022 19:15";5,23;5,50;"06.01.2022 12:15"'
Line 7: '07.01.2022;5,10;"07.01.2022 00:15";4,79;5,10;"07.01.2022 00:15"'
Line 8: '08.01.2022;4,33;"09.01.2022 00:00";4,62;4,80;"08.01.2022 12:15"'
Line 9: '09.01.2022;4,30;"09.01.2022 00:15";4,47;4,60;"09.01.2022 15:15"'
Line 10: '10.01.2022;4,43;"11.01.2022 00:00";4,58;4,60;"10.01.2022 00:15"'
Line 11: '11.01.2022;3,93;"12.01.2022

In [22]:
df_flughafen = pd.read_csv(
    file_path_flughafen,
    sep=";",
    encoding="cp1252",
    decimal=",",
    na_values=-777
)

df_flughafen["Datum"] = pd.to_datetime(
    df_flughafen["Datum"],
    format="%d.%m.%Y"
)

temperature_columns = [
    "Tagesminimum",
    "Tagesmittelwert",
    "Tagesmaximum"
]

print("Shape:", df_flughafen.shape)
print("Date range:", df_flughafen["Datum"].min(), "to", df_flughafen["Datum"].max())

print("\nMissing values:")
print(df_flughafen[temperature_columns].isna().sum())

print("\nTemperature summary:")
display(df_flughafen[temperature_columns].describe())

print("\nFirst rows:")
display(df_flughafen.head())

Shape: (1775, 11)
Date range: 2022-01-01 00:00:00 to 2026-11-10 00:00:00

Missing values:
Tagesminimum       19
Tagesmittelwert    19
Tagesmaximum       19
dtype: int64

Temperature summary:


,Tagesminimum,Tagesmittelwert,Tagesmaximum
count,1756.000000,1756.000000,1756.000000
mean,12.683161,13.180700,13.824055
std,7.306153,7.457711,7.699203
min,-0.400000,-0.300000,-0.300000
25%,5.500000,5.767500,6.100000
50%,12.760000,13.230000,14.000000
75%,20.000000,20.610000,21.300000
max,26.900000,28.000000,29.600000



First rows:


,Datum,Tagesminimum,Zeitpunkt Minimum,Tagesmittelwert,Tagesmaximum,Zeitpunkt Maximum,Stationsnummer: 5800303,Stationsname: Flughafensee,Gewässer: Flughafensee,Wasssertemperatur in °C,Fehlwerte: -777
0,2022-01-01,5.30,01.01.2022 00:15,5.78,6.10,01.01.2022 21:15,NaN,NaN,NaN,NaN,NaN
1,2022-01-02,5.80,02.01.2022 22:15,6.00,6.10,02.01.2022 00:15,NaN,NaN,NaN,NaN,NaN
2,2022-01-03,5.80,03.01.2022 07:15,5.94,6.10,03.01.2022 05:15,NaN,NaN,NaN,NaN,NaN
3,2022-01-04,5.70,04.01.2022 04:15,5.76,5.83,04.01.2022 00:00,NaN,NaN,NaN,NaN,NaN
4,2022-01-05,5.33,06.01.2022 00:00,5.48,5.70,05.01.2022 00:15,NaN,NaN,NaN,NaN,NaN


In [23]:
# Keep only the relevant measurement columns
df_flughafen = df_flughafen[
    ["Datum", "Tagesminimum", "Tagesmittelwert", "Tagesmaximum"]
].copy()

# Remove rows without a daily mean temperature
df_flughafen = df_flughafen.dropna(
    subset=["Tagesmittelwert"]
)

# Restrict the dataset to the common analysis period
df_flughafen = df_flughafen[
    df_flughafen["Datum"].between("2022-01-01", "2024-12-31")
].copy()

# Create a monthly period column
df_flughafen["Monat"] = df_flughafen["Datum"].dt.to_period("M")

# Check the cleaned dataset
print("Shape:", df_flughafen.shape)
print("Date range:", df_flughafen["Datum"].min(), "to", df_flughafen["Datum"].max())

print("\nRecords per year:")
print(df_flughafen.groupby(df_flughafen["Datum"].dt.year).size())

print("\nMissing values:")
print(df_flughafen.isna().sum())

display(df_flughafen.head())

Shape: (1096, 5)
Date range: 2022-01-01 00:00:00 to 2024-12-31 00:00:00

Records per year:
Datum
2022    365
2023    365
2024    366
dtype: int64

Missing values:
Datum              0
Tagesminimum       0
Tagesmittelwert    0
Tagesmaximum       0
Monat              0
dtype: int64


,Datum,Tagesminimum,Tagesmittelwert,Tagesmaximum,Monat
0,2022-01-01,5.30,5.78,6.10,2022-01
1,2022-01-02,5.80,6.00,6.10,2022-01
2,2022-01-03,5.80,5.94,6.10,2022-01
3,2022-01-04,5.70,5.76,5.83,2022-01
4,2022-01-05,5.33,5.48,5.70,2022-01


In [24]:
# Define the Orankesee file path
file_path_orankesee = "final_datasets/orankesee_water_temperature_daily.csv"

# Inspect the first 13 lines to identify the header and data format
with open(file_path_orankesee, "r", encoding="cp1252") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line.rstrip()!r}")
        if i >= 12:
            break

Line 0: 'Datum;Tagesminimum;"Zeitpunkt Minimum";Tagesmittelwert;Tagesmaximum;"Zeitpunkt Maximum";"Stationsnummer: 5800316";"Stationsname: Orankesee";"Gewässer: Orankesee";"Wasssertemperatur in °C";"Fehlwerte: -777"'
Line 1: '01.01.2022;5,20;"01.01.2022 00:30";5,59;6,30;"01.01.2022 23:30"'
Line 2: '02.01.2022;5,70;"02.01.2022 22:30";6,19;6,70;"02.01.2022 15:30"'
Line 3: '03.01.2022;5,70;"03.01.2022 00:30";6,09;6,40;"03.01.2022 05:30"'
Line 4: '04.01.2022;6,00;"04.01.2022 01:30";6,22;6,50;"04.01.2022 16:30"'
Line 5: '05.01.2022;6,00;"05.01.2022 07:30";6,13;6,30;"05.01.2022 00:30"'
Line 6: '06.01.2022;5,05;"07.01.2022 00:00";5,68;6,05;"06.01.2022 00:00"'
Line 7: '07.01.2022;5,05;"07.01.2022 00:00";4,95;5,05;"07.01.2022 00:00"'
Line 8: '08.01.2022;4,35;"09.01.2022 00:00";4,77;5,00;"08.01.2022 16:30"'
Line 9: '09.01.2022;3,60;"09.01.2022 06:30";3,96;4,35;"09.01.2022 00:00"'
Line 10: '10.01.2022;3,95;"11.01.2022 00:00";4,32;4,50;"10.01.2022 14:30"'
Line 11: '11.01.2022;3,30;"11.01.2022 02:30

In [25]:
# Read the daily monitoring CSV
df_orankesee = pd.read_csv(
    file_path_orankesee,
    sep=";",
    encoding="cp1252",
    decimal=",",
    na_values=-777
)

# Convert the date column to datetime
df_orankesee["Datum"] = pd.to_datetime(
    df_orankesee["Datum"],
    format="%d.%m.%Y"
)

# Keep only date and temperature measurements
df_orankesee = df_orankesee[
    ["Datum", "Tagesminimum", "Tagesmittelwert", "Tagesmaximum"]
].copy()

# Remove observations without a daily mean
df_orankesee = df_orankesee.dropna(
    subset=["Tagesmittelwert"]
)

# Restrict the analysis to 2022–2024
df_orankesee = df_orankesee[
    df_orankesee["Datum"].between("2022-01-01", "2024-12-31")
].copy()

# Create a monthly period column
df_orankesee["Monat"] = df_orankesee["Datum"].dt.to_period("M")

# Check the cleaned dataset
print("Shape:", df_orankesee.shape)
print("Date range:", df_orankesee["Datum"].min(), "to", df_orankesee["Datum"].max())

print("\nRecords per year:")
print(df_orankesee.groupby(df_orankesee["Datum"].dt.year).size())

print("\nMissing values:")
print(df_orankesee.isna().sum())

display(df_orankesee.head())

Shape: (1096, 5)
Date range: 2022-01-01 00:00:00 to 2024-12-31 00:00:00

Records per year:
Datum
2022    365
2023    365
2024    366
dtype: int64

Missing values:
Datum              0
Tagesminimum       0
Tagesmittelwert    0
Tagesmaximum       0
Monat              0
dtype: int64


,Datum,Tagesminimum,Tagesmittelwert,Tagesmaximum,Monat
0,2022-01-01,5.2,5.59,6.3,2022-01
1,2022-01-02,5.7,6.19,6.7,2022-01
2,2022-01-03,5.7,6.09,6.4,2022-01
3,2022-01-04,6.0,6.22,6.5,2022-01
4,2022-01-05,6.0,6.13,6.3,2022-01


In [26]:
# Define a reusable function for monthly aggregation
def calculate_monthly_temperature(df, lake_name):

    # Group daily observations by calendar month
    monthly = (
        df.groupby("Monat")
        .agg(
            monthly_mean_temp=("Tagesmittelwert", "mean"),
            monthly_min_temp=("Tagesminimum", "mean"),
            monthly_max_temp=("Tagesmaximum", "mean"),
            n_obs=("Tagesmittelwert", "count")
        )
        .reset_index()
    )

    # Add the lake name
    monthly["lake"] = lake_name

    return monthly


# Calculate monthly averages for both continuously monitored lakes
monthly_flughafensee = calculate_monthly_temperature(
    df_flughafen, "Flughafensee"
)

monthly_orankesee = calculate_monthly_temperature(
    df_orankesee, "Orankesee"
)

# Combine both monthly datasets
monthly_continuous = pd.concat(
    [monthly_flughafensee, monthly_orankesee],
    ignore_index=True
)

# Display the results
display(monthly_continuous.head(15))

# Check the number of monthly records and observations per lake
print("\nMonthly records per lake:")
print(monthly_continuous.groupby("lake").size())

print("\nDaily observations per month:")
print(
    monthly_continuous.groupby("lake")["n_obs"]
    .agg(["min", "median", "max"])
)

,Monat,monthly_mean_temp,monthly_min_temp,monthly_max_temp,n_obs,lake
0,2022-01,4.399355,4.242903,4.551935,31,Flughafensee
1,2022-02,4.557143,4.382143,4.835000,28,Flughafensee
2,2022-03,6.088387,5.691935,6.612581,31,Flughafensee
3,2022-04,9.497000,9.025333,10.001000,30,Flughafensee
4,2022-05,18.196129,17.506129,18.934194,31,Flughafensee
5,2022-06,22.236000,21.443667,23.081667,30,Flughafensee
6,2022-07,22.753226,22.058710,23.471613,31,Flughafensee
7,2022-08,24.043871,23.399677,24.765161,31,Flughafensee
8,2022-09,18.007333,17.592333,18.464667,30,Flughafensee
9,2022-10,14.263871,13.984194,14.583548,31,Flughafensee



Monthly records per lake:
lake
Flughafensee    36
Orankesee       36
dtype: int64

Daily observations per month:
              min  median  max
lake                          
Flughafensee   28    31.0   31
Orankesee      28    31.0   31


In [27]:
# Define the files and their corresponding lake names
sampling_files = {
    "Müggelsee": "final_datasets/mueggelsee_water_temperature_2022_2024.csv",
    "Tegeler See": "final_datasets/tegeler_see_water_temperature_2022_2024.csv",
    "Großer Wannsee": "final_datasets/wannsee_grosser_water_temperature_2022_2024.csv",
    "Kleiner Wannsee": "final_datasets/wannsee_kleiner_water_temperature_2022_2024.csv"
}

# Create an empty list to store the monthly results
monthly_sampling_list = []

# Process each lake separately
for lake_name, file_path in sampling_files.items():

    # Read the sampling dataset
    df = pd.read_csv(
        file_path,
        sep=";",
        encoding="cp1252",
        decimal=",",
        skiprows=5
    )

    # Convert the date column
    df["Datum"] = pd.to_datetime(
        df["Datum"],
        format="%d.%m.%Y"
    )

    # Create a calendar-month column
    df["Monat"] = df["Datum"].dt.to_period("M")

    # Calculate monthly mean temperature and observation count
    monthly = (
        df.groupby("Monat")
        .agg(
            monthly_mean_temp=("Wert", "mean"),
            n_obs=("Wert", "count")
        )
        .reset_index()
    )

    # Add lake name and measurement method
    monthly["lake"] = lake_name
    monthly["measurement_method"] = "Periodic sampling"

    monthly_sampling_list.append(monthly)


# Combine the four lakes
monthly_sampling = pd.concat(
    monthly_sampling_list,
    ignore_index=True
)

# Display the monthly results
display(monthly_sampling.head(15))

# Check monthly coverage and sample counts
print("\nMonthly records per lake:")
print(monthly_sampling.groupby("lake").size())

print("\nObservations per month:")
display(
    monthly_sampling.groupby("lake")["n_obs"]
    .agg(["min", "median", "max"])
)

,Monat,monthly_mean_temp,n_obs,lake,measurement_method
0,2022-01,2.20,1,Müggelsee,Periodic sampling
1,2022-02,4.30,1,Müggelsee,Periodic sampling
2,2022-03,4.60,1,Müggelsee,Periodic sampling
3,2022-04,6.50,1,Müggelsee,Periodic sampling
4,2022-05,14.60,1,Müggelsee,Periodic sampling
5,2022-06,20.20,1,Müggelsee,Periodic sampling
6,2022-07,23.10,1,Müggelsee,Periodic sampling
7,2022-08,22.25,2,Müggelsee,Periodic sampling
8,2022-09,14.30,1,Müggelsee,Periodic sampling
9,2022-11,11.70,1,Müggelsee,Periodic sampling



Monthly records per lake:
lake
Großer Wannsee     23
Kleiner Wannsee    27
Müggelsee          29
Tegeler See        25
dtype: int64

Observations per month:


,min,median,max
lake,,,
Großer Wannsee,1,1.0,2
Kleiner Wannsee,1,1.0,2
Müggelsee,1,1.0,2
Tegeler See,1,2.0,3


In [29]:
import numpy as np 

# Make copies so the original monthly datasets remain unchanged
continuous = monthly_continuous.copy()
sampling = monthly_sampling.copy()

# Identify the measurement method for continuous monitoring
continuous["measurement_method"] = "Continuous monitoring"

# Add missing temperature-range columns to the sampling dataset
# These measurements were not included in the sampling aggregation
sampling["monthly_min_temp"] = np.nan
sampling["monthly_max_temp"] = np.nan

# Keep the same columns and order in both datasets
common_columns = [
    "Monat",
    "monthly_mean_temp",
    "monthly_min_temp",
    "monthly_max_temp",
    "n_obs",
    "lake",
    "measurement_method"
]

continuous = continuous[common_columns]
sampling = sampling[common_columns]

# Combine all six lakes
monthly_all_lakes = pd.concat(
    [continuous, sampling],
    ignore_index=True
)

# Extract the year from the monthly period
monthly_all_lakes["year"] = (
    monthly_all_lakes["Monat"].dt.year
)

# Summarise monthly coverage by lake and year
coverage_summary = (
    monthly_all_lakes
    .groupby(["lake", "measurement_method", "year"])
    .agg(
        available_months=("Monat", "nunique"),
        total_observations=("n_obs", "sum"),
        median_observations_per_month=("n_obs", "median")
    )
    .reset_index()
)

# Display coverage and a sample of the combined dataset
display(coverage_summary)

print("\nCombined dataset shape:", monthly_all_lakes.shape)

display(
    monthly_all_lakes.sort_values(
        ["lake", "Monat"]
    ).head(15)
)

,lake,measurement_method,year,available_months,total_observations,median_observations_per_month
0,Flughafensee,Continuous monitoring,2022,12,365,31.0
1,Flughafensee,Continuous monitoring,2023,12,365,31.0
2,Flughafensee,Continuous monitoring,2024,12,366,31.0
3,Großer Wannsee,Periodic sampling,2022,10,12,1.0
4,Großer Wannsee,Periodic sampling,2023,8,11,1.0
5,Großer Wannsee,Periodic sampling,2024,5,8,2.0
6,Kleiner Wannsee,Periodic sampling,2022,10,12,1.0
7,Kleiner Wannsee,Periodic sampling,2023,9,12,1.0
8,Kleiner Wannsee,Periodic sampling,2024,8,12,1.5
9,Müggelsee,Periodic sampling,2022,11,12,1.0



Combined dataset shape: (176, 8)


,Monat,monthly_mean_temp,monthly_min_temp,monthly_max_temp,n_obs,lake,measurement_method,year
0,2022-01,4.399355,4.242903,4.551935,31,Flughafensee,Continuous monitoring,2022
1,2022-02,4.557143,4.382143,4.835000,28,Flughafensee,Continuous monitoring,2022
2,2022-03,6.088387,5.691935,6.612581,31,Flughafensee,Continuous monitoring,2022
3,2022-04,9.497000,9.025333,10.001000,30,Flughafensee,Continuous monitoring,2022
4,2022-05,18.196129,17.506129,18.934194,31,Flughafensee,Continuous monitoring,2022
5,2022-06,22.236000,21.443667,23.081667,30,Flughafensee,Continuous monitoring,2022
6,2022-07,22.753226,22.058710,23.471613,31,Flughafensee,Continuous monitoring,2022
7,2022-08,24.043871,23.399677,24.765161,31,Flughafensee,Continuous monitoring,2022
8,2022-09,18.007333,17.592333,18.464667,30,Flughafensee,Continuous monitoring,2022
9,2022-10,14.263871,13.984194,14.583548,31,Flughafensee,Continuous monitoring,2022


In [31]:
berlin_water_quality = pd.read_csv("merged_complete_datasets/berlin_water_quality_cleaned.csv")

# Inspect the columns in the original bacteria dataset
print("Bacteria dataset columns:")
print(berlin_water_quality.columns.tolist())

# Check the lake names in the bacteria dataset
print("\nBacteria dataset lake names:")
print(
    sorted(berlin_water_quality["lake"].dropna().unique())
)

# Check the date range of the bacteria dataset
print("\nBacteria dataset date range:")
print(
    berlin_water_quality["date"].min(),
    "to",
    berlin_water_quality["date"].max()
)

# Check the lake names in the monthly temperature dataset
print("\nTemperature dataset lake names:")
print(
    sorted(monthly_all_lakes["lake"].dropna().unique())
)

# Check the date range of the monthly temperature dataset
print("\nTemperature dataset month range:")
print(
    monthly_all_lakes["Monat"].min(),
    "to",
    monthly_all_lakes["Monat"].max()
)

Bacteria dataset columns:
['date', 'bathing_water_quality', 'coliform_bacteria', 'cyanobacteria_chl_a', 'e_coli', 'intestinal_enterococci', 'water_visibility_depth', 'water_temperature', 'lake', 'water_temperature_raw', 'water_temperature_qualifier', 'water_temperature_numeric', 'temperature_zero_flag', 'coliform_bacteria_raw', 'coliform_bacteria_qualifier', 'coliform_bacteria_numeric', 'e_coli_raw', 'e_coli_qualifier', 'e_coli_numeric', 'intestinal_enterococci_raw', 'intestinal_enterococci_qualifier', 'intestinal_enterococci_numeric', 'cyanobacteria_chl_a_raw', 'cyanobacteria_chl_a_qualifier', 'cyanobacteria_chl_a_numeric', 'water_visibility_depth_raw', 'water_visibility_depth_qualifier', 'water_visibility_depth_numeric']

Bacteria dataset lake names:
['Flughafensee', 'Müggelsee', 'Orankesee', 'Tegeler See', 'Wannsee', 'Weißensee']

Bacteria dataset date range:
2025-04-28 to 2026-09-15

Temperature dataset lake names:
['Flughafensee', 'Großer Wannsee', 'Kleiner Wannsee', 'Müggelsee', 

In [32]:
# Filter the existing dataset to Wannsee records
wannsee_records = berlin_water_quality[
    berlin_water_quality["lake"] == "Wannsee"
].copy()

# Display dates and available temperature measurements
display(
    wannsee_records[
        [
            "date",
            "water_temperature",
            "water_temperature_raw",
            "water_temperature_numeric",
            "water_temperature_qualifier"
        ]
    ].sort_values("date")
)

# Check how many Wannsee records have numeric temperatures
print(
    "\nWannsee records with numeric temperature:",
    wannsee_records["water_temperature_numeric"].notna().sum()
)

print(
    "Total Wannsee records:",
    len(wannsee_records)
)

,date,water_temperature,water_temperature_raw,water_temperature_numeric,water_temperature_qualifier
19,2025-04-29,"17,70","17,70",17.7,NaN
20,2025-05-27,"16,60","16,60",16.6,NaN
21,2025-06-12,NaN,NaN,NaN,NaN
22,2025-06-24,"null 21,30","null 21,30",21.3,NaN
23,2025-07-16,"null 21,80","null 21,80",21.8,NaN
24,2025-07-29,"null 21,20","null 21,20",21.2,NaN
25,2025-08-13,"null 22,20","null 22,20",22.2,NaN
26,2025-08-26,"null 19,80","null 19,80",19.8,NaN
27,2025-09-16,"18,70","18,70",18.7,NaN
28,2026-04-28,"12,30","12,30",12.3,NaN



Wannsee records with numeric temperature: 18
Total Wannsee records: 19


In [34]:
# Convert the date column to pandas datetime
berlin_water_quality["date"] = pd.to_datetime(
    berlin_water_quality["date"],
    errors="coerce"
)

# Check the resulting data type and any dates that failed conversion
print("Date data type:", berlin_water_quality["date"].dtype)

print(
    "Missing dates after conversion:",
    berlin_water_quality["date"].isna().sum()
)

Date data type: datetime64[us]
Missing dates after conversion: 0


In [35]:
# Summarise the number of bacteria sampling events and available temperatures
temperature_coverage = (
    berlin_water_quality
    .groupby([
        "lake",
        berlin_water_quality["date"].dt.year.rename("year")
    ])
    .agg(
        total_sampling_events=("date", "count"),
        available_temperatures=("water_temperature_numeric", "count"),
        missing_temperatures=("water_temperature_numeric",
                              lambda x: x.isna().sum())
    )
    .reset_index()
)

# Calculate the percentage of sampling events with temperature data
temperature_coverage["temperature_coverage_pct"] = (
    temperature_coverage["available_temperatures"]
    / temperature_coverage["total_sampling_events"]
    * 100
)

# Display the coverage summary
display(
    temperature_coverage.sort_values(["lake", "year"])
)

,lake,year,total_sampling_events,available_temperatures,missing_temperatures,temperature_coverage_pct
0,Flughafensee,2025,9,9,0,100.000000
1,Flughafensee,2026,10,10,0,100.000000
2,Müggelsee,2025,10,10,0,100.000000
3,Müggelsee,2026,10,10,0,100.000000
4,Orankesee,2025,10,10,0,100.000000
5,Orankesee,2026,10,9,1,90.000000
6,Tegeler See,2025,10,10,0,100.000000
7,Tegeler See,2026,10,9,1,90.000000
8,Wannsee,2025,9,8,1,88.888889
9,Wannsee,2026,10,10,0,100.000000


In [ ]:
water_quality_2022_2026 = pd.read_csv("final_datasets/berlin_water_quality_2022_2026.csv")

water_quality_2022_2026.head()

,date,bathing_water_quality,coliform_bacteria,cyanobacteria_chl_a,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature,lake,water_temperature_raw,...,e_coli_numeric,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric,cyanobacteria_chl_a_raw,cyanobacteria_chl_a_qualifier,cyanobacteria_chl_a_numeric,water_visibility_depth_raw,water_visibility_depth_qualifier,water_visibility_depth_numeric
0,2022-04-20,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"0,00",NaN,0.0
1,2022-05-17,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"0,00",NaN,0.0
2,2022-05-31,NaN,NaN,NaN,"< 15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"15,00",NaN,15.0,NaN,NaN,NaN,"> 1,00",>,1.0
3,2022-06-13,NaN,NaN,NaN,"< 15,00","< 15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"> 1,00",>,1.0
4,2022-06-27,NaN,NaN,NaN,"15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"15,00",NaN,15.0,NaN,NaN,NaN,"> 1,00",>,1.0


In [39]:
# Check the dataset dimensions and date format
print("Shape:", water_quality_2022_2026.shape)
print("Date dtype:", water_quality_2022_2026["date"].dtype)

# Convert date to datetime if it is currently stored as text
water_quality_2022_2026["date"] = pd.to_datetime(
    water_quality_2022_2026["date"],
    errors="coerce"
)

# Check the overall date range
print("\nDate range:")
print(water_quality_2022_2026["date"].min())
print(water_quality_2022_2026["date"].max())

# Check the number of records per lake
print("\nRecords per lake:")
print(
    water_quality_2022_2026["lake"]
    .value_counts()
    .sort_index()
)

# Check whether any lake/date combinations are duplicated
duplicates = water_quality_2022_2026.duplicated(
    subset=["date", "lake"],
    keep=False
)

print("\nDuplicate lake/date combinations:", duplicates.sum())

# Display duplicates if any exist
display(
    water_quality_2022_2026.loc[duplicates]
    .sort_values(["lake", "date"])
)

Shape: (298, 28)
Date dtype: str

Date range:
2022-04-20 00:00:00
2026-09-15 00:00:00

Records per lake:
lake
Flughafensee    49
Müggelsee       51
Orankesee       49
Tegeler See     50
Wannsee         49
Weißensee       50
Name: count, dtype: int64

Duplicate lake/date combinations: 0


,date,bathing_water_quality,coliform_bacteria,cyanobacteria_chl_a,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature,lake,water_temperature_raw,...,e_coli_numeric,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric,cyanobacteria_chl_a_raw,cyanobacteria_chl_a_qualifier,cyanobacteria_chl_a_numeric,water_visibility_depth_raw,water_visibility_depth_qualifier,water_visibility_depth_numeric


In [40]:
# Extract the year from each sampling date
water_quality_2022_2026["year"] = (
    water_quality_2022_2026["date"].dt.year
)

# Define the parameters we want to assess
parameters = [
    "e_coli_numeric",
    "intestinal_enterococci_numeric",
    "water_visibility_depth_numeric",
    "water_temperature_numeric",
    "cyanobacteria_chl_a_numeric",
    "coliform_bacteria_numeric"
]

# Count available numeric measurements per year and parameter
coverage = (
    water_quality_2022_2026
    .groupby("year")[parameters]
    .count()
)

print("Available numeric measurements by year:")
display(coverage)

# Calculate the percentage of records with available measurements
coverage_pct = (
    water_quality_2022_2026
    .groupby("year")[parameters]
    .count()
    .div(
        water_quality_2022_2026.groupby("year").size(),
        axis=0
    )
    .mul(100)
    .round(1)
)

print("Measurement coverage (%):")
display(coverage_pct)

Available numeric measurements by year:


,e_coli_numeric,intestinal_enterococci_numeric,water_visibility_depth_numeric,water_temperature_numeric,cyanobacteria_chl_a_numeric,coliform_bacteria_numeric
year,,,,,,
2022,56,56,54,0,0,0
2023,61,61,61,0,0,0
2024,63,63,55,0,0,0
2025,58,58,56,57,54,57
2026,60,60,53,58,0,60


Measurement coverage (%):


,e_coli_numeric,intestinal_enterococci_numeric,water_visibility_depth_numeric,water_temperature_numeric,cyanobacteria_chl_a_numeric,coliform_bacteria_numeric
year,,,,,,
2022,100.0,100.0,96.4,0.0,0.0,0.0
2023,100.0,100.0,100.0,0.0,0.0,0.0
2024,100.0,100.0,87.3,0.0,0.0,0.0
2025,100.0,100.0,96.6,98.3,93.1,98.3
2026,100.0,100.0,88.3,96.7,0.0,100.0


In [41]:
# Count qualifier types by year for E. coli
print("E. coli qualifiers:")
display(
    pd.crosstab(
        water_quality_2022_2026["year"],
        water_quality_2022_2026["e_coli_qualifier"],
        dropna=False
    )
)

# Count qualifier types by year for intestinal enterococci
print("\nIntestinal enterococci qualifiers:")
display(
    pd.crosstab(
        water_quality_2022_2026["year"],
        water_quality_2022_2026["intestinal_enterococci_qualifier"],
        dropna=False
    )
)

E. coli qualifiers:


e_coli_qualifier,<,NaN
year,,
2022,42,14
2023,41,20
2024,33,30
2025,35,23
2026,38,22



Intestinal enterococci qualifiers:


intestinal_enterococci_qualifier,<,NaN
year,,
2022,47,9
2023,46,15
2024,52,11
2025,50,8
2026,45,15


In [44]:
# List all available columns
print("Dataset columns:")
print(water_quality_2022_2026.columns.tolist())

# Inspect raw E. coli values
print("\nE. coli raw values:")
display(
    water_quality_2022_2026["e_coli_raw"]
    .value_counts(dropna=False)
    .sort_index()
)

# Inspect raw enterococci values
print("\nIntestinal enterococci raw values:")
display(
    water_quality_2022_2026["intestinal_enterococci_raw"]
    .value_counts(dropna=False)
    .sort_index()
)

Dataset columns:
['date', 'bathing_water_quality', 'coliform_bacteria', 'cyanobacteria_chl_a', 'e_coli', 'intestinal_enterococci', 'water_visibility_depth', 'water_temperature', 'lake', 'water_temperature_raw', 'water_temperature_qualifier', 'water_temperature_numeric', 'temperature_zero_flag', 'coliform_bacteria_raw', 'coliform_bacteria_qualifier', 'coliform_bacteria_numeric', 'e_coli_raw', 'e_coli_qualifier', 'e_coli_numeric', 'intestinal_enterococci_raw', 'intestinal_enterococci_qualifier', 'intestinal_enterococci_numeric', 'cyanobacteria_chl_a_raw', 'cyanobacteria_chl_a_qualifier', 'cyanobacteria_chl_a_numeric', 'water_visibility_depth_raw', 'water_visibility_depth_qualifier', 'water_visibility_depth_numeric', 'year']

E. coli raw values:


e_coli_raw
0,0000          1
109,00          1
110,00          3
1100,00         1
143,00          1
144,00          1
15,00          34
1561,00         1
159             1
161,00          1
197,00          1
213,00          1
251             1
30,00          21
4277,00         1
45,00           3
46,00           6
539,00          1
606,00          1
61,00           4
740             1
77,00           3
93,00           1
94,00           1
< 15,00       189
null 143        1
null 15,00     10
null 30,00      4
null 45,00      2
null 46,00      1
Name: count, dtype: int64


Intestinal enterococci raw values:


intestinal_enterococci_raw
0,0000          1
109             1
127,00          1
144,00          1
15,00          36
30,00           8
4005,00         1
46,00           1
61,00           1
77,00           1
918,00          1
< 15,00       240
null 109        1
null 15,00      4
Name: count, dtype: int64

In [45]:
# Inspect E. coli records with the literal "null" prefix
print("E. coli records with 'null' prefix:")
display(
    water_quality_2022_2026.loc[
        water_quality_2022_2026["e_coli_raw"]
        .astype("string")
        .str.startswith("null", na=False),
        [
            "date", "lake", "e_coli",
            "e_coli_raw", "e_coli_qualifier",
            "e_coli_numeric"
        ]
    ]
)

# Inspect the zero E. coli measurement
print("\nE. coli zero measurement:")
display(
    water_quality_2022_2026.loc[
        water_quality_2022_2026["e_coli_numeric"] == 0,
        [
            "date", "lake", "e_coli",
            "e_coli_raw", "e_coli_qualifier",
            "e_coli_numeric"
        ]
    ]
)

# Check whether the original long-format dataset is still available
print("\nOriginal historical dataset available:",
      "df_historical_long" in globals())

if "df_historical_long" in globals():
    print("Original columns:")
    print(df_historical_long.columns.tolist())
    if "Einheit" in df_historical_long.columns:
        print("\nOriginal measurement units:")
        print(df_historical_long["Einheit"].value_counts(dropna=False))

E. coli records with 'null' prefix:


,date,lake,e_coli,e_coli_raw,e_coli_qualifier,e_coli_numeric
33,2025-06-11,Flughafensee,"null 30,00","null 30,00",NaN,30.0
34,2025-07-10,Flughafensee,"null 15,00","null 15,00",NaN,15.0
35,2025-07-24,Flughafensee,null 143,null 143,NaN,143.0
36,2025-08-06,Flughafensee,"null 15,00","null 15,00",NaN,15.0
37,2025-08-20,Flughafensee,"null 30,00","null 30,00",NaN,30.0
86,2025-08-05,Müggelsee,"null 15,00","null 15,00",NaN,15.0
132,2025-06-11,Orankesee,"null 30,00","null 30,00",NaN,30.0
135,2025-07-24,Orankesee,"null 45,00","null 45,00",NaN,45.0
137,2025-08-20,Orankesee,"null 46,00","null 46,00",NaN,46.0
183,2025-06-23,Tegeler See,"null 15,00","null 15,00",NaN,15.0



E. coli zero measurement:


,date,lake,e_coli,e_coli_raw,e_coli_qualifier,e_coli_numeric
197,2026-09-07,Tegeler See,"0,0000","0,0000",NaN,0.0



Original historical dataset available: False


In [46]:
# Count literal "null" prefixes in both bacteria parameters
for parameter in ["e_coli", "intestinal_enterococci"]:

    raw_column = f"{parameter}_raw"

    null_count = (
        water_quality_2022_2026[raw_column]
        .astype("string")
        .str.startswith("null", na=False)
        .sum()
    )

    print(f"{parameter}: {null_count} records with 'null' prefix")

    display(
        water_quality_2022_2026.loc[
            water_quality_2022_2026[raw_column]
            .astype("string")
            .str.startswith("null", na=False),
            ["date", "lake", raw_column, f"{parameter}_qualifier",
             f"{parameter}_numeric"]
        ]
    )

e_coli: 18 records with 'null' prefix


,date,lake,e_coli_raw,e_coli_qualifier,e_coli_numeric
33,2025-06-11,Flughafensee,"null 30,00",NaN,30.0
34,2025-07-10,Flughafensee,"null 15,00",NaN,15.0
35,2025-07-24,Flughafensee,null 143,NaN,143.0
36,2025-08-06,Flughafensee,"null 15,00",NaN,15.0
37,2025-08-20,Flughafensee,"null 30,00",NaN,30.0
86,2025-08-05,Müggelsee,"null 15,00",NaN,15.0
132,2025-06-11,Orankesee,"null 30,00",NaN,30.0
135,2025-07-24,Orankesee,"null 45,00",NaN,45.0
137,2025-08-20,Orankesee,"null 46,00",NaN,46.0
183,2025-06-23,Tegeler See,"null 15,00",NaN,15.0


intestinal_enterococci: 5 records with 'null' prefix


,date,lake,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric
33,2025-06-11,Flughafensee,"null 15,00",NaN,15.0
34,2025-07-10,Flughafensee,"null 15,00",NaN,15.0
35,2025-07-24,Flughafensee,null 109,NaN,109.0
135,2025-07-24,Orankesee,"null 15,00",NaN,15.0
284,2025-07-24,Weißensee,"null 15,00",NaN,15.0


In [47]:
# Calculate the percentage of samples below the reporting limit
for parameter in ["e_coli", "intestinal_enterococci"]:

    qualifier_column = f"{parameter}_qualifier"

    # Count samples below the reporting limit by lake and year
    below_limit = (
        water_quality_2022_2026
        .assign(
            below_limit=lambda df: (
                df[qualifier_column] == "<"
            )
        )
        .groupby(["lake", "year"])["below_limit"]
        .agg(["sum", "count", "mean"])
        .reset_index()
    )

    # Convert the proportion into a percentage
    below_limit["percentage"] = (
        below_limit["mean"] * 100
    ).round(1)

    print(f"\n{parameter.upper()} — below-limit results by lake and year")

    display(
        below_limit[
            ["lake", "year", "sum", "count", "percentage"]
        ]
        .rename(columns={
            "sum": "below_limit_samples",
            "count": "total_samples"
        })
    )


E_COLI — below-limit results by lake and year


,lake,year,below_limit_samples,total_samples,percentage
0,Flughafensee,2022,6,9,66.7
1,Flughafensee,2023,5,11,45.5
2,Flughafensee,2024,2,10,20.0
3,Flughafensee,2025,2,9,22.2
4,Flughafensee,2026,5,10,50.0
5,Müggelsee,2022,10,10,100.0
6,Müggelsee,2023,10,10,100.0
7,Müggelsee,2024,9,11,81.8
8,Müggelsee,2025,9,10,90.0
9,Müggelsee,2026,9,10,90.0



INTESTINAL_ENTEROCOCCI — below-limit results by lake and year


,lake,year,below_limit_samples,total_samples,percentage
0,Flughafensee,2022,6,9,66.7
1,Flughafensee,2023,6,11,54.5
2,Flughafensee,2024,7,10,70.0
3,Flughafensee,2025,4,9,44.4
4,Flughafensee,2026,7,10,70.0
5,Müggelsee,2022,10,10,100.0
6,Müggelsee,2023,9,10,90.0
7,Müggelsee,2024,11,11,100.0
8,Müggelsee,2025,10,10,100.0
9,Müggelsee,2026,10,10,100.0


In [48]:
# Analyse unqualified reported concentrations only
for parameter in ["e_coli", "intestinal_enterococci"]:

    qualifier_column = f"{parameter}_qualifier"
    numeric_column = f"{parameter}_numeric"

    # Select measurements without a < or > qualifier
    exact_results = water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column].isna()
    ].copy()

    # Summarise the reported values by lake and year
    summary = (
        exact_results
        .groupby(["lake", "year"])[numeric_column]
        .agg(
            n="count",
            median="median",
            maximum="max"
        )
        .round(1)
        .reset_index()
    )

    print(f"\n{parameter.upper()} — unqualified reported values")
    display(summary)


E_COLI — unqualified reported values


,lake,year,n,median,maximum
0,Flughafensee,2022,3,15.0,45.0
1,Flughafensee,2023,6,153.5,606.0
2,Flughafensee,2024,8,38.0,144.0
3,Flughafensee,2025,7,30.0,143.0
4,Flughafensee,2026,5,30.0,740.0
5,Müggelsee,2024,2,2153.5,4277.0
6,Müggelsee,2025,1,15.0,15.0
7,Müggelsee,2026,1,15.0,15.0
8,Orankesee,2022,4,30.0,30.0
9,Orankesee,2023,5,46.0,94.0



INTESTINAL_ENTEROCOCCI — unqualified reported values


,lake,year,n,median,maximum
0,Flughafensee,2022,3,15.0,15.0
1,Flughafensee,2023,5,46.0,127.0
2,Flughafensee,2024,3,15.0,144.0
3,Flughafensee,2025,5,15.0,109.0
4,Flughafensee,2026,3,15.0,109.0
5,Müggelsee,2023,1,15.0,15.0
6,Orankesee,2022,3,15.0,15.0
7,Orankesee,2023,2,30.0,30.0
8,Orankesee,2024,2,22.5,30.0
9,Orankesee,2025,1,15.0,15.0


Bathing-water bacteria: variable results across lakes and years
- Most E. coli and intestinal enterococci results were reported below the 15-unit reporting threshold.
- A small number of unqualified results were substantially higher, including E. coli at Müggelsee (4,277 in 2024) and enterococci at Tegeler See (4,005 in 2022).
- These isolated high values highlight variability, but the small number of unqualified observations prevents robust year-to-year comparisons.

In [49]:
# Inspect raw bacteria records that contain the literal "null" prefix
for parameter in ["e_coli", "intestinal_enterococci"]:

    raw_column = f"{parameter}_raw"
    qualifier_column = f"{parameter}_qualifier"
    numeric_column = f"{parameter}_numeric"

    # Find raw values containing "null"
    null_rows = water_quality_2022_2026.loc[
        water_quality_2022_2026[raw_column]
        .astype("string")
        .str.contains("null", case=False, na=False),
        ["date", "lake", raw_column, qualifier_column, numeric_column]
    ].sort_values(["lake", "date"])

    print(f"\n{parameter.upper()} — records containing 'null'")
    print(f"Number of records: {len(null_rows)}")
    display(null_rows)


E_COLI — records containing 'null'
Number of records: 18


,date,lake,e_coli_raw,e_coli_qualifier,e_coli_numeric
33,2025-06-11,Flughafensee,"null 30,00",NaN,30.0
34,2025-07-10,Flughafensee,"null 15,00",NaN,15.0
35,2025-07-24,Flughafensee,null 143,NaN,143.0
36,2025-08-06,Flughafensee,"null 15,00",NaN,15.0
37,2025-08-20,Flughafensee,"null 30,00",NaN,30.0
86,2025-08-05,Müggelsee,"null 15,00",NaN,15.0
132,2025-06-11,Orankesee,"null 30,00",NaN,30.0
135,2025-07-24,Orankesee,"null 45,00",NaN,45.0
137,2025-08-20,Orankesee,"null 46,00",NaN,46.0
183,2025-06-23,Tegeler See,"null 15,00",NaN,15.0



INTESTINAL_ENTEROCOCCI — records containing 'null'
Number of records: 5


,date,lake,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric
33,2025-06-11,Flughafensee,"null 15,00",NaN,15.0
34,2025-07-10,Flughafensee,"null 15,00",NaN,15.0
35,2025-07-24,Flughafensee,null 109,NaN,109.0
135,2025-07-24,Orankesee,"null 15,00",NaN,15.0
284,2025-07-24,Weißensee,"null 15,00",NaN,15.0


In [50]:
# List CSV files in the final datasets folder
from pathlib import Path

dataset_folder = Path("final_datasets")

csv_files = sorted(dataset_folder.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file in csv_files:
    print(file.name)

CSV files found: 3
berlin_air_quality_2022_2026.csv
berlin_water_quality_2022_2026.csv
df_weather_combined_2022_2026.csv


In [51]:
# Compare raw values, parsed values and qualifiers
for parameter in ["e_coli", "intestinal_enterococci"]:

    raw_column = f"{parameter}_raw"
    qualifier_column = f"{parameter}_qualifier"
    numeric_column = f"{parameter}_numeric"

    print(f"\n{parameter.upper()}")

    # Count combinations of raw prefix and parsed qualifier
    prefix_check = water_quality_2022_2026.assign(
        has_null_prefix=water_quality_2022_2026[raw_column]
        .astype("string")
        .str.contains(r"^\s*null\b", case=False, na=False)
    )

    display(
        prefix_check
        .groupby(["year", "has_null_prefix", qualifier_column], dropna=False)
        .agg(
            records=(raw_column, "size"),
            numeric_values=(numeric_column, "count")
        )
        .reset_index()
        .sort_values(["year", "has_null_prefix", qualifier_column], na_position="last")
    )


E_COLI


,year,has_null_prefix,e_coli_qualifier,records,numeric_values
0,2022,False,<,42,42
1,2022,False,NaN,14,14
2,2023,False,<,41,41
3,2023,False,NaN,20,20
4,2024,False,<,33,33
5,2024,False,NaN,30,30
6,2025,False,<,35,35
7,2025,False,NaN,5,5
8,2025,True,NaN,18,18
9,2026,False,<,38,38



INTESTINAL_ENTEROCOCCI


,year,has_null_prefix,intestinal_enterococci_qualifier,records,numeric_values
0,2022,False,<,47,47
1,2022,False,NaN,9,9
2,2023,False,<,46,46
3,2023,False,NaN,15,15
4,2024,False,<,52,52
5,2024,False,NaN,11,11
6,2025,False,<,50,50
7,2025,False,NaN,3,3
8,2025,True,NaN,5,5
9,2026,False,<,45,45


In [52]:
# Inspect all available columns and their data types
for column in water_quality_2022_2026.columns:
    print(f"{column:<45} {water_quality_2022_2026[column].dtype}")

date                                          datetime64[us]
bathing_water_quality                         str
coliform_bacteria                             str
cyanobacteria_chl_a                           str
e_coli                                        str
intestinal_enterococci                        str
water_visibility_depth                        str
water_temperature                             str
lake                                          str
water_temperature_raw                         str
water_temperature_qualifier                   float64
water_temperature_numeric                     float64
temperature_zero_flag                         object
coliform_bacteria_raw                         str
coliform_bacteria_qualifier                   str
coliform_bacteria_numeric                     float64
e_coli_raw                                    str
e_coli_qualifier                              str
e_coli_numeric                                float64
intestinal_enterococ

In [53]:
# Document bacterial measurement units and reporting limits
bacteria_metadata = {
    "e_coli": {
        "unit": "KBE/100 ml",
        "lower_quantification_limit": 15
    },
    "intestinal_enterococci": {
        "unit": "KBE/100 ml",
        "lower_quantification_limit": 15
    }
}

display(pd.DataFrame(bacteria_metadata).T)

,unit,lower_quantification_limit
e_coli,KBE/100 ml,15
intestinal_enterococci,KBE/100 ml,15


In [54]:
# Create a transparent classification of bacterial observations
for parameter in ["e_coli", "intestinal_enterococci"]:

    raw_column = f"{parameter}_raw"
    qualifier_column = f"{parameter}_qualifier"

    # Identify raw values containing the literal "null" prefix
    has_null_prefix = (
        water_quality_2022_2026[raw_column]
        .astype("string")
        .str.contains(r"^\s*null\b", case=False, na=False)
    )

    # Classify each observation without modifying the original values
    water_quality_2022_2026[f"{parameter}_result_type"] = "other / missing"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column] == "<",
        f"{parameter}_result_type"
    ] = "below quantification limit"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column].isna() & ~has_null_prefix,
        f"{parameter}_result_type"
    ] = "unqualified — no null prefix"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column].isna() & has_null_prefix,
        f"{parameter}_result_type"
    ] = "unqualified — null prefix"

    print(f"\n{parameter.upper()}")
    display(
        water_quality_2022_2026[
            f"{parameter}_result_type"
        ].value_counts(dropna=False).rename("records").to_frame()
    )


E_COLI


,records
e_coli_result_type,
below quantification limit,189
unqualified — no null prefix,91
unqualified — null prefix,18



INTESTINAL_ENTEROCOCCI


,records
intestinal_enterococci_result_type,
below quantification limit,240
unqualified — no null prefix,53
unqualified — null prefix,5


In [55]:
# Create a transparent classification of bacterial observations
for parameter in ["e_coli", "intestinal_enterococci"]:

    raw_column = f"{parameter}_raw"
    qualifier_column = f"{parameter}_qualifier"

    # Identify raw values containing the literal "null" prefix
    has_null_prefix = (
        water_quality_2022_2026[raw_column]
        .astype("string")
        .str.contains(r"^\s*null\b", case=False, na=False)
    )

    # Classify each observation without modifying the original values
    water_quality_2022_2026[f"{parameter}_result_type"] = "other / missing"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column] == "<",
        f"{parameter}_result_type"
    ] = "below quantification limit"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column].isna() & ~has_null_prefix,
        f"{parameter}_result_type"
    ] = "unqualified — no null prefix"

    water_quality_2022_2026.loc[
        water_quality_2022_2026[qualifier_column].isna() & has_null_prefix,
        f"{parameter}_result_type"
    ] = "unqualified — null prefix"

    print(f"\n{parameter.upper()}")
    display(
        water_quality_2022_2026[
            f"{parameter}_result_type"
        ].value_counts(dropna=False).rename("records").to_frame()
    )


E_COLI


,records
e_coli_result_type,
below quantification limit,189
unqualified — no null prefix,91
unqualified — null prefix,18



INTESTINAL_ENTEROCOCCI


,records
intestinal_enterococci_result_type,
below quantification limit,240
unqualified — no null prefix,53
unqualified — null prefix,5


In [56]:
# Check temperature and bacteria coverage on matching sampling dates
temperature_coverage = (
    water_quality_2022_2026
    .groupby(["lake", "year"])
    .agg(
        total_samples=("date", "size"),
        temperature_available=("water_temperature_numeric", "count"),
        e_coli_available=("e_coli_numeric", "count"),
        enterococci_available=("intestinal_enterococci_numeric", "count"),
        temperature_zero_flags=("temperature_zero_flag", "count")
    )
    .reset_index()
)

display(temperature_coverage)

,lake,year,total_samples,temperature_available,e_coli_available,enterococci_available,temperature_zero_flags
0,Flughafensee,2022,9,0,9,9,0
1,Flughafensee,2023,11,0,11,11,0
2,Flughafensee,2024,10,0,10,10,0
3,Flughafensee,2025,9,9,9,9,9
4,Flughafensee,2026,10,10,10,10,10
5,Müggelsee,2022,10,0,10,10,0
6,Müggelsee,2023,10,0,10,10,0
7,Müggelsee,2024,11,0,11,11,0
8,Müggelsee,2025,10,10,10,10,10
9,Müggelsee,2026,10,10,10,10,10


In [57]:
# Check availability and structure of the historical temperature dataset
if "monthly_all_lakes" in globals():
    print("monthly_all_lakes is available")
    print("Shape:", monthly_all_lakes.shape)
    print("Columns:", monthly_all_lakes.columns.tolist())
    display(monthly_all_lakes.head())
else:
    print("monthly_all_lakes is not available in memory")

monthly_all_lakes is available
Shape: (176, 8)
Columns: ['Monat', 'monthly_mean_temp', 'monthly_min_temp', 'monthly_max_temp', 'n_obs', 'lake', 'measurement_method', 'year']


,Monat,monthly_mean_temp,monthly_min_temp,monthly_max_temp,n_obs,lake,measurement_method,year
0,2022-01,4.399355,4.242903,4.551935,31,Flughafensee,Continuous monitoring,2022
1,2022-02,4.557143,4.382143,4.835000,28,Flughafensee,Continuous monitoring,2022
2,2022-03,6.088387,5.691935,6.612581,31,Flughafensee,Continuous monitoring,2022
3,2022-04,9.497000,9.025333,10.001000,30,Flughafensee,Continuous monitoring,2022
4,2022-05,18.196129,17.506129,18.934194,31,Flughafensee,Continuous monitoring,2022


In [58]:
# Export cleaned water datasets and bacterial metadata
from pathlib import Path

output_folder = Path("final_datasets")
output_folder.mkdir(exist_ok=True)

# Save historical monthly water temperature
monthly_all_lakes.to_csv(
    output_folder / "berlin_water_temperature_monthly_2022_2024.csv",
    index=False
)

# Save bacterial measurement metadata
bacteria_metadata_df = (
    pd.DataFrame(bacteria_metadata)
    .T
    .rename_axis("parameter")
    .reset_index()
)

bacteria_metadata_df.to_csv(
    output_folder / "berlin_bacteria_metadata.csv",
    index=False
)

print("Export completed:")
print("1. berlin_water_temperature_monthly_2022_2024.csv")
print("2. berlin_bacteria_metadata.csv")
print("3. berlin_water_quality_2022_2026.csv (already exists)")

Export completed:
1. berlin_water_temperature_monthly_2022_2024.csv
2. berlin_bacteria_metadata.csv
3. berlin_water_quality_2022_2026.csv (already exists)
